# Experiment 1: synthetic Tanimoto benchmarks

Run this notebook from its folder with a **Windows x64 Python 3.12** kernel. **Run All defaults to the full experiment**: it generates 36 datasets, executes 42 settings across five methods in fresh processes, verifies assignments against generated truth, and writes 30 standalone figures in PNG, PDF and SVG. This is a long computation, especially for DBSCAN and Taylor–Butina; there is no fit time cutoff. Each worker has a 20 GiB private-memory limit. Full input generation requires roughly 13 GiB of disk space.

Only code and experimental design are embedded here: **no saved measurements, labels, fingerprints, data manifests, binaries, wheels or compressed archives**. The supplied plots are historical outputs. Running the full notebook produces new measurements and updates `plots/synthetic`; timings and memory naturally vary between runs. Pilot outputs stay outside the delivered folder.

The notebook materializes its readable Python/C helper cells in a separate work directory for native compilation and subprocess isolation. It installs pinned packages into an isolated environment and compiles the two C kernels using installed Visual Studio C++ Build Tools and the Windows SDK. No original Experiment 1A folder or machine-specific path is needed. Use `RUN_MODE = "pilot"` for one 256-row smoke check of all five methods before committing to the full run.
AMI and sampled-query/full-reference Tanimoto silhouette are generated after timing from the new saved assignments, with retained-population coverage reported alongside them.


In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import platform
import subprocess
import sys

RUN_MODE = "full"                         # "full", "pilot", or "check"
_candidates = [Path.cwd(), Path.cwd() / "experiment_1",
               Path.cwd() / "experiments" / "experiment_1",
               Path.cwd() / "classix_tanimoto_manhattan" / "experiments" / "experiment_1"]
NOTEBOOK_DIR = next((p.resolve() for p in _candidates
                     if (p / "01_synthetic_benchmarks.ipynb").is_file()), None)
if NOTEBOOK_DIR is None:
    raise RuntimeError("Start the kernel in this notebook folder or the repository root.")
WORK_DIR = NOTEBOOK_DIR.parent / "_experiment_1_work" / "synthetic_popcount10000"
PLOT_DIR = NOTEBOOK_DIR / "plots" / "synthetic"
PILOT_CASE = "sample_count_10000"
PILOT_ROWS = 256
INSTALL_DEPENDENCIES = True
PYTHON_FOR_RUNS = None                    # optional existing Windows x64 Python 3.12 executable

assert RUN_MODE in {"full", "pilot", "check"}
WORK_DIR = WORK_DIR.expanduser().resolve()
assert WORK_DIR != NOTEBOOK_DIR and not WORK_DIR.is_relative_to(NOTEBOOK_DIR),     "Choose a work directory outside the notebook folder."
SOURCES = {}


## Experimental design and timing

All datasets use seed **101**, the notebook's original generator and its subsequent shuffle, with `flip_prob=.3`. `n` is the number of points **per cluster**. Studies cover sample counts 10k–90k; maximum seed popcount 50–500 (10k total; 1,000 points per cluster); cluster counts 10, 20, 25, 40, 50, 80, 100 (10k total); dimensions 500–5000 (10k total); and radii .2–.7 on a shared 10k dataset. The omitted 100k sample setting and radius .1 are not rerun.

CLASSIX uses packed input, native popcounts, one connection bitmatrix and sparse minPts markers. Original BitBIRCH uses int64 input to avoid overflow; Lean uses packed input with zero padding to byte boundaries and is an optimized reference. Both use diameter merging, branching factor 50 and similarity `1-r`. Matching this number does not make their clustering criteria identical. Main-study DBSCAN uses Jaccard eps=.4/min_samples=5; Taylor–Butina uses cutoff=.35 with no reordering. Their radius-study cutoffs equal `r`.

Each fit uses a fresh process and one numerical thread. Timing includes estimator construction, all distance/neighborhood/tree/group calculations, clustering and label assignment. Generation, loading and native representation preparation are outside timing. Memory is the Windows process lifetime peak private commitment sampled immediately after fit, including imports and input; a flag records whether that peak increased during fit. ARI includes DBSCAN's `-1` noise label.

The explicit schedule below preserves the approved design: CLASSIX/BitBIRCH/Lean use one discarded warmup and three measured fits. DBSCAN/Butina use the historical mixed one/three-fit schedule, including retained warmups in earlier sample settings. The full run plots single measurements or medians accordingly. Terminal failures stop remaining repetitions and appear as status marks at their x positions, never as zeros. Pilot mode uses one measured fit per method without warmups and is labelled throughout.

The popcount study uses newly generated 10,000-row datasets and new dataset/case IDs. Its former 25,000-row results are not reused; other study settings are unchanged. The separate default work directory preserves earlier checkpoints.


In [ ]:
PLAN = json.loads(r'''{
  "case_count": 42,
  "unique_datasets": 36,
  "methods": [
    "dbscan",
    "classix_packed",
    "bitbirch",
    "bblean",
    "butina"
  ],
  "measured_repeats": 3,
  "discarded_warmups": 1,
  "fit_timeout_seconds": null,
  "job_memory_limit_bytes": 21474836480,
  "source_notebook_sha256": "ad1bcf5778b07e442ee6ff1c294ab32fcd60828d2b6af4e949a0335ce6da3848",
  "seed": 101,
  "butina_implementation": "standard_rdkit",
  "preparation_excluded": [
    "dataset generation",
    "loading",
    "conversion to each native input representation"
  ],
  "timed_operations": "estimator construction, all distance/neighborhood/tree/group construction, fit, and label assignment",
  "butina_timing_note": "RDKit ExplicitBitVect construction from exact generated packed bits is input representation preparation. It is excluded, like input packing for CLASSIX/Lean. Full Python distance-list construction, unchanged RDKit clustering and assignment are timed. No Morgan embedding applies to synthetic data.",
  "cases": [
    {
      "case_id": "sample_count_10000",
      "study": "sample_count",
      "x": 10000,
      "data_id": "e764dddbe2e39295",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "sample_count_20000",
      "study": "sample_count",
      "x": 20000,
      "data_id": "7a5b79151f07122b",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 2000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 20000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "sample_count_30000",
      "study": "sample_count",
      "x": 30000,
      "data_id": "9553f3b36bf4940c",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 3000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 30000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "sample_count_40000",
      "study": "sample_count",
      "x": 40000,
      "data_id": "072a17eb2de64ba0",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 4000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 40000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "sample_count_50000",
      "study": "sample_count",
      "x": 50000,
      "data_id": "95116e689c627897",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 5000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 50000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "sample_count_60000",
      "study": "sample_count",
      "x": 60000,
      "data_id": "31f5f6ddff7a107e",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 6000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 60000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "sample_count_70000",
      "study": "sample_count",
      "x": 70000,
      "data_id": "7aedfba859e0e320",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 7000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 70000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "sample_count_80000",
      "study": "sample_count",
      "x": 80000,
      "data_id": "44a70c5c2374cac9",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 8000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 80000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "sample_count_90000",
      "study": "sample_count",
      "x": 90000,
      "data_id": "1d938cf69524dd49",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 9000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 90000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "popcount_separation_50_n10000",
      "study": "popcount_separation",
      "x": 50,
      "data_id": "72b6051894807879",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          50
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_50"
    },
    {
      "case_id": "popcount_separation_100_n10000",
      "study": "popcount_separation",
      "x": 100,
      "data_id": "f7761b505bc2193c",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          100
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_100"
    },
    {
      "case_id": "popcount_separation_150_n10000",
      "study": "popcount_separation",
      "x": 150,
      "data_id": "0f8f49ce6e3c3977",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          150
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_150"
    },
    {
      "case_id": "popcount_separation_200_n10000",
      "study": "popcount_separation",
      "x": 200,
      "data_id": "ac21f5815100ae29",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          200
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_200"
    },
    {
      "case_id": "popcount_separation_250_n10000",
      "study": "popcount_separation",
      "x": 250,
      "data_id": "118040e5b3a88988",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          250
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_250"
    },
    {
      "case_id": "popcount_separation_300_n10000",
      "study": "popcount_separation",
      "x": 300,
      "data_id": "2194e1545c1bee6a",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          300
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_300"
    },
    {
      "case_id": "popcount_separation_350_n10000",
      "study": "popcount_separation",
      "x": 350,
      "data_id": "0d1babb663ced847",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          350
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_350"
    },
    {
      "case_id": "popcount_separation_400_n10000",
      "study": "popcount_separation",
      "x": 400,
      "data_id": "5b6d3028edaf8702",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          400
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_400"
    },
    {
      "case_id": "popcount_separation_450_n10000",
      "study": "popcount_separation",
      "x": 450,
      "data_id": "3d5c3162d55eadd7",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          450
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_450"
    },
    {
      "case_id": "popcount_separation_500_n10000",
      "study": "popcount_separation",
      "x": 500,
      "data_id": "a3ee8422bcb36651",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          500
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "source_cells_zero_based": [
        11,
        12
      ],
      "classix": {
        "radius": 0.3,
        "minPts": 5,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      },
      "historical_case_id": "popcount_separation_500"
    },
    {
      "case_id": "cluster_count_10",
      "study": "cluster_count",
      "x": 10,
      "data_id": "511ede241e000663",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          200
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "cluster_count_20",
      "study": "cluster_count",
      "x": 20,
      "data_id": "299d055cfbf8a488",
      "generator": {
        "num_clusters": 20,
        "pops": [
          100,
          200
        ],
        "d": 1000,
        "n": 500,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "cluster_count_25",
      "study": "cluster_count",
      "x": 25,
      "data_id": "e0a31bda1b0f0389",
      "generator": {
        "num_clusters": 25,
        "pops": [
          100,
          200
        ],
        "d": 1000,
        "n": 400,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "cluster_count_40",
      "study": "cluster_count",
      "x": 40,
      "data_id": "5ea7a508496b6ef3",
      "generator": {
        "num_clusters": 40,
        "pops": [
          100,
          200
        ],
        "d": 1000,
        "n": 250,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "cluster_count_50",
      "study": "cluster_count",
      "x": 50,
      "data_id": "56ef2504a53d3c89",
      "generator": {
        "num_clusters": 50,
        "pops": [
          100,
          200
        ],
        "d": 1000,
        "n": 200,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "cluster_count_80",
      "study": "cluster_count",
      "x": 80,
      "data_id": "f0baffcbecb535ce",
      "generator": {
        "num_clusters": 80,
        "pops": [
          100,
          200
        ],
        "d": 1000,
        "n": 125,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "cluster_count_100",
      "study": "cluster_count",
      "x": 100,
      "data_id": "83745654a6a6b2ed",
      "generator": {
        "num_clusters": 100,
        "pops": [
          100,
          200
        ],
        "d": 1000,
        "n": 100,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_500",
      "study": "dimension",
      "x": 500,
      "data_id": "0a5e20fb5570af7b",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 500,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 500,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_1000",
      "study": "dimension",
      "x": 1000,
      "data_id": "9c3290276fb22a15",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_1500",
      "study": "dimension",
      "x": 1500,
      "data_id": "7e68aee4bc5e4522",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 1500,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1500,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_2000",
      "study": "dimension",
      "x": 2000,
      "data_id": "7ace80cda4502b19",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 2000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 2000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_2500",
      "study": "dimension",
      "x": 2500,
      "data_id": "435f2331a62d2313",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 2500,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 2500,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_3000",
      "study": "dimension",
      "x": 3000,
      "data_id": "8a648bd3a67d4c53",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 3000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 3000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_3500",
      "study": "dimension",
      "x": 3500,
      "data_id": "9bb6ce07e21c1506",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 3500,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 3500,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_4000",
      "study": "dimension",
      "x": 4000,
      "data_id": "a16fabe868270a88",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 4000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 4000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_4500",
      "study": "dimension",
      "x": 4500,
      "data_id": "83d35ca6c03a0530",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 4500,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 4500,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "dimension_5000",
      "study": "dimension",
      "x": 5000,
      "data_id": "58c96a28f3bfc4b0",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          100
        ],
        "d": 5000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 5000,
      "classix": {
        "radius": 0.4,
        "minPts": 5,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.35,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0p2",
      "study": "radius",
      "x": 0.2,
      "data_id": "e764dddbe2e39295",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.2,
        "minPts": 40,
        "mergeScale": 2.23606797749979
      },
      "bitbirch": {
        "threshold": 0.8,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.2,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.2,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0p3",
      "study": "radius",
      "x": 0.3,
      "data_id": "e764dddbe2e39295",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.3,
        "minPts": 40,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.3,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.3,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0p4",
      "study": "radius",
      "x": 0.4,
      "data_id": "e764dddbe2e39295",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.4,
        "minPts": 40,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.4,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0p5",
      "study": "radius",
      "x": 0.5,
      "data_id": "e764dddbe2e39295",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.5,
        "minPts": 40,
        "mergeScale": 1.414213562373095
      },
      "bitbirch": {
        "threshold": 0.5,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.5,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.5,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0p6",
      "study": "radius",
      "x": 0.6,
      "data_id": "e764dddbe2e39295",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.6,
        "minPts": 40,
        "mergeScale": 1.2909944487358056
      },
      "bitbirch": {
        "threshold": 0.4,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.6,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.6,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0p7",
      "study": "radius",
      "x": 0.7,
      "data_id": "e764dddbe2e39295",
      "generator": {
        "num_clusters": 10,
        "pops": [
          100,
          400
        ],
        "d": 1000,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1000,
      "classix": {
        "radius": 0.7,
        "minPts": 40,
        "mergeScale": 1.1952286093343936
      },
      "bitbirch": {
        "threshold": 0.3,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.7,
        "min_samples": 5,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.7,
        "reordering": false
      }
    }
  ],
  "design_note": "42 settings; 100k sample case omitted; popcount and radius studies 10k; all data and results generated afresh."
}''')

REPETITION_POLICY = json.loads(r'''{
  "schedules": {
    "sample_count_10000": {
      "dbscan": [
        -1,
        0,
        1,
        2
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        -1,
        0,
        1,
        2
      ]
    },
    "sample_count_20000": {
      "dbscan": [
        -1,
        0,
        1,
        2
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        -1,
        0,
        1,
        2
      ]
    },
    "sample_count_30000": {
      "dbscan": [
        -1,
        0,
        1,
        2
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        -1,
        0,
        1,
        2
      ]
    },
    "sample_count_40000": {
      "dbscan": [
        -1,
        0,
        1,
        2
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        -1,
        0,
        1,
        2
      ]
    },
    "sample_count_50000": {
      "dbscan": [
        -1,
        0,
        1,
        2
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        -1,
        0,
        1,
        2
      ]
    },
    "sample_count_60000": {
      "dbscan": [
        -1,
        0,
        1,
        2
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        -1,
        0,
        1,
        2
      ]
    },
    "sample_count_70000": {
      "dbscan": [
        -1,
        0,
        1,
        2
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        -1,
        0,
        1,
        2
      ]
    },
    "sample_count_80000": {
      "dbscan": [
        -1,
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        -1,
        0
      ]
    },
    "sample_count_90000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "cluster_count_10": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "cluster_count_20": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "cluster_count_25": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "cluster_count_40": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "cluster_count_50": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "cluster_count_80": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "cluster_count_100": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_500": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_1000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_1500": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_2000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_2500": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_3000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_3500": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_4000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_4500": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "dimension_5000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "radius_0p2": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "radius_0p3": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "radius_0p4": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "radius_0p5": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "radius_0p6": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "radius_0p7": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_50_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_100_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_150_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_200_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_250_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_300_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_350_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_400_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_450_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    },
    "popcount_separation_500_n10000": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        -1,
        0,
        1,
        2
      ],
      "bitbirch": [
        -1,
        0,
        1,
        2
      ],
      "bblean": [
        -1,
        0,
        1,
        2
      ],
      "butina": [
        0
      ]
    }
  }
}''')

assert len(PLAN["cases"]) == 42
assert len({c["data_id"] for c in PLAN["cases"]}) == 36

**`code/generate_data_snapshot.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/generate_data_snapshot.py'] = r'''import numpy as np
import random

def generate_data(num_clusters, pops, d, n, flip_prob=0.1, seed=None):
    """
    Generate data for the CLASSIX_T dataset

        Parameters: \n
        num_clusters (int)  : number of clusters \n
        pops (list)         : the popcount of the different cluster centers. len(pops) must be equal to num_clusters \n
        d (int)             : dimension of the data \n
        n (int)             : number of total samples per cluster \n
        seed (int)          : seed for the random number generator \n


    """
    assert len(pops)>1, "You must provide at least 2 cluster mean populations in order to generate meaningful data"
    if len(pops) != num_clusters:
        try:
            print("The number of mean populations is not equal to the number of clusters. Trying to generate mean populations keeping in mind the minimum and maximum pop counts provided by the user")
            pops = np.linspace(min(pops), max(pops), num_clusters)
        except:
            raise ValueError("The number of cluster centers must be equal to the number of cluster populations")

    size_flag = [pops[x]<d for x in range(len(pops))]

    assert all(size_flag), "The number of dimensions must be greater than the pop counts of the clusters"

    clustc = []
    data = []
    if seed != None:
        np.random.seed(seed)
        random.seed(seed)

    labels = []

    for i in range(num_clusters):
        pop = pops[i]
        p1 = pop/d
        clustc.append(np.random.choice([0, 1], size=d, p=[1-p1, p1]))
        data.append(clustc[-1])
        labels.append(i)
        for j in range(n-1):
            labels.append(i)
            # Instead of flipping exactly pop*flip_prob bits, we flip each bit with a probability of flip_prob.
            flip_indices = np.random.choice([0, 1], size = d, p = [1-flip_prob*p1, flip_prob*p1])
            new_sample = np.logical_xor(clustc[-1], flip_indices).astype(np.int32)
            data.append(new_sample)
    return np.array(data).astype(np.int32), np.array(labels).astype(np.int32)
'''

**`code/worker.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/worker.py'] = r'''"""One fit with explicit pre-fit/post-fit checkpoints and separate validation."""
import os
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"
import argparse
import gc
import json
from pathlib import Path
import sys
import traceback
from time import perf_counter
sys.dont_write_bytecode = True
HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
BASE = ROOT
PACKED = HERE / "classix"
from memory_win import snapshot


def send(event, **values):
    print("BENCH_JSON " + json.dumps({"event": event, **values}), flush=True)


def handshake(expected):
    received = sys.stdin.readline().strip()
    if received != expected:
        raise RuntimeError(f"Expected {expected}, received {received}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--case", required=True)
    parser.add_argument("--variant", required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--rows", type=int)
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--plan", type=Path, default=ROOT/"plan.json")
    args = parser.parse_args()
    import numpy as np
    from threadpoolctl import threadpool_limits, threadpool_info
    plan = json.loads(args.plan.read_text(encoding="utf-8-sig"))
    case = next(c for c in plan["cases"] if c["case_id"] == args.case)
    directory = args.data_dir/case["data_id"]
    manifest = json.loads((directory/"manifest.json").read_text())
    parameters = {}
    library = {}
    if args.variant.startswith("classix"):
        sys.path.insert(0, str(PACKED))
        if args.variant == "classix":
            import classix_t_reference_adjbits as module
            estimator = module.CLASSIX_T
            kind = "int32"
        else:
            import classix_t_packed_adjbits as module
            estimator = module.CLASSIX_T_PACKED
            kind = "packed_little"
        module.print = lambda *a, **k: None
        module.tqdm = lambda iterable, *a, **k: iterable
        parameters = case["classix"]
    elif args.variant == "bitbirch":
        sys.path.insert(0, str(BASE/"vendor"/"bitbirch"))
        from bitbirch import bitbirch as module
        module.set_merge("diameter")
        estimator = module.BitBirch
        parameters = {k: case["bitbirch"][k] for k in ("threshold", "branching_factor")}
        kind = "int64"
        library = {"module": module.__file__, "merge_criterion": "diameter"}
    elif args.variant == "bblean":
        import bblean
        import bblean.similarity as similarity
        assert similarity._jt_sim_arr_vec_packed.__module__ == "bblean._cpp_similarity"
        estimator = bblean.BitBirch
        parameters = case["bitbirch"]
        kind = "packed_big"
        library = {"version": bblean.__version__, "native_kernel": similarity._jt_sim_arr_vec_packed.__module__,
                   "logical_features": case["features"], "native_features": 8*((case["features"]+7)//8),
                   "zero_padding_bits": (-case["features"]) % 8}
    elif args.variant == "dbscan":
        import sklearn
        from sklearn.cluster import DBSCAN
        estimator = DBSCAN
        parameters = case["dbscan"]
        kind = "int32"
        library = {"version": sklearn.__version__, "algorithm": "auto", "metric": "jaccard",
                   "input_note": "Notebook int32 input; sklearn performs its own boolean conversion inside fit"}
    elif args.variant == "butina":
        import rdkit
        from rdkit import DataStructs
        from rdkit.ML.Cluster import Butina
        parameters = case["butina"]
        kind = "packed_little"
        library = {"version": rdkit.__version__, "implementation": plan["butina_implementation"],
                   "module": Butina.__file__, "logical_features": case["features"],
                   "native_features": 8*((case["features"]+7)//8),
                   "zero_padding_bits": (-case["features"]) % 8}
    else:
        raise ValueError(args.variant)
    with threadpool_limits(limits=1):
        pool_info = threadpool_info()
        gc.collect()
        after_import = snapshot()
        data = np.load(directory/manifest["files"][kind]["file"], allow_pickle=False)
        if args.rows is not None:
            data = data[:args.rows].copy()
        representation_started = perf_counter()
        fingerprints = None
        if args.variant == "butina":
            fingerprints = [DataStructs.CreateFromBinaryText(row.tobytes()) for row in data]
        representation_seconds = perf_counter() - representation_started
        gc.collect()
        before = snapshot()
        send("ready", pid=os.getpid(), variant=args.variant, case=args.case, rows=len(data), features=case["features"],
             input_kind=kind, input_bytes=data.nbytes, parameters=parameters, library=library,
             representation_seconds=representation_seconds,
             threadpools=pool_info, after_import=after_import, before_fit=before)
        handshake("GO")
        started = perf_counter()
        try:
            model = None if args.variant == "butina" else estimator(**parameters)
            if args.variant == "butina":
                distances = []
                for i in range(1, len(fingerprints)):
                    similarities = DataStructs.BulkTanimotoSimilarity(fingerprints[i], fingerprints[:i])
                    distances.extend([1 - value for value in similarities])
                clusters = Butina.ClusterData(distances, len(fingerprints), parameters["cutoff"],
                                               isDistData=True, reordering=parameters["reordering"])
                fit_end = perf_counter()
                labels = np.full(len(data), -1, dtype=np.int64)
                for cluster_id, members in enumerate(clusters):
                    labels[list(members)] = cluster_id
            elif args.variant == "dbscan":
                model.fit(data)
                fit_end = perf_counter()
                labels = model.labels_
            elif args.variant == "bblean":
                # Native 0.10.3 unpacking requires a byte-aligned dimension.
                # Existing packbits padding is zero and preserves similarity.
                model.fit(data, input_is_packed=True, n_features=8*data.shape[1])
                fit_end = perf_counter()
                labels = model.get_assignments()
            elif args.variant == "bitbirch":
                model.fit(data)
                fit_end = perf_counter()
                labels = model.get_assignments(len(data))
            else:
                model.fit(data)
                fit_end = perf_counter()
                labels = model.labels
            ended = perf_counter()
        except Exception as error:
            send("fit_error", error_type=type(error).__name__, message=str(error),
                 elapsed_seconds=perf_counter()-started, memory=snapshot(), traceback=traceback.format_exc())
            return
        after = snapshot()
        send("fit_done", clustering_seconds=ended-started, construction_and_fit_seconds=fit_end-started,
             assignment_seconds=ended-fit_end, after_fit=after)
        handshake("VALIDATE")
        try:
            from quality import adjusted_rand_index
            from state_checks import array_hash, logical_hash, state_fingerprint
            labels = np.asarray(labels, dtype=np.int64)
            assert labels.shape == (len(data),) and np.all(labels >= (-1 if args.variant == "dbscan" else 0))
            noise_points = int(np.sum(labels == -1))
            ids, counts = np.unique(labels[labels >= 0], return_counts=True)
            truth = np.load(directory/manifest["files"]["truth"]["file"], allow_pickle=False)[:len(data)]
            ari = adjusted_rand_index(truth, labels)
            mapping = {}
            canonical = np.empty_like(labels)
            for i, value in enumerate(labels):
                canonical[i] = mapping.setdefault(int(value), len(mapping))
            args.output.parent.mkdir(parents=True, exist_ok=True)
            np.save(args.output, labels)
            bitorder = "little" if kind == "packed_little" else "big" if kind == "packed_big" else None
            logical = logical_hash(data, case["features"], bitorder)
            if args.rows is None:
                assert logical == manifest["files"]["int32"]["sha256"], "Input changed"
            state_file = None
            if args.variant.startswith("classix"):
                state_file = args.output.with_suffix(".state.json")
                state_file.write_text(json.dumps(state_fingerprint(model, case["features"], args.variant == "classix_packed"), indent=2)+"\n")
            send("validated", ari=ari, clusters=len(ids), noise_points=noise_points,
                 singletons=int(np.sum(counts == 1)), largest_cluster=int(counts.max()) if len(counts) else 0,
                 aggregation_groups=len(model.splist) if args.variant.startswith("classix") else None,
                 adjacency_bytes=model.Adj.nbytes if args.variant.startswith("classix") else None,
                 adjacency_storage="connection bits + sparse minPts markers" if args.variant.startswith("classix") else None,
                 minpts_marker_count=model.Adj.marker_count if args.variant.startswith("classix") else None,
                 minpts_marker_storage_bytes=model.Adj.marker_storage_bytes if args.variant.startswith("classix") else None,
                 state_file=str(state_file) if state_file else None,
                 canonical_partition_sha256=array_hash(canonical), logical_input_sha256=logical,
                 input_unchanged=True if args.rows is None else None, assignments_file=str(args.output))
        except Exception as error:
            send("validation_error", error_type=type(error).__name__, message=str(error), traceback=traceback.format_exc())


if __name__ == "__main__":
    main()
'''

**`code/quality.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/quality.py'] = r'''"""Adjusted Rand index from the contingency table, using Python integers."""
import numpy as np


def adjusted_rand_index(truth, labels):
    _, a = np.unique(truth, return_inverse=True)
    _, b = np.unique(labels, return_inverse=True)
    table = np.zeros((int(a.max())+1, int(b.max())+1), dtype=np.int64)
    np.add.at(table, (a, b), 1)
    def pairs(counts):
        return sum(int(x)*(int(x)-1)//2 for x in counts.flat)
    total = len(a)*(len(a)-1)//2
    cells, rows, cols = pairs(table), pairs(table.sum(axis=1)), pairs(table.sum(axis=0))
    denominator = total*(rows+cols)-2*rows*cols
    return 1.0 if denominator == 0 else 2*(total*cells-rows*cols)/denominator
'''

**`code/state_checks.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/state_checks.py'] = r'''"""Validation after measurement, using bounded temporary buffers for hashes."""
import hashlib
import json
import numpy as np


def array_hash(array):
    array = np.ascontiguousarray(array)
    return hashlib.sha256(memoryview(array).cast("B")).hexdigest()


def logical_hash(data, n_features, packing=None):
    result = hashlib.sha256()
    for start in range(0, len(data), 256):
        block = data[start:start+256]
        if packing:
            block = np.unpackbits(block, axis=1, count=n_features, bitorder=packing)
        block = np.ascontiguousarray(block, dtype=np.int32)
        result.update(memoryview(block).cast("B"))
    return result.hexdigest()


def state_fingerprint(model, n_features, packed=False):
    fields = ("labels", "aggregation_labels", "splist", "group_sizes", "group_labels",
              "initial_merging_labels", "ind", "unsort_ind", "Adj")
    state = {}
    for name in fields:
        value = getattr(model, name)
        if name == "Adj" and hasattr(value, "iter_row_blocks"):
            digest = hashlib.sha256()
            for block in value.iter_row_blocks(rows=max(1, min(128, (8*1024*1024)//max(1, value.shape[1])))):
                digest.update(memoryview(np.ascontiguousarray(block)).cast("B"))
            state[name] = {"shape": list(value.shape), "dtype": "int8", "sha256": digest.hexdigest()}
            continue
        array = np.asarray(value)
        state[name] = {"shape": list(array.shape), "dtype": str(array.dtype), "sha256": array_hash(array)}
    state["logical_centers"] = logical_hash(model.spdata, n_features, "little" if packed else None)
    state["aggregation_dict"] = hashlib.sha256(json.dumps(model.aggregation_dict, sort_keys=True, default=lambda v: int(v)).encode()).hexdigest()
    return state
'''

**`code/memory_win.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/memory_win.py'] = r'''"""Windows process counters; private commit is distinct from resident memory."""
import ctypes
from ctypes import wintypes


class Counters(ctypes.Structure):
    _fields_ = [("cb", wintypes.DWORD), ("PageFaultCount", wintypes.DWORD)] + [
        (name, ctypes.c_size_t) for name in (
            "PeakWorkingSetSize", "WorkingSetSize", "QuotaPeakPagedPoolUsage",
            "QuotaPagedPoolUsage", "QuotaPeakNonPagedPoolUsage",
            "QuotaNonPagedPoolUsage", "PagefileUsage", "PeakPagefileUsage", "PrivateUsage")]


kernel = ctypes.WinDLL("kernel32", use_last_error=True)
psapi = ctypes.WinDLL("psapi", use_last_error=True)
kernel.GetCurrentProcess.restype = wintypes.HANDLE
kernel.OpenProcess.argtypes = [wintypes.DWORD, wintypes.BOOL, wintypes.DWORD]
kernel.OpenProcess.restype = wintypes.HANDLE
kernel.CloseHandle.argtypes = [wintypes.HANDLE]
kernel.CloseHandle.restype = wintypes.BOOL
psapi.GetProcessMemoryInfo.argtypes = [wintypes.HANDLE, ctypes.POINTER(Counters), wintypes.DWORD]
psapi.GetProcessMemoryInfo.restype = wintypes.BOOL


def open_process(pid):
    handle = kernel.OpenProcess(0x1000, False, pid)
    if not handle:
        raise ctypes.WinError(ctypes.get_last_error())
    return handle


def close_process(handle):
    kernel.CloseHandle(handle)


def snapshot(handle=None):
    counters = Counters()
    counters.cb = ctypes.sizeof(counters)
    if not psapi.GetProcessMemoryInfo(handle or kernel.GetCurrentProcess(), ctypes.byref(counters), counters.cb):
        raise ctypes.WinError(ctypes.get_last_error())
    return {
        "private_commit_bytes": counters.PrivateUsage,
        "working_set_bytes": counters.WorkingSetSize,
        "lifetime_peak_private_commit_bytes": counters.PeakPagefileUsage,
        "lifetime_peak_working_set_bytes": counters.PeakWorkingSetSize,
    }
'''

**`code/job_limit.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/job_limit.py'] = r'''"""Windows job-object commit limit for a paused benchmark process tree.

The controller must start its child with CREATE_NO_WINDOW and keep the actual
worker paused on a READY/GO handshake until assign_job returns. Pass the actual
worker PID from READY, not the outer Windows venv-launcher PID: an already-running
launcher can use a separate nested job for its child, preventing both processes
from being added to a common new job. Only that explicitly identified worker is
assigned; subsequently created children inherit the job. The worker must have
no pre-existing subprocesses that require bounding. Process-tree discovery is
deliberately disabled: Windows parent PID snapshots can identify unrelated old
processes after a parent PID is reused. No breakaway or host-wide limit is
enabled. The outer launcher holds no benchmark arrays.

The memory limit concerns committed memory, not working-set size. Assignment
does not retroactively reject allocations made before joining the job; attach
before substantial allocations when a strict whole-run bound is needed.

API:
    job = assign_job(pid=ready["worker_pid"], max_bytes=4 * 1024**3)
    ... send GO, wait for the child ...
    close_job(job)  # also kills any remaining assigned descendants

Alternatively pass an existing process handle as the first positional argument.
An integer positional argument is a HANDLE, never interpreted as a PID.

References:
https://learn.microsoft.com/en-us/windows/win32/api/winnt/ns-winnt-jobobject_extended_limit_information
https://learn.microsoft.com/en-us/windows/win32/api/jobapi2/nf-jobapi2-assignprocesstojobobject
"""
import ctypes
from ctypes import wintypes
import os

if os.name != "nt":
    raise OSError("This benchmark resource limiter requires Windows")

_kernel = ctypes.WinDLL("kernel32", use_last_error=True)
_SIZE_T = ctypes.c_size_t
_ULONG_PTR = ctypes.c_size_t
_PROCESS_RIGHTS = 0x0001 | 0x0100 | 0x1000  # TERMINATE, SET_QUOTA, QUERY_LIMITED
_JOB_INFO_EXTENDED = 9
_JOB_OBJECT_LIMIT_JOB_MEMORY = 0x00000200
_JOB_OBJECT_LIMIT_KILL_ON_JOB_CLOSE = 0x00002000


class _BasicLimit(ctypes.Structure):
    _fields_ = [
        ("PerProcessUserTimeLimit", ctypes.c_int64),
        ("PerJobUserTimeLimit", ctypes.c_int64),
        ("LimitFlags", wintypes.DWORD),
        ("MinimumWorkingSetSize", _SIZE_T),
        ("MaximumWorkingSetSize", _SIZE_T),
        ("ActiveProcessLimit", wintypes.DWORD),
        ("Affinity", _ULONG_PTR),
        ("PriorityClass", wintypes.DWORD),
        ("SchedulingClass", wintypes.DWORD),
    ]


class _IoCounters(ctypes.Structure):
    _fields_ = [(name, ctypes.c_uint64) for name in (
        "ReadOperationCount", "WriteOperationCount", "OtherOperationCount",
        "ReadTransferCount", "WriteTransferCount", "OtherTransferCount",
    )]


class _ExtendedLimit(ctypes.Structure):
    _fields_ = [
        ("BasicLimitInformation", _BasicLimit),
        ("IoInfo", _IoCounters),
        ("ProcessMemoryLimit", _SIZE_T),
        ("JobMemoryLimit", _SIZE_T),
        ("PeakProcessMemoryUsed", _SIZE_T),
        ("PeakJobMemoryUsed", _SIZE_T),
    ]


_kernel.CreateJobObjectW.argtypes = [ctypes.c_void_p, wintypes.LPCWSTR]
_kernel.CreateJobObjectW.restype = wintypes.HANDLE
_kernel.SetInformationJobObject.argtypes = [wintypes.HANDLE, ctypes.c_int, ctypes.c_void_p, wintypes.DWORD]
_kernel.SetInformationJobObject.restype = wintypes.BOOL
_kernel.QueryInformationJobObject.argtypes = [wintypes.HANDLE, ctypes.c_int, ctypes.c_void_p, wintypes.DWORD, ctypes.c_void_p]
_kernel.QueryInformationJobObject.restype = wintypes.BOOL
_kernel.AssignProcessToJobObject.argtypes = [wintypes.HANDLE, wintypes.HANDLE]
_kernel.AssignProcessToJobObject.restype = wintypes.BOOL
_kernel.IsProcessInJob.argtypes = [wintypes.HANDLE, wintypes.HANDLE, ctypes.POINTER(wintypes.BOOL)]
_kernel.IsProcessInJob.restype = wintypes.BOOL
_kernel.OpenProcess.argtypes = [wintypes.DWORD, wintypes.BOOL, wintypes.DWORD]
_kernel.OpenProcess.restype = wintypes.HANDLE
_kernel.GetProcessId.argtypes = [wintypes.HANDLE]
_kernel.GetProcessId.restype = wintypes.DWORD
_kernel.CloseHandle.argtypes = [wintypes.HANDLE]
_kernel.CloseHandle.restype = wintypes.BOOL


def _failure(operation):
    code = ctypes.get_last_error()
    detail = ctypes.FormatError(code).strip()
    return OSError(code, f"{operation}: {detail}; benchmark must not proceed without its job limit")


def _open_process(pid):
    handle = _kernel.OpenProcess(_PROCESS_RIGHTS, False, pid)
    if not handle:
        raise _failure(f"OpenProcess({pid})")
    return handle


def process_in_job(job, *, pid):
    """Check a READY worker PID is contained before sending GO."""
    handle = _open_process(pid)
    try:
        present = wintypes.BOOL()
        if not _kernel.IsProcessInJob(handle, job, ctypes.byref(present)):
            raise _failure(f"IsProcessInJob({pid})")
        return bool(present.value)
    finally:
        _kernel.CloseHandle(handle)


def assign_job(process_handle=None, max_bytes=4 * 1024**3, *, pid=None,
               include_existing_descendants=False):
    """Bound the explicit worker; its future children inherit the same job.

    Exactly one of process_handle and pid is required. The worker must have no
    existing subprocesses requiring this limit. include_existing_descendants=True
    is rejected before opening/assigning any process; PID-only tree discovery is
    unsafe when Windows reuses PIDs. A failure, including incompatible
    enclosing/nested jobs, raises; partially assigned processes are terminated.
    The caller owns only the returned job handle, not a passed process handle.
    """
    if (process_handle is None) == (pid is None):
        raise ValueError("Pass exactly one process_handle or keyword pid")
    if include_existing_descendants is not False:
        raise ValueError("Existing-descendant discovery is disabled; pass the actual READY worker with no pre-existing subprocesses")
    if not isinstance(max_bytes, int) or not 0 < max_bytes <= ctypes.c_size_t(-1).value:
        raise ValueError("max_bytes must be a positive SIZE_T integer")
    own_process_handle = process_handle is None
    if own_process_handle:
        process_handle = _open_process(pid)
    job = None
    try:
        root_pid = int(_kernel.GetProcessId(process_handle))
        if not root_pid:
            raise _failure("GetProcessId")
        if root_pid == os.getpid():
            raise ValueError("Refusing to assign the controller itself to a kill-on-close job")
        job = _kernel.CreateJobObjectW(None, None)
        if not job:
            raise _failure("CreateJobObjectW")
        limits = _ExtendedLimit()
        limits.BasicLimitInformation.LimitFlags = (
            _JOB_OBJECT_LIMIT_JOB_MEMORY | _JOB_OBJECT_LIMIT_KILL_ON_JOB_CLOSE
        )
        limits.JobMemoryLimit = max_bytes
        if not _kernel.SetInformationJobObject(job, _JOB_INFO_EXTENDED, ctypes.byref(limits), ctypes.sizeof(limits)):
            raise _failure("SetInformationJobObject")
        if not _kernel.AssignProcessToJobObject(job, process_handle):
            raise _failure(f"AssignProcessToJobObject({root_pid}); nested-job incompatibility is possible")
        # No enumeration: only the READY process is assigned. Windows ensures
        # subsequently created children inherit the same aggregate job limit.
        return job
    except BaseException:
        if job:
            _kernel.CloseHandle(job)
        raise
    finally:
        if own_process_handle:
            _kernel.CloseHandle(process_handle)


def job_snapshot(job):
    """Return diagnostic Windows job counters.

    Peak counters can include a subsequently denied allocation attempt, so they
    may exceed the configured quota; do not interpret them as resident memory.
    """
    limits = _ExtendedLimit()
    if not _kernel.QueryInformationJobObject(job, _JOB_INFO_EXTENDED, ctypes.byref(limits), ctypes.sizeof(limits), None):
        raise _failure("QueryInformationJobObject")
    return {
        "memory_limit_bytes": int(limits.JobMemoryLimit),
        "peak_job_commit_bytes": int(limits.PeakJobMemoryUsed),
        "peak_process_commit_bytes": int(limits.PeakProcessMemoryUsed),
        "limit_flags": int(limits.BasicLimitInformation.LimitFlags),
    }


def close_job(job):
    """Close a valid job handle; any still-running member processes are killed."""
    if job and not _kernel.CloseHandle(job):
        raise _failure("CloseHandle(job)")
'''

**`code/classix/classix_t_packed_adjbits.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/classix/classix_t_packed_adjbits.py'] = r'''# Temporary storage-only prototype; source and focused diff are recorded.
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent))
from packed_adjacency import PackedAdjacency
#%%
import numpy as np
from time import time
from tqdm import tqdm
from copy import deepcopy
from collections import deque
import scipy.sparse as sparse
from scipy import sparse
from fractions import Fraction
from packed_native import packed_submatxvec, row_popcounts

class CLASSIX_T_PACKED:
    def __init__(self, sorting="popcount", radius=0.3, minPts=1, group_merging="tanimoto_distance", norm=True, mergeScale=1.4, post_alloc=True, mergeTinyGroups=True, verbose=1, short_log_form=True, use_spmv=True):
        self.group_merging = group_merging
        self.mergeScale = mergeScale
        self.mergeTinyGroups = mergeTinyGroups
        self.sorting = sorting
        self.radius = radius
        self.minPts = minPts
        self.norm = norm

        self.post_alloc = post_alloc
        self.mergeTinyGroups = mergeTinyGroups
        self.verbose = verbose
        self.truncate = short_log_form



    def __str__(self):
        return f"CLASSIX_T(sorting={self.sorting}, radius={self.radius}, minPts={self.minPts}, group_merging={self.group_merging}, mergeScale={self.mergeScale}, mergeTinyGroups={self.mergeTinyGroups})"

    def __repr__(self):
        return f"CLASSIX_T(sorting={self.sorting}, radius={self.radius}, minPts={self.minPts}, group_merging={self.group_merging}, mergeScale={self.mergeScale}, mergeTinyGroups={self.mergeTinyGroups})"

    def fit(self, data, r=None, mergeScale=None, minPts=None):
        total_time = time()
        if r is not None:
            self.radius = r

        if mergeScale is not None:
            self.mergeScale = mergeScale

        if minPts is not None:
            self.minPts = minPts

        # Input is np.packbits(binary_rows, axis=1, bitorder='little').
        # Any padding bits in the final byte must be zero.
        data = np.asarray(data)
        if data.ndim != 2 or data.dtype != np.uint8:
            raise ValueError("Expected a 2D uint8 array of packed binary fingerprints")
        if not data.shape[0] or not data.shape[1]:
            raise ValueError("Packed input must have at least one row and one byte")
        data = np.ascontiguousarray(data)
        n, fdim = data.shape
        sort_vals = row_popcounts(data).astype(np.int64, copy=False)

        # print(sort_vals)

        self.ind = np.argsort(sort_vals, kind='stable')
        self.unsort_ind = np.argsort(self.ind, kind='stable')
        sort_vals = sort_vals[self.ind] 
        data = data[self.ind,:] # sort data

        # print(data)

        print("\nOWN AGGREGATION")


        ips_time = 0.0
        search_time = 0.0
        loop_time = 0.0
        conversion_time = 0.0

        st = time()
        lab = 0

        # labels of the data points. Initially all are -1 to keep track of unallocated points
        self.labels = np.array([-1]*n)
        nr_dist = 0
        # Indices of group starting pts, equivalent of the gc list in the original code
        self.splist = []
        rhs = 1/(1-self.radius)+1; # rhs of tanimoto distance inequality
        rhsi = 1/rhs

        rhs_n = 2 - self.radius
        rhs_d = 1 - self.radius

        # Do a small check to see if radius is a rational number:
        # If it is, we can use the rhs_n and rhs_d values to avoid floating point errors!

        self.group_sizes = [] # group sizes
        self.group_starting_pts = [] # group starting pts in the sorted array

        # Packed rows are used directly: no CSR construction is required.
        self.aggregation_dict = {}


        # Aggregation
        for i in tqdm(range(n)):

            if self.labels[i] >= 0:
                continue

            clustc = data[i,:]

            # What if we take the starting point as a slice of the sparse array instead?
            # clustc = datas[i,:]
            #
            self.labels[i] = lab

            # num_group = 1
            self.splist.append(i)
            self.group_sizes.append(1)

            search_radius = sort_vals[i]/(1-self.radius) # Only the right side limit is needed

            st1 = time()
            last_j = np.searchsorted(sort_vals, search_radius, side='right')
            self.aggregation_dict[i] = last_j

            search_time += time() - st1

            st1 = time()

            # ips = data[i+1:last_j,:]@clustc.T
            ips = packed_submatxvec(data, i+1, last_j, clustc)

            ips_time += time() - st1
            # Let's make the product with rhsi times sort_vals the condition.
            # lhs = sort_vals[i+1:last_j]

            nr_dist += last_j - i - 1

            st_c = time()

            vec = ips/(sort_vals[i+1:last_j]+sort_vals[i])
            conversion_time += time() - st_c

            st1 = time()

            """
            Create a custom C function that takes in a range of indices, a vector, and a scalar value.
            If the value of the vector at index i is greater than 0, it will check if the value is greater than the scalar value,

            Maybe just try boolean masks first?
            Note: self.labels has size = n
                  vec has size = last_j - i - 1

            The boolean mask of vec needs to padded with False values to match the size of self.labels

            Then we can take the intersection of the two boolean masks.

            ```
            vec_mask = vec >= rhsi
            assigned_mask = self.labels >= 0

            ```

            """

            notAssigned = self.labels < 0
            vec_mask = vec >= rhsi
            vec_mask = np.pad(vec_mask, (i+1, n - last_j), 'constant', constant_values=False)
            reassignMask = np.logical_and(notAssigned, vec_mask)
            self.group_sizes[-1] += np.sum(reassignMask)
            self.labels[reassignMask] = lab


            # for j in range(i+1, last_j):
            #     if self.labels[j] >= 0:
            #         continue

            #     # if sort_vals[j]+sort_vals[i]<=rhs*ips[j-i-1]:
            #     if (sort_vals[j]+sort_vals[i])*rhs_d<=rhs_n*ips[j-i-1]: # No division!
            #         # num_group += 1
            #         self.labels[j] = lab
            #         self.group_sizes[-1] += 1

            loop_time += time() - st1

            lab += 1

        print("time for ips:", ips_time)
        print("time for search:", search_time)
        print("time for conversion:", conversion_time)
        print("time for loop:", loop_time)
        print("nr_dist:", nr_dist)

        self.aggregation_labels = deepcopy(np.array(self.labels))
        self.search_time = search_time
        self.ips_time = ips_time
        self.aggregation_time = time()-st

        # merging
        st = time()

        self.spdata = data[self.splist,:]
        # Group representatives also remain packed.
        spdata_len = len(self.splist)
        self.group_centers = self.spdata

        # Scores of the group starting points
        sort_vals_sp = sort_vals[self.splist]

        # Group labels of the group starting points
        label_sp = self.labels[self.splist]
        scale = self.mergeScale




        self.Adj = PackedAdjacency(len(self.splist))

        t1 = 0.0
        t2 = 0.0
        t3 = 0.0
        minPts = self.minPts

        for i in tqdm(range(len(self.splist))):
            if not self.mergeTinyGroups:
                if self.group_sizes[i] < minPts:
                    continue

            xi = self.spdata[i, :]
            search_radius = sort_vals_sp[i]/(1-self.mergeScale*self.radius)

            st1 = time()
            last_j = np.searchsorted(sort_vals_sp, search_radius, side='right')
            first_j = i
            ips = packed_submatxvec(self.spdata, first_j, last_j, xi)
            # ips = np.matmul(self.spdata[first_j:last_j,:],xi.T)
            tanimoto_distance = 1 - ips/(sort_vals_sp[i] + sort_vals_sp[first_j:last_j] - ips)
            # tanimoto_distance = 1 - tanimoto_score
            t1 += time() - st1

            st2 = time()
            inds = np.where(tanimoto_distance <= scale*self.radius)

            if not self.mergeTinyGroups:
                inds = inds[0][self.group_sizes[inds[0]] >= minPts]
                inds = i + inds

            else:
                inds = i + inds[0]



            self.Adj[i, inds] = 1
            self.Adj[inds, i] = 1

            spl = np.unique(label_sp[ inds ])

            minlab = np.min(spl)

            t2 += time() - st2

            st3 = time()
            for label in spl:
                label_sp[label_sp==label] = minlab
            t3 += time() - st3
            # continue

        self.initial_merging_labels = deepcopy(label_sp)

        print("  merging time:", time()-st)
        self.merging_time = time()-st

        # Cluster redistribution
        print(" minPts Merging")
        st = time()
        ############################################################################################################

        labels_sp_copy = deepcopy(label_sp)
        ul = np.unique(label_sp)
        cs = np.zeros(len(ul))
        self.group_sizes = np.array(self.group_sizes)
        for i in range(len(ul)):
            id = np.where(label_sp==ul[i])
            label_sp[id] = i
            cs[i] = np.sum(self.group_sizes[id])

        small_clusters = np.where(cs<self.minPts)[0]

        # print("small clusters", small_clusters)

        labels_sp_copy_2 = deepcopy(label_sp)

        for i in small_clusters:
            ii = np.where(labels_sp_copy_2==i)[0]
            for iii in ii:
                xi = self.spdata[iii, :]
                # d = Tanimoto distances to all other group centers
                # ips = np.matmul(self.spdata, xi.T)
                ips = packed_submatxvec(self.spdata, 0, spdata_len, xi)
                d = 1 - ips/(sort_vals_sp[iii] + sort_vals_sp - ips)

                o = np.argsort(d, kind='stable')
                # merge with the closest group that has more than minPts
                for j in o:
                    if cs[labels_sp_copy_2[j]]>=minPts:
                        label_sp[iii] = labels_sp_copy_2[j]
                        # Instead of having a separate Adjanceny matrix for the points merged with minPts criteria
                        # we can just update the old Adjacency matrix here. The issue is that the distance matrix will
                        # have some weird discrepancies.
                        self.Adj[iii, j] = 2
                        self.Adj[j, iii] = 2
                        break

        ul = np.unique(label_sp)
        cs = np.zeros(len(ul))
        for i in range(len(ul)):
            id = np.where(label_sp==ul[i])
            label_sp[id] = i
            cs[i] = np.sum(self.group_sizes[id])

        # print("final cluster sizes", cs)

        print("  minPts merging time:", time()-st)

        for idx, label in enumerate(self.labels):
            self.labels[idx] = label_sp[label]

        ############################################################################################################

        self.labels = self.labels[self.unsort_ind]
        # print(self.labels)
        self.aggregation_labels = self.aggregation_labels[self.unsort_ind]
        self.group_labels = label_sp
        self.group_centre_pts = self.spdata
        self.group_centers = self.splist
        print("Total time:", time()-total_time)
        return self

    def explain(self, ind1=None, ind2=None):
        if ind1 is None and ind2 is None:
            # If there are no specific points to explain, print the general information
            print(f"The data was clustered into {len(self.group_labels)} groups. These were further merged into {len(np.unique(self.group_labels))} clusters.")

        if ind1 is not None and ind2 is None:
            # If only one index is provided, print the cluster information for that index
            print(f"The data point at index {ind1} was assigned to cluster {self.labels[ind1]}")

        if ind1 is not None and ind2 is not None:
            # If two indices are provided, print the cluster information
            if self.labels[ind1] == self.labels[ind2]:
                # If the two data points belong to the same cluster, check if they are in the same group
                print(f"The data points at indices {ind1} and {ind2} belong to the same cluster.")
                agg_label_1 = self.aggregation_labels[ind1]
                agg_label_2 = self.aggregation_labels[ind2]
                if agg_label_1 == agg_label_2:
                    # If the two data points are in the same group, print the information
                    print(f"The data points at indices {ind1} and {ind2} are in the same group {agg_label_1}.")
                    return None

                else:
                    # If the two data points are in different groups, find the shortest connection path between the two groups
                    print(f"The data points at indices {ind1} and {ind2} are in different groups.")
                    connected_path = self.bfs_shortest_path(self.Adj, agg_label_1, agg_label_2)
                    if connected_path is not None:
                        # If a connection path is found, print the path
                        print(f'The connections between {ind1} and {ind2} are via this path: {connected_path[0]} ', end="")
                        for k in range(len(connected_path)-1):
                            if self.Adj[connected_path[k]-1, connected_path[k+1]-1] == 2:
                                print(f'(minPts) -> {connected_path[k+1]}', end="")
                            else:
                                print(f' -> {connected_path[k+1]}', end="")

                        return connected_path

                    else:
                        # If no connection path is found, print that there is no connection, there must be something wrong with the code
                        print(f"No connection path found between {ind1} and {ind2}, although they belong to different groups in the same cluster. Please check the program for bugs!")
                        return None

            else:
                print(f"The data points at indices {ind1} and {ind2} belong to different clusters.")


    def bfs_shortest_path(self, adj_matrix, start, goal):
        # Convert start and goal from 1-based to 0-based indices
        start -= 1
        goal -= 1

        # Initialize the queue with the start node and a path containing only the start node
        queue = deque([(start, [start])])

        # Keep track of visited nodes to avoid cycles
        visited = set()

        while queue:
            # Dequeue a node and the path that led to it
            current_node, path = queue.popleft()

            # If the current node is the goal, return the path
            if current_node == goal:
                return [node + 1 for node in path]  # Convert back to 1-based indices

            # Mark the current node as visited
            visited.add(current_node)

            # Enqueue all unvisited neighbors
            for neighbor, connected in enumerate(adj_matrix[current_node]):
                if connected and neighbor not in visited:
                    queue.append((neighbor, path + [neighbor]))

        return None
# %%
'''

**`code/classix/classix_t_reference_adjbits.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/classix/classix_t_reference_adjbits.py'] = r'''# Temporary storage-only prototype; source and focused diff are recorded.
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent))
from packed_adjacency import PackedAdjacency
#%%
import numpy as np
from time import time
from tqdm import tqdm
from copy import deepcopy
from collections import deque
import scipy.sparse as sparse
from scipy import sparse
from fractions import Fraction
from packed_native import original_spsubmatxvec as spsubmatxvec

class CLASSIX_T:
    def __init__(self, sorting="popcount", radius=0.3, minPts=1, group_merging="tanimoto_distance", norm=True, mergeScale=1.4, post_alloc=True, mergeTinyGroups=True, verbose=1, short_log_form=True, use_spmv=True):
        self.group_merging = group_merging
        self.mergeScale = mergeScale
        self.mergeTinyGroups = mergeTinyGroups
        self.sorting = sorting
        self.radius = radius
        self.minPts = minPts
        self.norm = norm

        self.post_alloc = post_alloc
        self.mergeTinyGroups = mergeTinyGroups
        self.verbose = verbose
        self.truncate = short_log_form



    def __str__(self):
        return f"CLASSIX_T(sorting={self.sorting}, radius={self.radius}, minPts={self.minPts}, group_merging={self.group_merging}, mergeScale={self.mergeScale}, mergeTinyGroups={self.mergeTinyGroups})"

    def __repr__(self):
        return f"CLASSIX_T(sorting={self.sorting}, radius={self.radius}, minPts={self.minPts}, group_merging={self.group_merging}, mergeScale={self.mergeScale}, mergeTinyGroups={self.mergeTinyGroups})"

    def fit(self, data, r=None, mergeScale=None, minPts=None):
        total_time = time()
        if r is not None:
            self.radius = r

        if mergeScale is not None:
            self.mergeScale = mergeScale

        if minPts is not None:
            self.minPts = minPts

        n, fdim = data.shape
        sort_vals = np.sum(data, axis=1)

        # print(sort_vals)

        self.ind = np.argsort(sort_vals, kind='stable')
        self.unsort_ind = np.argsort(self.ind, kind='stable')
        sort_vals = sort_vals[self.ind] 
        data = data[self.ind,:] # sort data

        # print(data)

        print("\nOWN AGGREGATION")


        ips_time = 0.0
        search_time = 0.0
        loop_time = 0.0
        conversion_time = 0.0

        st = time()
        lab = 0

        # labels of the data points. Initially all are -1 to keep track of unallocated points
        self.labels = np.array([-1]*n)
        nr_dist = 0
        # Indices of group starting pts, equivalent of the gc list in the original code
        self.splist = []
        rhs = 1/(1-self.radius)+1; # rhs of tanimoto distance inequality
        rhsi = 1/rhs

        rhs_n = 2 - self.radius
        rhs_d = 1 - self.radius

        # Do a small check to see if radius is a rational number:
        # If it is, we can use the rhs_n and rhs_d values to avoid floating point errors!

        self.group_sizes = [] # group sizes
        self.group_starting_pts = [] # group starting pts in the sorted array

        datas = sparse.csr_matrix(data)
        datas_data = datas.data
        datas_indices = datas.indices
        datas_indptr = datas.indptr
        self.aggregation_dict = {}


        # Aggregation
        for i in tqdm(range(n)):

            if self.labels[i] >= 0:
                continue

            clustc = data[i,:]

            # What if we take the starting point as a slice of the sparse array instead?
            # clustc = datas[i,:]
            #
            self.labels[i] = lab

            # num_group = 1
            self.splist.append(i)
            self.group_sizes.append(1)

            search_radius = sort_vals[i]/(1-self.radius) # Only the right side limit is needed

            st1 = time()
            last_j = np.searchsorted(sort_vals, search_radius, side='right')
            self.aggregation_dict[i] = last_j

            search_time += time() - st1

            st1 = time()

            # ips = data[i+1:last_j,:]@clustc.T
            ips = spsubmatxvec(datas_data, datas_indptr, datas_indices, i+1, last_j, clustc)

            ips_time += time() - st1
            # Let's make the product with rhsi times sort_vals the condition.
            # lhs = sort_vals[i+1:last_j]

            nr_dist += last_j - i - 1

            st_c = time()

            vec = ips/(sort_vals[i+1:last_j]+sort_vals[i])
            conversion_time += time() - st_c

            st1 = time()

            """
            Create a custom C function that takes in a range of indices, a vector, and a scalar value.
            If the value of the vector at index i is greater than 0, it will check if the value is greater than the scalar value,

            Maybe just try boolean masks first?
            Note: self.labels has size = n
                  vec has size = last_j - i - 1

            The boolean mask of vec needs to padded with False values to match the size of self.labels

            Then we can take the intersection of the two boolean masks.

            ```
            vec_mask = vec >= rhsi
            assigned_mask = self.labels >= 0

            ```

            """

            notAssigned = self.labels < 0
            vec_mask = vec >= rhsi
            vec_mask = np.pad(vec_mask, (i+1, n - last_j), 'constant', constant_values=False)
            reassignMask = np.logical_and(notAssigned, vec_mask)
            self.group_sizes[-1] += np.sum(reassignMask)
            self.labels[reassignMask] = lab


            # for j in range(i+1, last_j):
            #     if self.labels[j] >= 0:
            #         continue

            #     # if sort_vals[j]+sort_vals[i]<=rhs*ips[j-i-1]:
            #     if (sort_vals[j]+sort_vals[i])*rhs_d<=rhs_n*ips[j-i-1]: # No division!
            #         # num_group += 1
            #         self.labels[j] = lab
            #         self.group_sizes[-1] += 1

            loop_time += time() - st1

            lab += 1

        print("time for ips:", ips_time)
        print("time for search:", search_time)
        print("time for conversion:", conversion_time)
        print("time for loop:", loop_time)
        print("nr_dist:", nr_dist)

        self.aggregation_labels = deepcopy(np.array(self.labels))
        self.search_time = search_time
        self.ips_time = ips_time
        self.aggregation_time = time()-st

        # merging
        st = time()

        self.spdata = data[self.splist,:]
        spdatas = sparse.csr_matrix(self.spdata)
        spdatas_data = spdatas.data
        spdatas_indices = spdatas.indices
        spdatas_indptr = spdatas.indptr
        spdata_len = len(self.splist)
        self.group_centers = self.spdata

        # Scores of the group starting points
        sort_vals_sp = sort_vals[self.splist]

        # Group labels of the group starting points
        label_sp = self.labels[self.splist]
        scale = self.mergeScale




        self.Adj = PackedAdjacency(len(self.splist))

        t1 = 0.0
        t2 = 0.0
        t3 = 0.0
        minPts = self.minPts

        for i in tqdm(range(len(self.splist))):
            if not self.mergeTinyGroups:
                if self.group_sizes[i] < minPts:
                    continue

            xi = self.spdata[i, :]
            search_radius = sort_vals_sp[i]/(1-self.mergeScale*self.radius)

            st1 = time()
            last_j = np.searchsorted(sort_vals_sp, search_radius, side='right')
            first_j = i
            ips = spsubmatxvec(spdatas_data, spdatas_indptr, spdatas_indices, first_j, last_j, xi)
            # ips = np.matmul(self.spdata[first_j:last_j,:],xi.T)
            tanimoto_distance = 1 - ips/(sort_vals_sp[i] + sort_vals_sp[first_j:last_j] - ips)
            # tanimoto_distance = 1 - tanimoto_score
            t1 += time() - st1

            st2 = time()
            inds = np.where(tanimoto_distance <= scale*self.radius)

            if not self.mergeTinyGroups:
                inds = inds[0][self.group_sizes[inds[0]] >= minPts]
                inds = i + inds

            else:
                inds = i + inds[0]



            self.Adj[i, inds] = 1
            self.Adj[inds, i] = 1

            spl = np.unique(label_sp[ inds ])

            minlab = np.min(spl)

            t2 += time() - st2

            st3 = time()
            for label in spl:
                label_sp[label_sp==label] = minlab
            t3 += time() - st3
            # continue

        self.initial_merging_labels = deepcopy(label_sp)

        print("  merging time:", time()-st)
        self.merging_time = time()-st

        # Cluster redistribution
        print(" minPts Merging")
        st = time()
        ############################################################################################################

        labels_sp_copy = deepcopy(label_sp)
        ul = np.unique(label_sp)
        cs = np.zeros(len(ul))
        self.group_sizes = np.array(self.group_sizes)
        for i in range(len(ul)):
            id = np.where(label_sp==ul[i])
            label_sp[id] = i
            cs[i] = np.sum(self.group_sizes[id])

        small_clusters = np.where(cs<self.minPts)[0]

        # print("small clusters", small_clusters)

        labels_sp_copy_2 = deepcopy(label_sp)

        for i in small_clusters:
            ii = np.where(labels_sp_copy_2==i)[0]
            for iii in ii:
                xi = self.spdata[iii, :]
                # d = Tanimoto distances to all other group centers
                # ips = np.matmul(self.spdata, xi.T)
                ips = spsubmatxvec(spdatas_data, spdatas_indptr, spdatas_indices, 0, spdata_len, xi)
                d = 1 - ips/(sort_vals_sp[iii] + sort_vals_sp - ips)

                o = np.argsort(d, kind='stable')
                # merge with the closest group that has more than minPts
                for j in o:
                    if cs[labels_sp_copy_2[j]]>=minPts:
                        label_sp[iii] = labels_sp_copy_2[j]
                        # Instead of having a separate Adjanceny matrix for the points merged with minPts criteria
                        # we can just update the old Adjacency matrix here. The issue is that the distance matrix will
                        # have some weird discrepancies.
                        self.Adj[iii, j] = 2
                        self.Adj[j, iii] = 2
                        break

        ul = np.unique(label_sp)
        cs = np.zeros(len(ul))
        for i in range(len(ul)):
            id = np.where(label_sp==ul[i])
            label_sp[id] = i
            cs[i] = np.sum(self.group_sizes[id])

        # print("final cluster sizes", cs)

        print("  minPts merging time:", time()-st)

        for idx, label in enumerate(self.labels):
            self.labels[idx] = label_sp[label]

        ############################################################################################################

        self.labels = self.labels[self.unsort_ind]
        # print(self.labels)
        self.aggregation_labels = self.aggregation_labels[self.unsort_ind]
        self.group_labels = label_sp
        self.group_centre_pts = self.spdata
        self.group_centers = self.splist
        print("Total time:", time()-total_time)
        return self

    def explain(self, ind1=None, ind2=None):
        if ind1 is None and ind2 is None:
            # If there are no specific points to explain, print the general information
            print(f"The data was clustered into {len(self.group_labels)} groups. These were further merged into {len(np.unique(self.group_labels))} clusters.")

        if ind1 is not None and ind2 is None:
            # If only one index is provided, print the cluster information for that index
            print(f"The data point at index {ind1} was assigned to cluster {self.labels[ind1]}")

        if ind1 is not None and ind2 is not None:
            # If two indices are provided, print the cluster information
            if self.labels[ind1] == self.labels[ind2]:
                # If the two data points belong to the same cluster, check if they are in the same group
                print(f"The data points at indices {ind1} and {ind2} belong to the same cluster.")
                agg_label_1 = self.aggregation_labels[ind1]
                agg_label_2 = self.aggregation_labels[ind2]
                if agg_label_1 == agg_label_2:
                    # If the two data points are in the same group, print the information
                    print(f"The data points at indices {ind1} and {ind2} are in the same group {agg_label_1}.")
                    return None

                else:
                    # If the two data points are in different groups, find the shortest connection path between the two groups
                    print(f"The data points at indices {ind1} and {ind2} are in different groups.")
                    connected_path = self.bfs_shortest_path(self.Adj, agg_label_1, agg_label_2)
                    if connected_path is not None:
                        # If a connection path is found, print the path
                        print(f'The connections between {ind1} and {ind2} are via this path: {connected_path[0]} ', end="")
                        for k in range(len(connected_path)-1):
                            if self.Adj[connected_path[k]-1, connected_path[k+1]-1] == 2:
                                print(f'(minPts) -> {connected_path[k+1]}', end="")
                            else:
                                print(f' -> {connected_path[k+1]}', end="")

                        return connected_path

                    else:
                        # If no connection path is found, print that there is no connection, there must be something wrong with the code
                        print(f"No connection path found between {ind1} and {ind2}, although they belong to different groups in the same cluster. Please check the program for bugs!")
                        return None

            else:
                print(f"The data points at indices {ind1} and {ind2} belong to different clusters.")


    def bfs_shortest_path(self, adj_matrix, start, goal):
        # Convert start and goal from 1-based to 0-based indices
        start -= 1
        goal -= 1

        # Initialize the queue with the start node and a path containing only the start node
        queue = deque([(start, [start])])

        # Keep track of visited nodes to avoid cycles
        visited = set()

        while queue:
            # Dequeue a node and the path that led to it
            current_node, path = queue.popleft()

            # If the current node is the goal, return the path
            if current_node == goal:
                return [node + 1 for node in path]  # Convert back to 1-based indices

            # Mark the current node as visited
            visited.add(current_node)

            # Enqueue all unvisited neighbors
            for neighbor, connected in enumerate(adj_matrix[current_node]):
                if connected and neighbor not in visited:
                    queue.append((neighbor, path + [neighbor]))

        return None
# %%
'''

**`code/classix/packed_adjacency.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/classix/packed_adjacency.py'] = r'''"""CLASSIX adjacency with packed connections and sparse minPts edge markers.

Each logical entry is 0, 1, or 2. One bit records whether an edge exists; sparse
row-to-column sets mark edges whose logical value is 2. The connection buffer is
exactly G * ceil(G / 8) bytes for a G by G matrix. Sparse marker overhead is
separate and exposed as a Python-object size estimate. No dense graph is
allocated by construction, reads of individual edges, or writes. Decoding a row
returns an independent int8 array, not a writable view.

Supported writes match CLASSIX's existing operations:
    graph[i, columns] = 1
    graph[rows, j] = 1
    graph[i, j] = 2
The same forms accept 0 to clear edges. Repeated indices and different bits in
one byte are handled correctly using unbuffered NumPy ufunc.at updates.

Call iter_row_blocks(rows=...) for bounded logical hashing or validation.
Implicit conversion of the whole graph to a dense NumPy array is prohibited.
"""
import operator
import sys

import numpy as np


def _integer(value, name):
    if isinstance(value, (bool, np.bool_)):
        raise TypeError(f"{name} must be an integer, not a boolean")
    try:
        return operator.index(value)
    except TypeError as error:
        raise TypeError(f"{name} must be an integer") from error


class PackedAdjacency:
    """Logical int8 adjacency: one bit/entry plus sparse markers for value 2."""

    __slots__ = ("_n", "_row_bytes", "_connections", "_minpts")
    dtype = np.dtype(np.int8)
    ndim = 2

    def __init__(self, groups):
        groups = _integer(groups, "groups")
        if groups < 0:
            raise ValueError("groups must be nonnegative")
        self._n = groups
        self._row_bytes = (groups + 7) // 8
        self._connections = np.zeros((groups, self._row_bytes), dtype=np.uint8)
        self._minpts = {}

    @property
    def shape(self):
        return (self._n, self._n)

    @property
    def size(self):
        return self._n * self._n

    @property
    def nbytes(self):
        """Exact connection-buffer bytes; sparse marker objects are separate."""
        return self._connections.nbytes

    @property
    def connection_bits(self):
        """Read-only packed rows marking every connection, including minPts edges."""
        view = self._connections.view()
        view.flags.writeable = False
        return view

    @property
    def marker_count(self):
        """Number of directed edges marked as minPts edges."""
        return sum(len(columns) for columns in self._minpts.values())

    @property
    def marker_storage_bytes(self):
        """Estimated sparse-marker Python object bytes, for reporting only.

        Counts the dictionary, sets, row/column integers once by object identity.
        It excludes allocator overhead and may include shared/interned integers;
        therefore this is an estimate, not measured process memory. Cost is O(M)
        in stored directed markers, so avoid querying inside the clustering loop.
        """
        seen = set()
        def size_once(value):
            identity = id(value)
            if identity in seen:
                return 0
            seen.add(identity)
            return sys.getsizeof(value)
        total = size_once(self._minpts)
        for row, columns in self._minpts.items():
            total += size_once(row) + size_once(columns)
            total += sum(size_once(column) for column in columns)
        return total

    @property
    def logical_nbytes(self):
        """Bytes the equivalent dense int8 matrix would require."""
        return self.size

    def __len__(self):
        return self._n

    def __repr__(self):
        return f"PackedAdjacency(shape={self.shape}, nbytes={self.nbytes}, marker_count={self.marker_count})"

    def __array__(self, dtype=None, copy=None):
        raise TypeError("Whole-graph conversion is disabled; use iter_row_blocks(rows=...) or graph[i]")

    def _index(self, index):
        index = _integer(index, "index")
        if index < 0:
            index += self._n
        if not 0 <= index < self._n:
            raise IndexError("adjacency index out of bounds")
        return index

    def _selection(self, selection):
        if isinstance(selection, (int, np.integer)) and not isinstance(selection, (bool, np.bool_)):
            return self._index(selection)
        values = np.asarray(selection)
        if values.ndim != 1:
            raise TypeError("indices must be a scalar integer or a one-dimensional integer array")
        if values.size == 0:
            return np.empty(0, dtype=np.intp)
        if values.dtype.kind not in "iu":
            raise TypeError("indices must have an integer dtype")
        # Check unsigned values before casting to np.intp to prevent wraparound.
        if np.any(values >= self._n) or (values.dtype.kind == "i" and np.any(values < -self._n)):
            raise IndexError("adjacency index out of bounds")
        values = values.astype(np.intp, copy=False)
        if np.any(values < 0):
            values = np.where(values < 0, values + self._n, values)
        return values

    def _pair(self, key):
        if not isinstance(key, tuple) or len(key) != 2:
            raise IndexError("edge access requires exactly two indices")
        row, column = (self._selection(index) for index in key)
        if isinstance(row, np.ndarray) and isinstance(column, np.ndarray):
            raise TypeError("at most one index may be a vector")
        return row, column

    def __getitem__(self, key):
        if not isinstance(key, tuple):
            row = self._index(key)
            decoded = np.unpackbits(self._connections[row], bitorder="little", count=self._n).view(np.int8)
            markers = self._minpts.get(row)
            if markers:
                decoded[np.fromiter(markers, dtype=np.intp, count=len(markers))] = 2
            return decoded
        row, column = self._pair(key)
        byte_index = column >> 3
        offset = column & 7
        result = (self._connections[row, byte_index] >> offset) & 1
        if np.ndim(result) == 0:
            return np.int8(2 if column in self._minpts.get(row, ()) else result)
        result = np.asarray(result, dtype=np.int8)
        if not self._minpts:
            return result
        if isinstance(row, int):
            markers = self._minpts.get(row)
            if markers:
                mask = np.fromiter((int(c) in markers for c in column), dtype=bool, count=len(column))
                result[mask] = 2
        else:
            mask = np.fromiter((column in self._minpts.get(int(r), ()) for r in row), dtype=bool, count=len(row))
            result[mask] = 2
        return result

    def _write_marker(self, row, column, marked):
        if marked:
            markers = self._minpts.get(row)
            if markers is None:
                self._minpts[row] = {column}
            else:
                markers.add(column)
        else:
            markers = self._minpts.get(row)
            if markers is not None:
                markers.discard(column)
                if not markers:
                    del self._minpts[row]

    def __setitem__(self, key, value):
        value = _integer(value, "edge value")
        if not 0 <= value <= 2:
            raise ValueError("edge values must be 0, 1, or 2")
        row, column = self._pair(key)
        if isinstance(row, int) and isinstance(column, int):
            byte_index = column >> 3
            mask = np.uint8(1 << (column & 7))
            clear_mask = np.bitwise_not(mask)
            current = self._connections[row, byte_index]
            self._connections[row, byte_index] = current | mask if value else current & clear_mask
            self._write_marker(row, column, value == 2)
            return
        if isinstance(column, np.ndarray):
            byte_indices = column >> 3
            masks = np.left_shift(np.uint8(1), (column & 7).astype(np.uint8))
            target = (row, byte_indices)
        else:
            masks = np.uint8(1 << (column & 7))
            target = (row, column >> 3)
        if value:
            np.bitwise_or.at(self._connections, target, masks)
        else:
            np.bitwise_and.at(self._connections, target, np.bitwise_not(masks))
        if isinstance(row, int):
            if value == 2 and column.size:
                markers = self._minpts.get(row)
                if markers is None:
                    self._minpts[row] = set(map(int, column))
                else:
                    markers.update(map(int, column))
            elif row in self._minpts:
                markers = self._minpts[row]
                markers.difference_update(map(int, column))
                if not markers:
                    del self._minpts[row]
        elif value == 2 or self._minpts:
            # Normal vector writes during merging incur no Python per-edge loop
            # while the marker dictionary is empty. minPts writes are scalar in
            # CLASSIX; support vector marker writes for a complete tested API.
            for item in map(int, row):
                self._write_marker(item, column, value == 2)

    def iter_row_blocks(self, rows=64):
        """Yield dense int8 row chunks with shape at most (rows, G).

        One decoded int8 chunk is allocated. Callers should process chunks sequentially rather than
        collecting them, to keep memory bounded.
        """
        rows = _integer(rows, "rows")
        if rows <= 0:
            raise ValueError("rows must be positive")
        for start in range(0, self._n, rows):
            stop = min(start + rows, self._n)
            decoded = np.unpackbits(self._connections[start:stop], axis=1,
                                    bitorder="little", count=self._n).view(np.int8)
            for row in range(start, stop):
                markers = self._minpts.get(row)
                if markers:
                    decoded[row - start, np.fromiter(markers, dtype=np.intp, count=len(markers))] = 2
            yield decoded
'''

**`code/classix/packed_native.py`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/classix/packed_native.py'] = r'''"""Strict ctypes interfaces for the isolated packed-bit trial.

The original interface deliberately matches the original Python wrapper's hot
path and assumes valid immutable SciPy CSR arrays and row/vector bounds. Call
validate_original_csr once before timing; ctypes argtypes still enforce dtype,
dimensionality, and contiguity. Extra per-call validation would introduce
artificial overhead absent from the published wrapper.
"""
from ctypes import CDLL, c_int
from pathlib import Path

import numpy as np

_ROOT = Path(__file__).resolve().parent
_packed = CDLL(str(_ROOT / "packed_kernel.dll"))
_original = CDLL(str(_ROOT / "original_spmv.dll"))
_int32_1d = np.ctypeslib.ndpointer(dtype=np.int32, ndim=1, flags="C_CONTIGUOUS")
_uint8_1d = np.ctypeslib.ndpointer(dtype=np.uint8, ndim=1, flags="C_CONTIGUOUS")
_uint8_2d = np.ctypeslib.ndpointer(dtype=np.uint8, ndim=2, flags="C_CONTIGUOUS")
_packed.hardware_popcount_available.argtypes = []
_packed.hardware_popcount_available.restype = c_int
_packed.row_popcounts.argtypes = [_uint8_2d, c_int, c_int, _int32_1d]
_packed.row_popcounts.restype = None
_packed.packed_submatxvec.argtypes = [
    _uint8_2d, c_int, c_int, c_int, _uint8_1d, _int32_1d
]
_packed.packed_submatxvec.restype = None
_original.multiply.argtypes = [
    _int32_1d, _int32_1d, _int32_1d, c_int, c_int, _int32_1d, _int32_1d
]
_original.multiply.restype = None
HARDWARE_POPCOUNT = bool(_packed.hardware_popcount_available())
_INT_MAX = np.iinfo(np.int32).max


def _array(value, dtype, ndim, name):
    if not isinstance(value, np.ndarray):
        raise TypeError(f"{name} must be a numpy array")
    if value.dtype != dtype or value.ndim != ndim or not value.flags.c_contiguous:
        raise TypeError(f"{name} must be a C-contiguous {ndim}D {np.dtype(dtype)} array")
    return value


def _bounds(start, end, nrows):
    if not isinstance(start, (int, np.integer)) or not isinstance(end, (int, np.integer)):
        raise TypeError("row bounds must be integers")
    if not 0 <= start <= end <= nrows <= _INT_MAX:
        raise ValueError("invalid row bounds or too many rows for native int32 API")


def _packed_shape(data):
    _array(data, np.uint8, 2, "data")
    nrows, nbytes = data.shape
    if nrows > _INT_MAX or nbytes > _INT_MAX // 8:
        raise ValueError("packed shape exceeds native int32 API")
    return nrows, nbytes


def row_popcounts(data):
    nrows, nbytes = _packed_shape(data)
    output = np.empty(nrows, dtype=np.int32)
    _packed.row_popcounts(data, nrows, nbytes, output)
    return output


def packed_submatxvec(data, start, end, vector):
    nrows, nbytes = _packed_shape(data)
    _bounds(start, end, nrows)
    _array(vector, np.uint8, 1, "vector")
    if vector.size != nbytes:
        raise ValueError("packed vector length differs from packed matrix row width")
    output = np.empty(end - start, dtype=np.int32)
    _packed.packed_submatxvec(data, nbytes, start, end, vector, output)
    return output


def validate_original_csr(values, indptr, indices, nfeatures):
    """Validate full CSR index contents once, outside timing boundaries."""
    _array(values, np.int32, 1, "values")
    _array(indptr, np.int32, 1, "indptr")
    _array(indices, np.int32, 1, "indices")
    if indptr.size == 0 or values.size != indices.size:
        raise ValueError("invalid CSR array lengths")
    if indptr[0] != 0 or indptr[-1] != values.size or np.any(indptr[1:] < indptr[:-1]):
        raise ValueError("invalid CSR row pointers")
    if indices.size and (indices.min() < 0 or indices.max() >= nfeatures):
        raise ValueError("CSR column indices exceed vector bounds")
    if indptr.size - 1 > _INT_MAX:
        raise ValueError("too many CSR rows")


def original_spsubmatxvec(values, indptr, indices, start, end, vector):
    """Original hot path; callers supply valid SciPy CSR arrays and bounds."""
    output = np.zeros(end - start).astype(np.int32)
    _original.multiply(values, indptr, indices, start, end, vector, output)
    return output
'''

**`code/classix/packed_kernel.c`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/classix/packed_kernel.c'] = r'''/* Isolated binary-fingerprint prototype. Does not modify the original source.
 * Input: row-major np.packbits(..., axis=1, bitorder="little") uint8 arrays.
 * Padding bits must be zero. memcpy avoids alignment/aliasing assumptions.
 */
#include <stdint.h>
#include <stddef.h>
#include <string.h>
#include <intrin.h>

#define EXPORT __declspec(dllexport)

EXPORT int hardware_popcount_available(void) {
    int regs[4];
    __cpuid(regs, 1);
    return (regs[2] & (1 << 23)) != 0;
}

static __forceinline unsigned int portable_popcount64(uint64_t value) {
    value -= (value >> 1) & UINT64_C(0x5555555555555555);
    value = (value & UINT64_C(0x3333333333333333)) +
            ((value >> 2) & UINT64_C(0x3333333333333333));
    value = (value + (value >> 4)) & UINT64_C(0x0f0f0f0f0f0f0f0f);
    return (unsigned int)((value * UINT64_C(0x0101010101010101)) >> 56);
}

static __forceinline unsigned int popcount64(uint64_t value, int hardware) {
    return hardware ? (unsigned int)__popcnt64(value) : portable_popcount64(value);
}

EXPORT void row_popcounts(const uint8_t *packed, int nrows, int nbytes,
                          int32_t *output) {
    const int hardware = hardware_popcount_available();
    for (int row = 0; row < nrows; ++row) {
        const uint8_t *current = packed + (size_t)row * (size_t)nbytes;
        int32_t count = 0;
        int column = 0;
        for (; column + 8 <= nbytes; column += 8) {
            uint64_t word;
            memcpy(&word, current + column, sizeof(word));
            count += (int32_t)popcount64(word, hardware);
        }
        for (; column < nbytes; ++column) {
            count += (int32_t)popcount64(current[column], hardware);
        }
        output[row] = count;
    }
}

EXPORT void packed_submatxvec(const uint8_t *packed, int nbytes,
                             int start_row, int end_row,
                             const uint8_t *vector, int32_t *output) {
    const int hardware = hardware_popcount_available();
    for (int row = start_row; row < end_row; ++row) {
        const uint8_t *current = packed + (size_t)row * (size_t)nbytes;
        int32_t intersection = 0;
        int column = 0;
        for (; column + 8 <= nbytes; column += 8) {
            uint64_t lhs, rhs;
            memcpy(&lhs, current + column, sizeof(lhs));
            memcpy(&rhs, vector + column, sizeof(rhs));
            intersection += (int32_t)popcount64(lhs & rhs, hardware);
        }
        for (; column < nbytes; ++column) {
            intersection += (int32_t)popcount64(current[column] & vector[column], hardware);
        }
        output[row - start_row] = intersection;
    }
}
'''

**`code/classix/spsubmatxvec.c`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/classix/spsubmatxvec.c'] = r'''void multiply(
    const int *m_data, const int *m_indptr, const int *m_indices,
    int start_row, int end_row, const int *v_data, int *result) {

    int j = 0;
    int val = 0;
    int num_vals_matrix = 0;
    int idx = 0;
    int m_indptri = 0;

    for (int i = start_row; i < end_row; i++) {
        m_indptri = m_indptr[i];
        num_vals_matrix = m_indptr[i+1] - m_indptri;
        val = 0;
        for (j = m_indptri; j < m_indptri + num_vals_matrix; j++) { 
            idx = m_indices[j];
            val += m_data[j] * v_data[idx];
        }
        result[i-start_row] = val;
    }
}'''

**`code/classix/LICENSE`**

Preserved implementation text; line endings are normalized when materialized.

In [ ]:
SOURCES['code/classix/LICENSE'] = r'''MIT License

Copyright (c) 2026 Kaustubh Roy

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
'''

**`vendor/bitbirch/bitbirch/__init__.py`**

Preserved implementation text; line endings are normalized when materialized. Upstream BitBIRCH commit `311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f`.

In [ ]:
SOURCES['vendor/bitbirch/bitbirch/__init__.py'] = r''''''

**`vendor/bitbirch/bitbirch/bitbirch.py`**

Preserved implementation text; line endings are normalized when materialized. Upstream BitBIRCH commit `311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f`.

In [ ]:
SOURCES['vendor/bitbirch/bitbirch/bitbirch.py'] = r'''# BitBIRCH is an open-source clustering module based on iSIM
#
# Please, cite the BitBIRCH paper: https://www.biorxiv.org/content/10.1101/2024.08.10.607459v1
#
# BitBIRCH is free software; you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, version 3.
#
# BitBIRCH is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU Lesser General Public License for more details.
#
# BitBIRCH authors (PYTHON): Ramon Alain Miranda Quintana <ramirandaq@gmail.com>, <quintana@chem.ufl.edu>
#                            Vicky (Vic) Jung <jungvicky@ufl.edu>
#                            Kenneth Lopez Perez <klopezperez@chem.ufl.edu>
#                            Kate Huddleston <kdavis2@chem.ufl.edu>
#
# BitBIRCH License: LGPL-3.0 https://www.gnu.org/licenses/lgpl-3.0.en.html#license-text
#
### Part of the tree-management code was derived from https://scikit-learn.org/stable/modules/generated/sklearn.cluster.Birch.html
### Authors: Manoj Kumar <manojkumarsivaraj334@gmail.com>
###          Alexandre Gramfort <alexandre.gramfort@telecom-paristech.fr>
###          Joel Nothman <joel.nothman@gmail.com>
### License: BSD 3 clause

import numpy as np
from scipy import sparse
from bitbirch.pruning import lazyPrune

def set_merge(merge_criterion, tolerance=0.05):
    """
    Sets merge_accept function for merge_subcluster, based on user specified merge_criteria. 

    Radius: merge subcluster based on comparison to centroid of the cluster
    Diameter: merge subcluster based on instant Tanimoto similarity of cluster
    Tolerance: applies tolerance threshold to diameter merge criteria, which will merge subcluster with stricter threshold for newly added molecules

    Parameters:
    -----------
    merge_criterion: str(); 
                        radius, diameter or tolerance
    tolerance: float; 
                        sets penalty value for similarity threshold when callng tolerance merge criteria

    Returns:
    --------
    merge_accept(): function 
                        if cluster is accepted to merge, merge the cluster based on the criteria specified
    """
    if merge_criterion == 'radius':
        def merge_accept(threshold, new_ls, new_centroid, new_n, old_ls, nom_ls, old_n, nom_n):
            jt_sim = jt_isim(new_ls + new_centroid, new_n + 1) * (new_n + 1) - jt_isim(new_ls, new_n) * (new_n - 1)
            return jt_sim >= threshold*2
    elif merge_criterion == 'diameter':
        def merge_accept(threshold, new_ls, new_centroid, new_n, old_ls, nom_ls, old_n, nom_n): 
            jt_radius = jt_isim(new_ls, new_n)
            return jt_radius >= threshold
    elif merge_criterion == 'tolerance_tough':
        def merge_accept(threshold, new_ls, new_centroid, new_n, old_ls, nom_ls, old_n, nom_n):
            jt_radius = jt_isim(new_ls, new_n)
            if jt_radius < threshold:
                return False
            else:
                if old_n == 1 and nom_n == 1:
                    return True
                elif nom_n == 1:
                    return (jt_isim(old_ls + nom_ls, old_n + 1) * (old_n + 1) - jt_isim(old_ls, old_n) * (old_n - 1))/2 >= jt_isim(old_ls, old_n) - tolerance and (jt_radius >= threshold)
                else:
                    return (jt_isim(old_ls + nom_ls, old_n + nom_n) * (old_n + nom_n) * (old_n + nom_n - 1) 
                    - jt_isim(old_ls, old_n) * old_n * (old_n - 1)
                    - jt_isim(nom_ls, nom_n) * nom_n * (nom_n - 1))/(2 * old_n * nom_n) >= jt_isim(old_ls, old_n) - tolerance and (jt_radius >= threshold)
    elif merge_criterion == 'tolerance':
        def merge_accept(threshold, new_ls, new_centroid, new_n, old_ls, nom_ls, old_n, nom_n):
            jt_radius = jt_isim(new_ls, new_n)
            if jt_radius < threshold:
                return False
            else:
                if old_n == 1 and nom_n == 1:
                    return True
                elif nom_n == 1:
                    return (jt_isim(old_ls + nom_ls, old_n + 1) * (old_n + 1) - jt_isim(old_ls, old_n) * (old_n - 1))/2 >= jt_isim(old_ls, old_n) - tolerance and (jt_radius >= threshold)
                else:
                    return True
    globals()['merge_accept'] = merge_accept

def jt_isim(c_total, n_objects):
    """iSIM Tanimoto calculation

    https://pubs.rsc.org/en/content/articlelanding/2024/dd/d4dd00041b

    Parameters
    ----------
    c_total : np.ndarray
              Sum of the elements column-wise

    n_objects : int
                Number of elements

    Returns
    ----------
    isim : float
           iSIM Jaccard-Tanimoto value
    """
    sum_kq = np.sum(c_total)
    sum_kqsq = np.dot(c_total, c_total)
    a = (sum_kqsq - sum_kq)/2

    return a/(a + n_objects * sum_kq - sum_kqsq)

def max_separation(X):
    """Finds two objects in X that are very separated
    This is an approximation (not guaranteed to find
    the two absolutely most separated objects), but it is
    a very robust O(N) implementation. Quality of clustering
    does not diminish in the end.

    Algorithm:
    a) Find centroid of X
    b) mol1 is the molecule most distant from the centroid
    c) mol2 is the molecule most distant from mol1

    Returns
    -------
    (mol1, mol2) : (int, int)
                   indices of mol1 and mol2
    1 - sims_mol1 : np.ndarray
                   Distances to mol1
    1 - sims_mol2: np.ndarray
                   Distances to mol2
    These are needed for node1_dist and node2_dist in _split_node
    """
    # Get the centroid of the set
    n_samples = len(X)
    linear_sum = np.sum(X, axis = 0)
    centroid = calc_centroid(linear_sum, n_samples)

    # Get the similarity of each molecule to the centroid
    pop_counts = np.sum(X, axis = 1)
    a_centroid = np.dot(X, centroid)
    sims_med = a_centroid / (pop_counts + np.sum(centroid) - a_centroid)

    # Get the least similar molecule to the centroid
    mol1 = np.argmin(sims_med)

    # Get the similarity of each molecule to mol1
    a_mol1 = np.dot(X, X[mol1])
    sims_mol1 = a_mol1 / (pop_counts + pop_counts[mol1] - a_mol1)

    # Get the least similar molecule to mol1
    mol2 = np.argmin(sims_mol1)

    # Get the similarity of each molecule to mol2
    a_mol2 = np.dot(X, X[mol2])
    sims_mol2 = a_mol2 / (pop_counts + pop_counts[mol2] - a_mol2)

    return (mol1, mol2), sims_mol1, sims_mol2

def calc_centroid(linear_sum, n_samples):
    """Calculates centroid

    Parameters
    ----------

    linear_sum : np.ndarray
                 Sum of the elements column-wise
    n_samples : int
                Number of samples

    Returns
    -------
    centroid : np.ndarray
               Centroid fingerprints of the given set
    """
    return np.where(linear_sum >= n_samples * 0.5 , 1, 0)

def _iterate_sparse_X(X):
    """This little hack returns a densified row when iterating over a sparse
    matrix, instead of constructing a sparse matrix for every row that is
    expensive.
    """
    n_samples, n_features = X.shape
    X_indices = X.indices
    X_data = X.data
    X_indptr = X.indptr

    for i in range(n_samples):
        row = np.zeros(n_features)
        startptr, endptr = X_indptr[i], X_indptr[i + 1]
        nonzero_indices = X_indices[startptr:endptr]
        row[nonzero_indices] = X_data[startptr:endptr]
        yield row

def _split_node(node, threshold, branching_factor, singly):
    """The node has to be split if there is no place for a new subcluster
    in the node.
    1. Two empty nodes and two empty subclusters are initialized.
    2. The pair of distant subclusters are found.
    3. The properties of the empty subclusters and nodes are updated
       according to the nearest distance between the subclusters to the
       pair of distant subclusters.
    4. The two nodes are set as children to the two subclusters.
    """
    new_subcluster1 = _BFSubcluster()
    new_subcluster2 = _BFSubcluster()
    new_node1 = _BFNode(
        threshold=threshold,
        branching_factor=branching_factor,
        is_leaf=node.is_leaf,
        n_features=node.n_features,
        dtype=node.init_centroids_.dtype,
    )
    new_node2 = _BFNode(
        threshold=threshold,
        branching_factor=branching_factor,
        is_leaf=node.is_leaf,
        n_features=node.n_features,
        dtype=node.init_centroids_.dtype,
    )
    new_subcluster1.child_ = new_node1
    new_subcluster2.child_ = new_node2

    if node.is_leaf:
        if node.prev_leaf_ is not None:
            node.prev_leaf_.next_leaf_ = new_node1
        new_node1.prev_leaf_ = node.prev_leaf_
        new_node1.next_leaf_ = new_node2
        new_node2.prev_leaf_ = new_node1
        new_node2.next_leaf_ = node.next_leaf_
        if node.next_leaf_ is not None:
            node.next_leaf_.prev_leaf_ = new_node2  

    # O(N) implementation of max separation
    farthest_idx, node1_dist, node2_dist = max_separation(node.centroids_)    
    # Notice that max_separation is returning similarities and not distances
    node1_closer = node1_dist > node2_dist
    # Make sure node1 is closest to itself even if all distances are equal.
    # This can only happen when all node.centroids_ are duplicates leading to all
    # distances between centroids being zero.
    node1_closer[farthest_idx[0]] = True

    for idx, subcluster in enumerate(node.subclusters_):
        if node1_closer[idx]:
            new_node1.append_subcluster(subcluster)
            new_subcluster1.update(subcluster)
            if not singly:
                subcluster.parent_ = new_subcluster1
        else:
            new_node2.append_subcluster(subcluster)
            new_subcluster2.update(subcluster)
            if not singly:
                subcluster.parent_ = new_subcluster2
    return new_subcluster1, new_subcluster2


class _BFNode:
    """Each node in a BFTree is called a BFNode.

    The BFNode can have a maximum of branching_factor
    number of BFSubclusters.

    Parameters
    ----------
    threshold : float
        Threshold needed for a new subcluster to enter a BFSubcluster.

    branching_factor : int
        Maximum number of BF subclusters in each node.

    is_leaf : bool
        We need to know if the BFNode is a leaf or not, in order to
        retrieve the final subclusters.

    n_features : int
        The number of features.

    Attributes
    ----------
    subclusters_ : list
        List of subclusters for a particular BFNode.

    prev_leaf_ : _BFNode
        Useful only if is_leaf is True.

    next_leaf_ : _BFNode
        next_leaf. Useful only if is_leaf is True.
        the final subclusters.

    init_centroids_ : ndarray of shape (branching_factor + 1, n_features)
        Manipulate ``init_centroids_`` throughout rather than centroids_ since
        the centroids are just a view of the ``init_centroids_`` .

    centroids_ : ndarray of shape (branching_factor + 1, n_features)
        View of ``init_centroids_``.

    """

    def __init__(self, *, threshold, branching_factor, is_leaf, n_features, dtype):
        self.threshold = threshold
        self.branching_factor = branching_factor
        self.is_leaf = is_leaf
        self.n_features = n_features

        # The list of subclusters, centroids and squared norms
        # to manipulate throughout.
        self.subclusters_ = []
        self.init_centroids_ = np.zeros((branching_factor + 1, n_features), dtype=dtype)
        self.prev_leaf_ = None
        self.next_leaf_ = None

    def append_subcluster(self, subcluster):
        n_samples = len(self.subclusters_)
        self.subclusters_.append(subcluster)
        self.init_centroids_[n_samples] = subcluster.centroid_

        # Keep centroids as views. In this way
        # if we change init_centroids, it is sufficient
        self.centroids_ = self.init_centroids_[: n_samples + 1, :]

    def update_split_subclusters(self, subcluster, new_subcluster1, new_subcluster2, singly):
        """Remove a subcluster from a node and update it with the
        split subclusters.
        """
        if not singly:
            new_subcluster1.parent_ = self.subclusters_[0].parent_
            new_subcluster2.parent_ = self.subclusters_[0].parent_

        ind = self.subclusters_.index(subcluster)
        self.subclusters_[ind] = new_subcluster1
        self.init_centroids_[ind] = new_subcluster1.centroid_
        self.centroids_[ind] = new_subcluster1.centroid_
        self.append_subcluster(new_subcluster2)

    def insert_bf_subcluster(self, subcluster, set_bits, ps, singly):
        """Insert a new subcluster into the node."""
        if not self.subclusters_:
            self.append_subcluster(subcluster)
            return False

        threshold = self.threshold
        branching_factor = self.branching_factor
        # We need to find the closest subcluster among all the
        # subclusters so that we can insert our new subcluster.
        a = np.dot(self.centroids_, subcluster.centroid_)
        sim_matrix = a / (np.sum(self.centroids_, axis = 1) + set_bits - a)
        closest_index = np.argmax(sim_matrix)
        closest_subcluster = self.subclusters_[closest_index]

        # If the subcluster has a child, we need a recursive strategy.
        if closest_subcluster.child_ is not None:
            ps = closest_subcluster
            split_child = closest_subcluster.child_.insert_bf_subcluster(subcluster, set_bits, ps, singly)

            if not split_child:
                # If it is determined that the child need not be split, we
                # can just update the closest_subcluster
                closest_subcluster.update(subcluster)
                self.init_centroids_[closest_index] = self.subclusters_[closest_index].centroid_
                self.centroids_[closest_index] = self.subclusters_[closest_index].centroid_
                return False

            # things not too good. we need to redistribute the subclusters in
            # our child node, and add a new subcluster in the parent
            # subcluster to accommodate the new child.
            else:
                new_subcluster1, new_subcluster2 = _split_node(
                    closest_subcluster.child_,
                    threshold,
                    branching_factor,
                    singly
                )
                self.update_split_subclusters(
                    closest_subcluster, new_subcluster1, new_subcluster2, singly
                )

                if len(self.subclusters_) > self.branching_factor:
                    return True
                return False

        # good to go!
        else:
            merged = closest_subcluster.merge_subcluster(subcluster, self.threshold)
            if merged:
                self.centroids_[closest_index] = closest_subcluster.centroid_
                self.init_centroids_[closest_index] = closest_subcluster.centroid_
                if not singly:
                    closest_subcluster.parent_ = ps
                return False

            # not close to any other subclusters, and we still
            # have space, so add.
            elif len(self.subclusters_) < self.branching_factor:
                self.append_subcluster(subcluster)
                if not singly:
                    closest_subcluster.parent_ = ps
                return False

            # We do not have enough space nor is it closer to an
            # other subcluster. We need to split.
            else:
                self.append_subcluster(subcluster)
                return True


class _BFSubcluster:
    """Each subcluster in a BFNode is called a BFSubcluster.

    A BFSubcluster can have a BFNode has its child.

    Parameters
    ----------
    linear_sum : ndarray of shape (n_features,), default=None
        Sample. This is kept optional to allow initialization of empty
        subclusters.

    Attributes
    ----------
    n_samples_ : int
        Number of samples that belong to each subcluster.

    linear_sum_ : ndarray
        Linear sum of all the samples in a subcluster. Prevents holding
        all sample data in memory.

    centroid_ : ndarray of shape (branching_factor + 1, n_features)
        Centroid of the subcluster. Prevent recomputing of centroids when
        ``BFNode.centroids_`` is called.

    mol_indices : list, default=[]
        List of indices of molecules included in the given cluster.

    child_ : _BFNode
        Child Node of the subcluster. Once a given _BFNode is set as the child
        of the _BFNode, it is set to ``self.child_``.
    """

    def __init__(self, *, linear_sum = None, mol_indices = []):
        if linear_sum is None:
            self.n_samples_ = 0
            self.centroid_ = self.linear_sum_ = 0
            self.mol_indices = []
        else:
            self.n_samples_ = 1
            self.centroid_ = self.linear_sum_ = linear_sum
            self.mol_indices = mol_indices

        self.child_ = None
        self.parent_ = None

    def update(self, subcluster):
        self.n_samples_ += subcluster.n_samples_
        self.linear_sum_ += subcluster.linear_sum_
        self.mol_indices += subcluster.mol_indices
        self.centroid_ = calc_centroid(self.linear_sum_, self.n_samples_)

    def merge_subcluster(self, nominee_cluster, threshold):
        """Check if a cluster is worthy enough to be merged. If
        yes then merge.
        """
        new_ls = self.linear_sum_ + nominee_cluster.linear_sum_
        new_n = self.n_samples_ + nominee_cluster.n_samples_
        new_centroid = calc_centroid(new_ls, new_n)

        if merge_accept(threshold, new_ls, new_centroid, new_n, self.linear_sum_, nominee_cluster.linear_sum_, self.n_samples_, nominee_cluster.n_samples_):
            (
                self.n_samples_,
                self.linear_sum_,
                self.centroid_,
                self.mol_indices
            ) = (new_n, new_ls, new_centroid, self.mol_indices + nominee_cluster.mol_indices)
            return True
        return False


class BitBirch():
    """Implements the BitBIRCH clustering algorithm.

    BitBIRCH paper: 

    Memory- and time-efficient, online-learning algorithm.
    It constructs a tree data structure with the cluster centroids being read off the leaf.

    Parameters
    ----------
    threshold : float, default=0.5
        The similarity radius of the subcluster obtained by merging a new sample and the
        closest subcluster should be greater than the threshold. Otherwise a new
        subcluster is started. Setting this value to be very low promotes
        splitting and vice-versa.

    branching_factor : int, default=50
        Maximum number of BF subclusters in each node. If a new samples enters
        such that the number of subclusters exceed the branching_factor then
        that node is split into two nodes with the subclusters redistributed
        in each. The parent subcluster of that node is removed and two new
        subclusters are added as parents of the 2 split nodes.

    Attributes
    ----------
    root_ : _BFNode
        Root of the BFTree.

    dummy_leaf_ : _BFNode
        Start pointer to all the leaves.

    subcluster_centers_ : ndarray
        Centroids of all subclusters read directly from the leaves.

    Notes
    -----
    The tree data structure consists of nodes with each node consisting of
    a number of subclusters. The maximum number of subclusters in a node
    is determined by the branching factor. Each subcluster maintains a
    linear sum, mol_indices and the number of samples in that subcluster.
    In addition, each subcluster can also have a node as its child, if the
    subcluster is not a member of a leaf node.

    For a new point entering the root, it is merged with the subcluster closest
    to it and the linear sum, mol_indices and the number of samples of that
    subcluster are updated. This is done recursively till the properties of
    the leaf node are updated.
    """


    def __init__(
        self,
        *,
        threshold=0.5,
        branching_factor=50,
    ):
        self.threshold = threshold
        self.branching_factor = branching_factor
        self.index_tracker = 0
        self.first_call = True

    def fit(self, X, singly=True):
        """
        Build a BF Tree for the input data.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Input data.

        Returns
        -------
        self
            Fitted estimator.
        """

        # TODO: Add input verification

        return self._fit(X, singly)

    def _fit(self, X, singly=True):
        threshold = self.threshold
        branching_factor = self.branching_factor

        n_features = X.shape[1]
        d_type = X.dtype

        # If partial_fit is called for the first time or fit is called, we
        # start a new tree.
        if self.first_call:
            # The first root is the leaf. Manipulate this object throughout.
            self.root_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )

            # To enable getting back subclusters.
            self.dummy_leaf_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
            self.dummy_leaf_.next_leaf_ = self.root_
            self.root_.prev_leaf_ = self.dummy_leaf_

        # Cannot vectorize. Enough to convince to use cython.
        if not sparse.issparse(X):
            iter_func = iter
        else:
            iter_func = _iterate_sparse_X

        for sample in iter_func(X):
            set_bits = np.sum(sample)
            subcluster = _BFSubcluster(linear_sum=sample, mol_indices = [self.index_tracker])
            split = self.root_.insert_bf_subcluster(subcluster, set_bits,subcluster.parent_, singly)

            if split:
                new_subcluster1, new_subcluster2 = _split_node(
                    self.root_, threshold, branching_factor, singly
                )
                del self.root_
                self.root_ = _BFNode(
                    threshold=threshold,
                    branching_factor=branching_factor,
                    is_leaf=False,
                    n_features=n_features,
                    dtype=d_type,
                )
                self.root_.append_subcluster(new_subcluster1)
                self.root_.append_subcluster(new_subcluster2)

                if not singly:
                    for i in new_subcluster1.child_.subclusters_:
                        i.parent_ = new_subcluster1
                    for i in new_subcluster2.child_.subclusters_:
                        i.parent_ = new_subcluster2

            self.index_tracker += 1

        centroids = np.concatenate([leaf.centroids_ for leaf in self._get_leaves()])
        self.subcluster_centers_ = centroids
        self._n_features_out = self.subcluster_centers_.shape[0]

        self.first_call = False
        return self

    def fit_reinsert(self, X, reinsert_indices, singly=False):
        """X corresponds to only the molecules that will be reinserted into the tree
        reinsert indices are the indices of the molecules that will be reinserted into the tree"""
        threshold = self.threshold
        branching_factor = self.branching_factor

        n_features = X.shape[1]
        d_type = X.dtype

        # If partial_fit is called for the first time or fit is called, we
        # start a new tree.
        if self.first_call:
            # The first root is the leaf. Manipulate this object throughout.
            self.root_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )

            # To enable getting back subclusters.
            self.dummy_leaf_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
            self.dummy_leaf_.next_leaf_ = self.root_
            self.root_.prev_leaf_ = self.dummy_leaf_

        # Cannot vectorize. Enough to convince to use cython.
        if not sparse.issparse(X):
            iter_func = iter
        else:
            iter_func = _iterate_sparse_X

        for sample, mol_ind in zip(iter_func(X), reinsert_indices):
            set_bits = np.sum(sample)
            subcluster = _BFSubcluster(linear_sum=sample, mol_indices = [mol_ind])
            split = self.root_.insert_bf_subcluster(subcluster, set_bits, subcluster.parent_, False)
            if split:
                new_subcluster1, new_subcluster2 = _split_node(
                    self.root_, threshold, branching_factor, False
                )
                del self.root_
                self.root_ = _BFNode(
                    threshold=threshold,
                    branching_factor=branching_factor,
                    is_leaf=False,
                    n_features=n_features,
                    dtype=d_type,
                )
                self.root_.append_subcluster(new_subcluster1)
                self.root_.append_subcluster(new_subcluster2)

                if not singly:
                    for i in new_subcluster1.child_.subclusters_:
                        i.parent_ = new_subcluster1
                    for i in new_subcluster2.child_.subclusters_:
                        i.parent_ = new_subcluster2

        centroids = np.concatenate([leaf.centroids_ for leaf in self._get_leaves()])
        self.subcluster_centers_ = centroids
        self._n_features_out = self.subcluster_centers_.shape[0]

        self.first_call = False
        return self

    def fit_BFs(self, X):
        """
        Method to fit a BitBirch model with the given BitFeatyres.

        Parameters:
        -----------
        X : list of BitFeatures

        Returns:
        --------
        self
        """

        # Check that the input is a list of BitFeatures
        if type(X) != list or len(X[0]) != 3:
            raise ValueError('The input must be a list of BitFeatures')

        threshold = self.threshold
        branching_factor = self.branching_factor

        n_features = len(X[0][1])
        d_type = X[0][1].dtype

        # If partial_fit is called for the first time or fit is called, we
        # start a new tree.
        if self.first_call:
            # The first root is the leaf. Manipulate this object throughout.
            self.root_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )

            # To enable getting back subclusters.
            self.dummy_leaf_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
            self.dummy_leaf_.next_leaf_ = self.root_
            self.root_.prev_leaf_ = self.dummy_leaf_

        for sample in iter(X):

            cluster = _BFSubcluster()
            cluster.n_samples_, cluster.linear_sum_, cluster.mol_indices = sample[0], sample[1], sample[2]
            cluster.centroid_ = calc_centroid(cluster.linear_sum_, cluster.n_samples_)

            set_bits = np.sum(cluster.centroid_)
            split = self.root_.insert_bf_subcluster(cluster, set_bits, cluster.parent_, True)

            if split:
                new_subcluster1, new_subcluster2 = _split_node(
                    self.root_, threshold, branching_factor, True
                )
                del self.root_
                self.root_ = _BFNode(
                    threshold=threshold,
                    branching_factor=branching_factor,
                    is_leaf=False,
                    n_features=n_features,
                    dtype=d_type,
                )
                self.root_.append_subcluster(new_subcluster1)
                self.root_.append_subcluster(new_subcluster2)
            self.index_tracker += 1

        centroids = np.concatenate([leaf.centroids_ for leaf in self._get_leaves()])
        self.subcluster_centers_ = centroids
        self._n_features_out = self.subcluster_centers_.shape[0]

        self.first_call = False
        return self


    def _get_leaves(self):
        """
        Retrieve the leaves of the BF Node.

        Returns
        -------
        leaves : list of shape (n_leaves,)
            List of the leaf nodes.
        """
        leaf_ptr = self.dummy_leaf_.next_leaf_
        leaves = []
        while leaf_ptr is not None:
            leaves.append(leaf_ptr)
            leaf_ptr = leaf_ptr.next_leaf_
        return leaves

    def get_centroids_mol_ids(self):
        """Method to return a dictionary containing the centroids and mol indices of the leaves"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')

        centroids = []
        mol_ids = []
        for leaf in self._get_leaves():
            for subcluster in leaf.subclusters_:
                centroids.append(subcluster.centroid_)
                mol_ids.append(subcluster.mol_indices)

        dict_centroids_mol_ids = {'centroids': centroids, 'mol_ids': mol_ids}

        return dict_centroids_mol_ids

    def get_centroids(self):
        """Method to return a list of Numpy arrays containing the centroids' fingerprints"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')

        centroids = []
        for leaf in self._get_leaves():
            for subcluster in leaf.subclusters_:
                centroids.append(subcluster.centroid_)

        return centroids

    def get_cluster_mol_ids(self):
        """Method to return the indices of molecules in each cluster"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')

        clusters_mol_id = []
        for leaf in self._get_leaves():
            for subcluster in leaf.subclusters_:
                clusters_mol_id.append(subcluster.mol_indices)

        # Sort the clusters by the number of samples in the cluster
        clusters_mol_id = sorted(clusters_mol_id, key = lambda x: len(x), reverse = True)

        return clusters_mol_id

    def _get_BFs(self):
        """Method to return the BitFeatures of the leaves"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')

        BFs = []
        for leaf in self._get_leaves():
            for subcluster in leaf.subclusters_:
                BFs.append(subcluster)

        # Sort the BitFeatures by the number of samples in the cluster
        BFs = sorted(BFs, key = lambda x: x.n_samples_, reverse = True)

        return BFs

    def prepare_data_BFs(self, fps, initial_mol = 0):
        """Method to prepare the BitFeatures of the largest cluster and the rest of the clusters"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')

        BFs = self._get_BFs()
        big, rest = BFs[0], BFs[1:]

        data = []
        for BF in rest:
            data.append([BF.n_samples_, BF.linear_sum_.astype(np.int64), BF.mol_indices])

        bigs = []
        for mol in big.mol_indices:
            bigs.append([1, fps[mol - initial_mol].astype(np.int64), [mol]])

        return data, bigs

    def get_assignments(self, n_mols):
        clustered_ids = self.get_cluster_mol_ids()

        assignments = np.full(n_mols, -1, dtype=int)
        for i, cluster in enumerate(clustered_ids):
            assignments[cluster] = i + 1

        # Check that there are no unassigned molecules
        assert np.all(assignments != -1)

        return assignments

#Refinement functionality 

    def _get_prune_indices(self):
        """Method to return the indices of molecules in the largest cluster, specifically to be used in fit_reinsert."""
        largest_cluster = max(
            (
                (leaf_idx, cluster_idx, len(cluster.mol_indices))
                for leaf_idx, leaf in enumerate(self._get_leaves())
                for cluster_idx, cluster in enumerate(leaf.subclusters_)
            ),
            key=lambda x: x[2],  # Sort by the cluster size
            default=(None, None, 0),  # Default if no clusters are found
        )
        prune_indices = lazyPrune(largest_cluster[0], largest_cluster[1], self) 
        return prune_indices

    def prune(self, X):
        """
        Retrieves the molecules in the largest cluster and "prunes" the tree by redistributing these molecules. 

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Input data.

        Returns
        -------
        self
            Fitted estimator.
        """
        mol_indices = self._get_prune_indices()
        cluster_fps = X[mol_indices]
        self.fit_reinsert(cluster_fps, mol_indices)
        return self 

    def _calc_centroid(self, X, cluster):
        """Calculates centroid

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
                Input data.
        cluster: np.array of the molecule indices within the cluster

        Returns
        -------
        centroid : np.ndarray
               Centroid fingerprints of the given set
    """

        full_fp = X[cluster]
        linear_sum = np.sum(full_fp, axis=0)
        n_samples = len(full_fp)
        return calc_centroid(linear_sum, n_samples)

    def _get_top_cluster_params(self, X, top):
        """Method to recieve the cluster mol indices, centroids, and fingerprints of the top user-specified clusters.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Input data.

        top: int
            default: 20; specifies number of top largest clusters to reassign

        Returns
        -------
        top_clusters: sorted list
            list of len top; containing the cluster mol ids of given clusters

        centroids: np.array; shape (top, n_features)
            centroids of the top clusters

        mol_indices: list of indices with the top clusters; len of number of molecules in all top clusters

        data_top_clusters: np.array; shape (n_molecules, n_features)
            fingerprints of the molecules in the top clusters
        """
        top_clusters = sorted(self.get_cluster_mol_ids(), key=len, reverse=True)[:top]
        centroids = np.array([self._calc_centroid(X, c) for c in top_clusters])
        mol_indices = [i for c in top_clusters for i in c]
        data_top_clusters = np.array([X[i] for i in mol_indices])
        assert data_top_clusters.shape[0] == len(mol_indices)

        return top_clusters, centroids, mol_indices, data_top_clusters

    def _get_sim_matrix(self, data_top_clusters, centroids):
        """Method to receive the similarity matrix of the user-sepcified top clusters for reassign.

        Parameters
        ----------
        data_top_clusters: np.array; shape (n_molecules, n_features)
            fingerprints of the molecules in the top clusters

        centroids: np.array; shape (top, n_features)
            centroids of the top clusters

        Returns
        -------
        row_max: np.array; shape(n_molecules)
            Each entry is the index of the cluster centroid to which the molecule was closest
        """
        pop_counts = np.sum(centroids, axis=1)
        pop_mols = np.sum(data_top_clusters, axis=1).reshape(-1,1)
        ei = np.einsum("ij,kj->ik", data_top_clusters, centroids)
        tanis = ei/(pop_counts + pop_mols -ei)
        row_max = np.argmax(tanis, axis=1)
        assert row_max.shape == data_top_clusters.shape[0:1]

        return row_max


    def reassign(self, X, top=20, quick=False):
        """
        Reassign molecules across the user-specified (default 20) top largest clusters in the tree based on the tanimoto similarity matrix. 

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Input data.

        top: int
            default: 20; specifies number of top largest clusters to reassign

        quick: boolean
            default: False; if quick specifed, the whole tree will not be return, but a dictionary of top cluster data will be returned

        Return
        ------
        quick return: dict
            dictionary sorted by largest to smallest newly reassigned top clusters

        self
            Fitted estimator.
        """
        top_clusters, centroids, mol_indices, data_top_clusters = self._get_top_cluster_params(X, top)
        row_max = self._get_sim_matrix(data_top_clusters, centroids)


        final_clusters = {i: [] for i in np.unique(row_max)}
        for mol, cluster_idx in zip(mol_indices, row_max):
            final_clusters[cluster_idx].append(mol)

        if quick:
            return dict(sorted(final_clusters.items(), key=lambda item: len(item[1]), reverse=True))    #Return dict sorted by largest to smallest newly reassigned top clusters

        else:
            sub_clusters = [sc for leaf in self._get_leaves() for sc in leaf.subclusters_]
            assert len(sub_clusters) == len(self.get_cluster_mol_ids())

            top_sub_clusters = sorted(sub_clusters, key=lambda c: c.n_samples_, reverse=True)[:top]

            for sc, tc in zip(top_sub_clusters, top_clusters):
                assert sc.n_samples_ == len(tc)

            for cluster, c_ids in zip(top_sub_clusters, final_clusters.values()):
                cluster.n_samples_ = len(c_ids)
                cluster.linear_sum = np.sum(X[c_ids], axis=0)
                cluster.mol_indices = c_ids
                cluster.centroid_ = calc_centroid(cluster.linear_sum_, cluster.n_samples_)
        return self

'''

**`vendor/bitbirch/bitbirch/pruning.py`**

Preserved implementation text; line endings are normalized when materialized. Upstream BitBIRCH commit `311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f`.

In [ ]:
SOURCES['vendor/bitbirch/bitbirch/pruning.py'] = r'''# BitBIRCH is an open-source clustering module based on iSIM
#
# Please, cite the BitBIRCH paper: https://www.biorxiv.org/content/10.1101/2024.08.10.607459v1
#
# BitBIRCH is free software; you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, version 3.
#
# BitBIRCH is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU Lesser General Public License for more details.
#
# BitBIRCH authors (PYTHON): Ramon Alain Miranda Quintana <ramirandaq@gmail.com>, <quintana@chem.ufl.edu>
#                            Vicky (Vic) Jung <jungvicky@ufl.edu>
#                            Kenneth Lopez Perez <klopezperez@chem.ufl.edu>
#                            Kate Huddleston <kdavis2@chem.ufl.edu>
#
# BitBIRCH License: LGPL-3.0 https://www.gnu.org/licenses/lgpl-3.0.en.html#license-text

import numpy as np
import copy

# when deleting from init_centroids, we need to make sure that the elements are shifted
# to the left to fill the space and make sure the empty space on the right is 0 because
# init_centroids is an array with a pre-defined size
def shift_delete(array, index):
    for i in range(index, len(array)-1):
        array[i]=array[i+1]
    array[-1]=0

def recursivelyTraverse(node, children):
    if node is None:
        return

    for i in node.subclusters_:
        children.append(i)
        recursivelyTraverse(i.child_, children)

def pSubPath(subcluster, pSubs, brc):
    # figure out the path of subclusters from leaf level to root

    if subcluster is None:
        return 

    if subcluster.parent_ is None:
        pSubs.append(brc.root_.subclusters_.index(subcluster))
    else:
        pSubs.append(subcluster.parent_.child_.subclusters_.index(subcluster))

    pSubPath(subcluster.parent_, pSubs, brc)

# in order to preserve the insertion order of pNodes, make sure python is updated to 3.7+
def pNodePath(pSubs, pNodes, root):
    temp=root

    for i in pSubs[:0:-1]:
        index=temp.subclusters_.index(i)
        pNodes[temp]=index
        temp=temp.subclusters_[index].child_


def calc_centroid(linear_sum, n_samples):
    """Calculates centroid

    Parameters
    ----------

    linear_sum : np.ndarray
                 Sum of the elements column-wise
    n_samples : int
                Number of samples

    Returns
    -------
    centroid : np.ndarray
               Centroid fingerprints of the given set
    """
    return np.where(linear_sum >= n_samples * 0.5 , 1, 0)

def lazyPrune(leaf_index, subc_index, brc):
    #from bitbirch_tools import bitbirch_double_link_tolerance as bitbirch
    # path from leaf level to root, indices are marked 
    pSubs=[]
    pSubPath(brc._get_leaves()[leaf_index].subclusters_[subc_index], pSubs, brc)

    # traverse all the way down to the leaf node
    node=brc.root_
    for index in pSubs[:0:-1]:
        node=node.subclusters_[index].child_


    # information needed to update all preceding subclusters
    cluster_ls = node.subclusters_[subc_index].linear_sum_
    cluster_n = node.subclusters_[subc_index].n_samples_
    cluster_mol_indices = node.subclusters_[subc_index].mol_indices

    ''' + "'''" + r'''# IMPORTANT: tree must have at least three levels, at least that is what is assumed when inputting dataset
    parent_sub = node.subclusters_[0].parent_       # needed to traverse node upwards
    parent_node = parent_sub.parent_.child_         # needed to reassign leaf nodes''' + "'''" + r'''

    parent_sub = node.subclusters_[0].parent_
    if parent_sub.parent_ is None:
        parent_node = brc.root_
    else:
        parent_node = parent_sub.parent_.child_

    # eliminate the biggest subcluster from the leaf node
    node.subclusters_.pop(subc_index)
    node.centroids_ = np.delete(node.centroids_, subc_index, 0)
    shift_delete(node.init_centroids_, subc_index)

    if len(node.subclusters_) == 0:
        parent_sub.child_ = None

    # rearrange leaf pointers if need to

    # if the leaf node disappears, we assign parent node to replace leaf node
    num_childs=0
    for i in parent_node.subclusters_:
        if i.child_ is not None:
            num_childs+=1
    if num_childs == 0:
        parent_node.is_leaf = True
        if node.prev_leaf_ is not None:
            node.prev_leaf_.next_leaf_ = parent_node
        parent_node.prev_leaf_ = node.prev_leaf_
        parent_node.next_leaf_ = node.next_leaf_
        if node.next_leaf_ is not None:
            node.next_leaf_.prev_leaf_ = parent_node

    # leaf node disappears and there are no reassignments
    elif len(node.subclusters_) == 0:
        if node.prev_leaf_ is not None:
            node.prev_leaf_.next_leaf_ = node.next_leaf_
        if node.next_leaf_ is not None:
            node.next_leaf_.prev_leaf_ = node.prev_leaf_


    if parent_sub.parent_ is None:
        node = brc.root_
    else:
        node = parent_sub.parent_.child_ 


    # here, we update all the preceding subclusters all the way up to the root
    for index in pSubs[1:]:
        sub=node.subclusters_[index]

        sub.linear_sum_ -= cluster_ls
        sub.n_samples_ -= cluster_n
        sub.centroid_ = calc_centroid(sub.linear_sum_, sub.n_samples_)
        sub.mol_indices = [x for x in sub.mol_indices if x not in cluster_mol_indices]

        node.centroids_[index] = sub.centroid_
        node.init_centroids_[index] = sub.centroid_

        if sub.parent_ is None:
            break
        elif sub.parent_.parent_ is None:
            node=brc.root_
        else:
            node=sub.parent_.parent_.child_

    return cluster_mol_indices

'''

**`vendor/bitbirch/LICENSE`**

Preserved implementation text; line endings are normalized when materialized. Upstream BitBIRCH commit `311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f`.

In [ ]:
SOURCES['vendor/bitbirch/LICENSE'] = r'''GNU LESSER GENERAL PUBLIC LICENSE
                       Version 3, 29 June 2007

 Copyright (C) 2007 Free Software Foundation, Inc. <https://fsf.org/>
 Everyone is permitted to copy and distribute verbatim copies
 of this license document, but changing it is not allowed.


  This version of the GNU Lesser General Public License incorporates
the terms and conditions of version 3 of the GNU General Public
License, supplemented by the additional permissions listed below.

  0. Additional Definitions.

  As used herein, "this License" refers to version 3 of the GNU Lesser
General Public License, and the "GNU GPL" refers to version 3 of the GNU
General Public License.

  "The Library" refers to a covered work governed by this License,
other than an Application or a Combined Work as defined below.

  An "Application" is any work that makes use of an interface provided
by the Library, but which is not otherwise based on the Library.
Defining a subclass of a class defined by the Library is deemed a mode
of using an interface provided by the Library.

  A "Combined Work" is a work produced by combining or linking an
Application with the Library.  The particular version of the Library
with which the Combined Work was made is also called the "Linked
Version".

  The "Minimal Corresponding Source" for a Combined Work means the
Corresponding Source for the Combined Work, excluding any source code
for portions of the Combined Work that, considered in isolation, are
based on the Application, and not on the Linked Version.

  The "Corresponding Application Code" for a Combined Work means the
object code and/or source code for the Application, including any data
and utility programs needed for reproducing the Combined Work from the
Application, but excluding the System Libraries of the Combined Work.

  1. Exception to Section 3 of the GNU GPL.

  You may convey a covered work under sections 3 and 4 of this License
without being bound by section 3 of the GNU GPL.

  2. Conveying Modified Versions.

  If you modify a copy of the Library, and, in your modifications, a
facility refers to a function or data to be supplied by an Application
that uses the facility (other than as an argument passed when the
facility is invoked), then you may convey a copy of the modified
version:

   a) under this License, provided that you make a good faith effort to
   ensure that, in the event an Application does not supply the
   function or data, the facility still operates, and performs
   whatever part of its purpose remains meaningful, or

   b) under the GNU GPL, with none of the additional permissions of
   this License applicable to that copy.

  3. Object Code Incorporating Material from Library Header Files.

  The object code form of an Application may incorporate material from
a header file that is part of the Library.  You may convey such object
code under terms of your choice, provided that, if the incorporated
material is not limited to numerical parameters, data structure
layouts and accessors, or small macros, inline functions and templates
(ten or fewer lines in length), you do both of the following:

   a) Give prominent notice with each copy of the object code that the
   Library is used in it and that the Library and its use are
   covered by this License.

   b) Accompany the object code with a copy of the GNU GPL and this license
   document.

  4. Combined Works.

  You may convey a Combined Work under terms of your choice that,
taken together, effectively do not restrict modification of the
portions of the Library contained in the Combined Work and reverse
engineering for debugging such modifications, if you also do each of
the following:

   a) Give prominent notice with each copy of the Combined Work that
   the Library is used in it and that the Library and its use are
   covered by this License.

   b) Accompany the Combined Work with a copy of the GNU GPL and this license
   document.

   c) For a Combined Work that displays copyright notices during
   execution, include the copyright notice for the Library among
   these notices, as well as a reference directing the user to the
   copies of the GNU GPL and this license document.

   d) Do one of the following:

       0) Convey the Minimal Corresponding Source under the terms of this
       License, and the Corresponding Application Code in a form
       suitable for, and under terms that permit, the user to
       recombine or relink the Application with a modified version of
       the Linked Version to produce a modified Combined Work, in the
       manner specified by section 6 of the GNU GPL for conveying
       Corresponding Source.

       1) Use a suitable shared library mechanism for linking with the
       Library.  A suitable mechanism is one that (a) uses at run time
       a copy of the Library already present on the user's computer
       system, and (b) will operate properly with a modified version
       of the Library that is interface-compatible with the Linked
       Version.

   e) Provide Installation Information, but only if you would otherwise
   be required to provide such information under section 6 of the
   GNU GPL, and only to the extent that such information is
   necessary to install and execute a modified version of the
   Combined Work produced by recombining or relinking the
   Application with a modified version of the Linked Version. (If
   you use option 4d0, the Installation Information must accompany
   the Minimal Corresponding Source and Corresponding Application
   Code. If you use option 4d1, you must provide the Installation
   Information in the manner specified by section 6 of the GNU GPL
   for conveying Corresponding Source.)

  5. Combined Libraries.

  You may place library facilities that are a work based on the
Library side by side in a single library together with other library
facilities that are not Applications and are not covered by this
License, and convey such a combined library under terms of your
choice, if you do both of the following:

   a) Accompany the combined library with a copy of the same work based
   on the Library, uncombined with any other library facilities,
   conveyed under the terms of this License.

   b) Give prominent notice with the combined library that part of it
   is a work based on the Library, and explaining where to find the
   accompanying uncombined form of the same work.

  6. Revised Versions of the GNU Lesser General Public License.

  The Free Software Foundation may publish revised and/or new versions
of the GNU Lesser General Public License from time to time. Such new
versions will be similar in spirit to the present version, but may
differ in detail to address new problems or concerns.

  Each version is given a distinguishing version number. If the
Library as you received it specifies that a certain numbered version
of the GNU Lesser General Public License "or any later version"
applies to it, you have the option of following the terms and
conditions either of that published version or of any later version
published by the Free Software Foundation. If the Library as you
received it does not specify a version number of the GNU Lesser
General Public License, you may choose any version of the GNU Lesser
General Public License ever published by the Free Software Foundation.

  If the Library as you received it specifies that a proxy can decide
whether future versions of the GNU Lesser General Public License shall
apply, that proxy's public statement of acceptance of any version is
permanent authorization for you to choose that version for the
Library.
'''

**`code/prepare_dataset.py`**

The generator and shuffle are unchanged. Historical data-manifest dependencies are removed: every array and hash is generated here, and cached arrays are rechecked on resume.

In [ ]:
SOURCES['code/prepare_dataset.py'] = r'''"""Generate shared data afresh; validate cached arrays against this run's hashes."""
import argparse
import gc
import hashlib
import json
from pathlib import Path
import time
import numpy as np
from generate_data_snapshot import generate_data
HERE = Path(__file__).resolve().parent
ROOT = HERE.parent

def digest(array):
    result = hashlib.sha256()
    row_bytes = array.dtype.itemsize * int(np.prod(array.shape[1:], dtype=np.int64))
    rows = max(1, (8 * 1024 * 1024) // max(1, row_bytes))
    for start in range(0, len(array), rows):
        result.update(memoryview(np.ascontiguousarray(array[start:start + rows])).cast("B"))
    return result.hexdigest()

def validate(directory, metadata, case):
    assert metadata["data_id"] == case["data_id"]
    assert metadata["parameters"] == case["generator"]
    assert metadata["shape"] == [case["rows"], case["features"]]
    assert metadata["generator_snapshot_sha256"] == hashlib.sha256((HERE/"generate_data_snapshot.py").read_bytes()).hexdigest()
    n, d, k = case["rows"], case["features"], case["generator"]["num_clusters"]
    shapes = {"int32": (n,d), "int64": (n,d), "packed_little": (n,(d+7)//8),
              "packed_big": (n,(d+7)//8), "truth": (n,), "permutation": (n,), "centers": (k,d)}
    assert set(metadata["files"]) == set(shapes)
    for name, shape in shapes.items():
        rec = metadata["files"][name]
        assert Path(rec["file"]).name == rec["file"]
        array = np.load(directory/rec["file"], allow_pickle=False, mmap_mode="r")
        assert array.shape == shape and str(array.dtype) == rec["dtype"] and array.nbytes == rec["bytes"]
        assert digest(array) == rec["sha256"], f"Cached array changed: {name}"
    return len(shapes)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("data_id")
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--plan", type=Path, default=ROOT/"plan.json")
    args = parser.parse_args()
    plan = json.loads(args.plan.read_text())
    case = next(c for c in plan["cases"] if c["data_id"] == args.data_id)
    directory = args.data_dir.resolve()/args.data_id
    assert directory.is_relative_to((ROOT/"runs").resolve())
    directory.mkdir(parents=True, exist_ok=True)
    manifest_path = directory/"manifest.json"
    if manifest_path.exists():
        metadata = json.loads(manifest_path.read_text())
        print(json.dumps({"reusing": args.data_id, "array_hashes_verified": validate(directory, metadata, case)}), flush=True)
        return
    params = case["generator"]
    start = time.perf_counter()
    data, truth = generate_data(**params)
    centers = data[np.arange(params["num_clusters"])*params["n"]].copy()
    permutation = np.arange(len(data))
    np.random.shuffle(permutation)
    data, truth = np.ascontiguousarray(data[permutation]), truth[permutation]
    assert data.shape == (case["rows"], case["features"])
    assert np.all((data == 0) | (data == 1))
    generation_seconds = time.perf_counter()-start
    files = {}
    def save(name, array):
        np.save(directory/f"{name}.npy", array)
        files[name] = {"file": name+".npy", "bytes": array.nbytes,
                       "dtype": str(array.dtype), "sha256": digest(array)}
    save("int32", data)
    wide = data.astype(np.int64)
    save("int64", wide)
    del wide
    gc.collect()
    for order in ("little", "big"):
        save("packed_"+order, np.packbits(data, axis=1, bitorder=order))
    save("truth", truth)
    save("permutation", permutation)
    save("centers", centers)
    row_pops = data.sum(axis=1)
    metadata = {"data_id": args.data_id, "parameters": params, "shape": list(data.shape),
        "files": files, "zero_fraction": float(1-data.mean()), "zero_rows": int(np.sum(row_pops == 0)),
        "minimum_popcount": int(row_pops.min()), "maximum_popcount": int(row_pops.max()),
        "center_popcounts": centers.sum(axis=1).tolist(), "generation_seconds": generation_seconds,
        "source_notebook_sha256": plan["source_notebook_sha256"],
        "generator_snapshot_sha256": hashlib.sha256((HERE/"generate_data_snapshot.py").read_bytes()).hexdigest(),
        "shuffle": "np.random.shuffle after generator, continuing its seeded NumPy RNG"}
    assert metadata["zero_rows"] == 0, "Tanimoto protocol requires nonempty vectors."
    validate(directory, metadata, case)
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(metadata, indent=2)+"\n")
    temporary.replace(manifest_path)
    print(json.dumps({"generated": args.data_id, "shape": metadata["shape"]}), flush=True)

if __name__ == "__main__":
    main()
'''

**`code/reproduce.py`**

Preserves the validated process/timing/memory/repetition controller. Only the generated design-policy path and reporting dependency inventory change. The timed worker above is unchanged.

In [ ]:
SOURCES['code/reproduce.py'] = r'''"""Fresh Windows reproductions; recorded evidence is never used as new results.

Default: inspect the plan, inputs to generation, dependencies and native imports.
Use --prepare to generate data, or --run to execute the selected cases. Every
write stays beneath experiment_1A/runs/<name>. PAUSE stops at the next fit
boundary; STOP_CURRENT interrupts the current attempt and leaves it retriable.
Remove the control file and repeat the same command to resume.
"""
import argparse
from contextlib import contextmanager
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import queue
import statistics
import subprocess
import sys
import threading
import time
import traceback

sys.dont_write_bytecode = True
HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
RUN_DIR = None
METHODS = ("dbscan", "classix_packed", "bitbirch", "bblean", "butina")
PACKAGES = ("numpy", "scipy", "scikit-learn", "joblib", "cloudpickle", "narwhals",
            "rdkit", "bblean", "threadpoolctl", "tqdm", "matplotlib")
TERMINAL = {"timeout", "memory_limit", "algorithm_error", "worker_error",
            "startup_timeout", "validation_timeout", "validation_error"}


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save(value, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def require(condition, message):
    if not condition:
        raise ValueError(message)


def load_design():
    plan = read(ROOT / "plan.json")
    policy = read(ROOT / "design" / "repetition_policy.json")
    require(tuple(plan["methods"]) == METHODS, "Expected the five recorded methods")
    cases = plan["cases"]
    require(len(cases) == len({case["case_id"] for case in cases}) == 42,
            "Expected the final 42-setting plan")
    require(len({case["data_id"] for case in cases}) == 36, "Expected 36 shared datasets")
    require("sample_count_100000" not in {case["case_id"] for case in cases},
            "The cancelled 100k sample case must remain excluded")
    require(plan["fit_timeout_seconds"] is None and plan["job_memory_limit_bytes"] == 20 * 1024**3,
            "The recorded resource policy is no fit cutoff and a 20-GiB job limit")
    for case in cases:
        generator = case["generator"]
        require(generator["seed"] == 101 and case["rows"] == generator["n"] * generator["num_clusters"],
                "Dataset seed or total size differs")
        if case["study"] == "radius":
            require(case["rows"] == 10000 and .2 <= case["x"] <= .7, "Radius dataset differs")
        for method in METHODS:
            ids = policy["schedules"][case["case_id"]][method]
            require(ids == sorted(set(ids)) and ids and any(value >= 0 for value in ids)
                    and all(value in (-1, 0, 1, 2) for value in ids), "Invalid recorded repetition schedule")
    return plan, policy


def select_cases(plan, requested):
    known = {case["case_id"] for case in plan["cases"]}
    require(not requested or set(requested) <= known, "Unknown case ID; --check lists available cases")
    return [case for case in plan["cases"] if not requested or case["case_id"] in requested]


def output_root(value):
    root = Path(value).resolve()
    allowed = (ROOT / "runs").resolve()
    require(root != allowed and root.is_relative_to(allowed),
            "--output-dir must be a named folder beneath experiment_1A/runs; recorded evidence is read-only")
    return root


def environment_check(plan, policy, cases, methods):
    """Import dependencies and native bindings, without constructing estimators."""
    require(os.name == "nt", "The recorded native kernels and memory protocol require 64-bit Windows")
    require(platform.machine().lower() in ("amd64", "x86_64") and sys.maxsize > 2**32,
            "Use a 64-bit x86 Windows Python with the included native binaries")
    for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[name] = "1"
    versions = {name: importlib.metadata.version(name) for name in PACKAGES}
    sys.path.insert(0, str(HERE / "classix"))
    sys.path.insert(0, str(ROOT / "vendor" / "bitbirch"))
    for name in ("numpy", "scipy", "sklearn.cluster", "rdkit.DataStructs", "rdkit.ML.Cluster.Butina",
                 "bblean", "classix_t_packed_adjbits", "bitbirch.bitbirch", "memory_win", "job_limit"):
        importlib.import_module(name)
    similarity = importlib.import_module("bblean.similarity")
    require(similarity._jt_sim_arr_vec_packed.__module__ == "bblean._cpp_similarity",
            "BBLean must use its native packed-bit kernel")
    native = importlib.import_module("packed_native")
    schedule_counts = {method: {
        "warmups": sum(-1 in policy["schedules"][case["case_id"]][method] for case in cases),
        "measured": sum(sum(value >= 0 for value in policy["schedules"][case["case_id"]][method]) for case in cases),
    } for method in methods}
    return {"status": "passed", "python": sys.version, "executable": sys.executable,
            "platform": platform.platform(), "machine": platform.machine(), "packages": versions,
            "hardware_popcount": bool(native.HARDWARE_POPCOUNT), "selected_cases": len(cases),
            "selected_methods": methods, "case_ids": [case["case_id"] for case in cases],
            "selected_datasets": len({case["data_id"] for case in cases}), "repetitions": schedule_counts,
            "fit_timeout_seconds": plan["fit_timeout_seconds"], "job_memory_limit_bytes": plan["job_memory_limit_bytes"],
            "note": "Imports only; no dataset generation or clustering fits were run."}


def protocol(plan, environment, pilot_rows=None):
    files = [HERE / name for name in ("reproduce.py", "worker.py", "prepare_dataset.py",
             "generate_data_snapshot.py", "quality.py", "state_checks.py", "memory_win.py", "job_limit.py")]
    files += [path for path in (HERE / "classix").rglob("*") if path.is_file() and path.suffix in (".py", ".dll", ".c")]
    files += [path for path in (ROOT / "vendor" / "bitbirch").rglob("*.py")]
    return {"kind": "standalone_fresh_reproduction", "schema": 1,
            "plan_sha256": sha(ROOT / "plan.json"),
            "repetition_amendment_sha256": sha(ROOT / "design" / "repetition_policy.json"),
            "source_sha256": {path.relative_to(ROOT).as_posix(): sha(path) for path in sorted(set(files))},
            "packages": environment["packages"], "python": sys.version,
            "platform": platform.platform(), "fit_timeout_seconds": plan["fit_timeout_seconds"],
            "job_memory_limit_bytes": plan["job_memory_limit_bytes"], "threads": 1, "pilot_rows": pilot_rows,
            "repetitions": "one measured fit per method, no warmup; diagnostic pilot" if pilot_rows else "recorded per-case schedules",
            "timing_scope": plan["timed_operations"]}


def execute(case, variant, repeat, plan, output_directory, pilot_rows=None):
    labels_path = output_directory / f"{variant}_{repeat}.npy"
    command = [sys.executable, str(HERE/"worker.py"), "--case", case["case_id"], "--variant", variant, "--output", str(labels_path), "--data-dir", str(RUN_DIR/"data"), "--plan", str(ROOT/"plan.json")]
    if pilot_rows:
        command += ["--rows", str(pilot_rows)]
    environment = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONHASHSEED="0")
    for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        environment[variable] = "1"
    process = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, env=environment, creationflags=subprocess.CREATE_NO_WINDOW)
    messages = queue.Queue()
    def reader():
        for line in process.stdout:
            messages.put(line.rstrip())
        messages.put(None)
    threading.Thread(target=reader, daemon=True).start()
    result = {"case_id": case["case_id"], "variant": variant, "repeat": repeat, "warmup": repeat < 0,
              "status": "running", "logs": [], "limits": {"fit_seconds": plan["fit_timeout_seconds"], "job_memory_bytes": plan["job_memory_limit_bytes"]}}
    phase, phase_start = "startup", time.perf_counter()
    job, handle = None, None
    sampled = {"samples": 0, "target_interval_seconds": .02, "max_interval_seconds": 0,
               "peak_private_commit_bytes": 0, "peak_working_set_bytes": 0}
    last_sample_time = None
    def sample():
        nonlocal last_sample_time
        now, values = time.perf_counter(), snapshot(handle)
        if last_sample_time is not None:
            sampled["max_interval_seconds"] = max(sampled["max_interval_seconds"], now-last_sample_time)
        last_sample_time = now
        sampled["samples"] += 1
        for metric in ("private_commit", "working_set"):
            key = "peak_"+metric+"_bytes"
            sampled[key] = max(sampled[key], values[metric+"_bytes"])
    heartbeat = time.perf_counter()
    last_control_check = time.perf_counter() - 2
    try:
        while True:
            elapsed = time.perf_counter()-phase_start
            if job is not None and time.perf_counter() - last_control_check >= 1:
                last_control_check = time.perf_counter()
                if (RUN_DIR / "STOP_CURRENT").exists():
                    result["status"] = "interrupted"
                    result["interruption_phase"] = phase
                    result["elapsed_in_phase_seconds"] = elapsed
                    break
            if time.perf_counter() - heartbeat >= 15:
                save({"case": case["case_id"], "variant": variant, "repeat": repeat,
                      "phase": phase, "elapsed_in_phase_seconds": elapsed,
                      "sampled_memory": sampled, "updated_utc": datetime.now(timezone.utc).isoformat()}, RUN_DIR/"active_fit.json")
                heartbeat = time.perf_counter()
            deadline = plan["fit_timeout_seconds"] if phase == "fit" else 300
            if deadline is not None and elapsed > deadline:
                result["status"] = "timeout" if phase == "fit" else phase+"_timeout"
                result["timeout_phase"] = phase
                result["elapsed_in_phase_seconds"] = elapsed
                break
            if phase == "fit" and handle:
                try:
                    sample()
                except OSError:
                    pass  # Read queued final/error output if the worker just exited.
            try:
                line = messages.get(timeout=.02 if phase == "fit" else .1)
            except queue.Empty:
                if process.poll() is not None and not messages.qsize():
                    break
                continue
            if line is None:
                break
            if not line.startswith("BENCH_JSON "):
                result["logs"].append(line)
                continue
            event = json.loads(line[len("BENCH_JSON "):])
            kind = event.pop("event")
            result[kind] = event
            if kind == "ready":
                # Monitor/constrain the actual Python PID, not its venv launcher.
                actual_pid = event["pid"]
                job = assign_job(pid=actual_pid, max_bytes=plan["job_memory_limit_bytes"])
                handle = open_process(actual_pid)
                result["monitored_pid"] = actual_pid
                result["launcher_pid"] = process.pid
                sample()
                phase, phase_start = "fit", time.perf_counter()
                process.stdin.write("GO\n")
                process.stdin.flush()
            elif kind == "fit_done":
                sample()
                phase, phase_start = "validation", time.perf_counter()
                process.stdin.write("VALIDATE\n")
                process.stdin.flush()
            elif kind == "fit_error":
                is_memory = "MemoryError" in event["error_type"] or "Unable to allocate" in event["message"]
                result["status"] = "memory_limit" if is_memory else "algorithm_error"
            elif kind == "validation_error":
                result["status"] = "validation_error"
            elif kind == "validated":
                result["status"] = "ok"
        if result["status"] == "running":
            result["status"] = "worker_error"
    finally:
        if job is not None:
            close_job(job)
        if process.poll() is None:
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
        result["exit_code"] = process.wait(timeout=10)
        if handle:
            close_process(handle)
    if sampled["samples"]:
        result["sampled_memory"] = sampled
    if "fit_done" in result:
        result["native_peaks_attained_in_fit_window"] = {metric:
            result["fit_done"]["after_fit"]["lifetime_peak_"+metric+"_bytes"] > result["ready"]["before_fit"]["lifetime_peak_"+metric+"_bytes"]
            for metric in ("private_commit", "working_set")}
    save({"case": case["case_id"], "variant": variant, "repeat": repeat,
          "finished": True, "status": result["status"],
          "updated_utc": datetime.now(timezone.utc).isoformat()}, RUN_DIR/"active_fit.json")
    return result


def summarize(runs, methods):
    output = []
    for method in methods:
        all_runs = [r for r in runs if r["variant"] == method]
        failures = [r for r in all_runs if r["status"] != "ok"]
        measured = [r for r in all_runs if r["status"] == "ok" and not r["warmup"]]
        row = {"method": method, "status": failures[-1]["status"] if failures else "ok", "measured_repeats": len(measured),
               "warning_runs": sum(bool(r["logs"]) for r in all_runs)}
        if measured:
            values = [r["fit_done"]["clustering_seconds"] for r in measured]
            row.update(median_seconds=statistics.median(values), min_seconds=min(values), max_seconds=max(values), all_seconds=values,
                ari_values=sorted({r["validated"]["ari"] for r in measured}), cluster_counts=sorted({r["validated"]["clusters"] for r in measured}),
                aggregation_groups=sorted({r["validated"]["aggregation_groups"] for r in measured if r["validated"]["aggregation_groups"] is not None}),
                median_peak_private_bytes=statistics.median(r["fit_done"]["after_fit"]["lifetime_peak_private_commit_bytes"] for r in measured),
                median_peak_working_set_bytes=statistics.median(r["fit_done"]["after_fit"]["lifetime_peak_working_set_bytes"] for r in measured),
                private_peak_verified_in_fit_window=all(r["native_peaks_attained_in_fit_window"]["private_commit"] for r in measured),
                working_set_peak_verified_in_fit_window=all(r["native_peaks_attained_in_fit_window"]["working_set"] for r in measured))
            row["noise_counts"] = sorted({r["validated"]["noise_points"] for r in measured})
        output.append(row)
    return output


def saved_path(checked, kind):
    relative = checked.get("output_relative_" + kind)
    return RUN_DIR / relative if relative else Path(checked[kind])


def method_finished(runs, ids):
    if not runs:
        return False
    sequence = [run["repeat"] for run in runs]
    require(sequence == ids[:len(sequence)], "Saved repeats are not a unique ordered prefix of the recorded schedule")
    require(all(run["warmup"] == (run["repeat"] == -1) for run in runs), "Warmup flag differs")
    failed = [index for index, run in enumerate(runs) if run["status"] != "ok"]
    if failed:
        require(failed == [len(runs) - 1] and runs[-1]["status"] in TERMINAL,
                "Invalid terminal failure or additional attempts after failure")
        return True
    return sequence == ids


def verify_run(run, current_protocol, partitions, state_reference):
    require(run["protocol"] == current_protocol, "Existing run uses a different reproduction protocol; select a new output folder")
    if run["status"] != "ok":
        return state_reference
    checked = run["validated"]
    require(saved_path(checked, "assignments_file").is_file(), "Saved assignments are missing")
    partition = checked["canonical_partition_sha256"]
    require(partitions.setdefault(run["variant"], partition) == partition, "Partition changed between repetitions")
    if checked["state_file"]:
        state = read(saved_path(checked, "state_file"))
        if state_reference is None:
            state_reference = state
        require(state == state_reference, "CLASSIX state changed between repetitions")
        checked["classix_state_repeats_consistent"] = True
    return state_reference


def prepare(case):
    command = [sys.executable, str(HERE / "prepare_dataset.py"), case["data_id"],
               "--data-dir", str(RUN_DIR / "data"), "--plan", str(ROOT / "plan.json")]
    subprocess.run(command, check=True, cwd=ROOT, creationflags=subprocess.CREATE_NO_WINDOW,
                   env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))


@contextmanager
def prevent_idle_sleep():
    """Match the historical runner's Windows power request outside fit timing."""
    import ctypes
    kernel = ctypes.WinDLL("kernel32", use_last_error=True)
    kernel.SetThreadExecutionState.argtypes = [ctypes.c_uint32]
    kernel.SetThreadExecutionState.restype = ctypes.c_uint32
    if not kernel.SetThreadExecutionState(0x80000001):
        raise ctypes.WinError(ctypes.get_last_error())
    try:
        yield
    finally:
        kernel.SetThreadExecutionState(0x80000000)


def controlled_pause(case_id=None):
    if any((RUN_DIR / name).exists() for name in ("PAUSE", "STOP_CURRENT")):
        save({"paused": True, "case": case_id, "utc": datetime.now(timezone.utc).isoformat()}, RUN_DIR / "live_progress.json")
        print("Paused. Remove PAUSE/STOP_CURRENT in this output folder and rerun the same command to resume.", flush=True)
        return True
    return False


def run_cases(plan, policy, cases, methods, current_protocol, pilot_rows=None):
    results_dir = RUN_DIR / "results"
    results_dir.mkdir(exist_ok=True)
    for case_index, case in enumerate(cases):
        case_id = case["case_id"]
        schedules = {method: [0] for method in METHODS} if pilot_rows else policy["schedules"][case_id]
        path = results_dir / (case_id + ".json")
        if path.exists():
            result = read(path)
            require(result["case"] == case and result["protocol"] == current_protocol,
                    "Saved case or protocol differs; use a new output folder")
        else:
            result = {"case": case, "runs": [], "pilot_rows": pilot_rows,
                      "measured_repeats_requested": 1 if pilot_rows else plan["measured_repeats"], "protocol": current_protocol,
                      "method_repeat_ids": schedules, "reproduction": True}
        partitions, state_reference = {}, None
        require(all(run["variant"] in METHODS for run in result["runs"]), "Unexpected saved method")
        for run in result["runs"]:
            state_reference = verify_run(run, current_protocol, partitions, state_reference)
        finished = {method: method_finished([run for run in result["runs"] if run["variant"] == method], schedules[method])
                    for method in METHODS}
        if all(finished[method] for method in methods):
            print("REUSE " + case_id, flush=True)
            continue
        if controlled_pause(case_id):
            return False
        prepare(case)
        result["data_manifest"] = str(RUN_DIR / "data" / case["data_id"] / "manifest.json")
        result["output_relative_data_manifest"] = (Path("data") / case["data_id"] / "manifest.json").as_posix()
        save(result, path)
        existing = {(run["variant"], run["repeat"]) for run in result["runs"]}
        for repeat in sorted({value for method in methods for value in schedules[method]}):
            offset = max(repeat, 0) % len(methods)
            for variant in methods[offset:] + methods[:offset]:
                if finished[variant] or repeat not in schedules[variant] or (variant, repeat) in existing:
                    continue
                if controlled_pause(case_id):
                    return False
                stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
                assignments = results_dir / (case_id + "_assignments") / f"{variant}_{repeat}_{stamp}"
                save({"case_index": case_index + 1, "case_total": len(cases), "case": case_id,
                      "variant": variant, "repeat": repeat, "started_utc": datetime.now(timezone.utc).isoformat()},
                     RUN_DIR / "live_progress.json")
                run = execute(case, variant, repeat, plan, assignments, pilot_rows)
                run["protocol"] = current_protocol
                if "validated" in run:
                    checked = run["validated"]
                    for kind in ("assignments_file", "state_file"):
                        if checked.get(kind):
                            checked["output_relative_" + kind] = Path(checked[kind]).relative_to(RUN_DIR).as_posix()
                if run["status"] == "interrupted":
                    attempts = RUN_DIR / "interrupted_attempts"
                    attempts.mkdir(exist_ok=True)
                    save(run, attempts / f"{case_id}_{variant}_{repeat}_{stamp}.json")
                    save({"paused": True, "interrupted_current_fit": True, "case": case_id,
                          "variant": variant, "repeat": repeat}, RUN_DIR / "live_progress.json")
                    print("Interrupted attempt preserved; rerun after removing the control file to retry.", flush=True)
                    return False
                try:
                    state_reference = verify_run(run, current_protocol, partitions, state_reference)
                except Exception as error:
                    run["status"] = "validation_error"
                    run["validation_error"] = {"error_type": type(error).__name__, "message": str(error)}
                result["runs"].append(run)
                existing.add((variant, repeat))
                finished[variant] = method_finished([value for value in result["runs"] if value["variant"] == variant], schedules[variant])
                result["summary"] = summarize(result["runs"], METHODS)
                result["completed"] = all(finished.values())
                save(result, path)
                print(json.dumps({"case": case_id, "variant": variant, "repeat": repeat,
                                  "status": run["status"], "seconds": run.get("fit_done", {}).get("clustering_seconds")}), flush=True)
                if run["status"] == "validation_error":
                    raise RuntimeError("Validation failed; preserved evidence requires review before continuing")
        result["summary"] = summarize(result["runs"], METHODS)
        result["completed"] = all(finished.values())
        save(result, path)
    save({"complete": True, "selected_cases": len(cases), "selected_methods": methods,
          "note": "Completion concerns this selection; per-case completed flags cover all five methods."}, RUN_DIR / "live_progress.json")
    return True


def main():
    global RUN_DIR, open_process, close_process, snapshot, assign_job, close_job
    parser = argparse.ArgumentParser(description=__doc__)
    mode = parser.add_mutually_exclusive_group()
    mode.add_argument("--check", action="store_true", help="Check dependencies and design without generating data or fitting; default")
    mode.add_argument("--prepare", action="store_true", help="Generate shared data for selected cases only")
    mode.add_argument("--run", action="store_true", help="Run/resume selected cases using the recorded repetition schedule")
    parser.add_argument("--output-dir", type=Path, default=ROOT / "runs" / "reproduction")
    parser.add_argument("--cases", nargs="+", help="Case IDs listed by --check")
    parser.add_argument("--methods", nargs="+", choices=METHODS)
    parser.add_argument("--pilot-rows", type=int, help="Diagnostic prefix only: one measured fit per method, no warmup; requires --run and --cases")
    args = parser.parse_args()
    plan, policy = load_design()
    cases = select_cases(plan, args.cases)
    if args.pilot_rows is not None:
        require(args.run and args.cases and 0 < args.pilot_rows <= min(case["rows"] for case in cases),
                "--pilot-rows requires --run, explicit --cases, and a positive prefix within each dataset")
    methods = [method for method in METHODS if not args.methods or method in args.methods]
    RUN_DIR = output_root(args.output_dir)
    environment = environment_check(plan, policy, cases, methods)
    if args.pilot_rows:
        environment["pilot_rows"] = args.pilot_rows
        environment["repetitions"] = {method: {"warmups": 0, "measured": len(cases)} for method in methods}
        environment["pilot_note"] = "Functional smoke check only. Full arrays load before slicing, so pilot memory does not describe a standalone small dataset."
    print(json.dumps(environment, indent=2), flush=True)
    if not (args.prepare or args.run):
        return 0
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    lock = RUN_DIR / "run.lock"
    descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    os.write(descriptor, json.dumps({"pid": os.getpid(), "utc": datetime.now(timezone.utc).isoformat()}).encode())
    os.close(descriptor)
    try:
        current_protocol = protocol(plan, environment, args.pilot_rows)
        protocol_file = RUN_DIR / ("preparation_protocol.json" if args.prepare else "protocol.json")
        if protocol_file.exists():
            require(read(protocol_file) == current_protocol, "Output folder uses a different protocol; choose a new run name")
        else:
            save(current_protocol, protocol_file)
        save(environment, RUN_DIR / "environment.json")
        if args.prepare:
            done = set()
            for case in cases:
                if controlled_pause(case["case_id"]):
                    return 0
                if case["data_id"] not in done:
                    prepare(case)
                    done.add(case["data_id"])
            return 0
        from memory_win import open_process, close_process, snapshot
        from job_limit import assign_job, close_job
        with prevent_idle_sleep():
            environment["prevents_idle_sleep_during_run"] = True
            save(environment, RUN_DIR / "environment.json")
            run_cases(plan, policy, cases, methods, current_protocol, args.pilot_rows)
        return 0
    except BaseException as error:
        save({"status": "controller_error", "error_type": type(error).__name__, "message": str(error),
              "traceback": traceback.format_exc(), "utc": datetime.now(timezone.utc).isoformat()}, RUN_DIR / "controller_error.json")
        raise
    finally:
        lock.unlink(missing_ok=True)


if __name__ == "__main__":
    raise SystemExit(main())
'''

**`code/notebook_report.py`**

The method aggregation and plotting defaults are preserved. The 10,000-row popcount runtime figure uses a 6.4×4.0-inch layout, plain method names, an upper-left legend and no footer; timing and failure metadata remain in the generated report. Other figure styles are unchanged. New collection and independent ARI verification read only this notebook’s freshly generated files.

In [ ]:
SOURCES['code/notebook_report.py'] = r'''import json
import math
from pathlib import Path
import re
import statistics
import textwrap

MIB = 1024 ** 2


METHODS = {
    "classix_packed": ("CLASSIX_T (packed)", "red", "x", "--"),
    "dbscan": ("DBSCAN", "blue", "o", "-."),
    "bitbirch": ("BitBIRCH", "green", "v", ":"),
    "bblean": ("BitBIRCH-Lean (reference)", "#9467bd", "s", "--"),
    "butina": ("Taylor-Butina", "#e68613", "D", "-"),
}


STUDIES = (
    ("sample_count", "Scaling with number of data points", "Number of points", "log"),
    ("popcount_separation", "Scaling with separability of data", "Max pop of seed vector", "log"),
    ("cluster_count", "Scaling with number of clusters", "Number of clusters", "linear"),
    ("dimension", "Scaling with dimensionality", "Number of dimensions", "linear"),
    ("radius", "Sensitivity to Tanimoto radius", "Tanimoto radius r", "log"),
)


WARNING = re.compile(r"\b[A-Za-z]*Warning\b")


STATUS_MARKS = {
    "memory_limit": ("M", "memory limit"),
    "timeout": ("T", "fit timeout"),
    "pending": ("P", "pending"),
    "running": ("P", "pending"),
    "incomplete_repeats": ("I", "incomplete repeats"),
    "startup_timeout": ("ST", "startup timeout"),
    "validation_timeout": ("VT", "validation timeout"),
    "algorithm_error": ("A", "algorithm error"),
    "validation_error": ("V", "validation error"),
    "worker_error": ("W", "worker error"),
}


SHORT_NAMES = {"classix_packed": "CLASSIX_T", "dbscan": "DBSCAN", "bitbirch": "BitBIRCH",
               "bblean": "Lean", "butina": "Butina"}


def status_mark(status):
    return STATUS_MARKS.get(status, ("E", "other error"))


def method_record(case, raw, method, expected_repeats, expected_ids=None):
    """Retain censored/partial evidence but require the full protocol for curves."""
    expected_ids = list(expected_ids) if expected_ids is not None else [-1] + list(range(expected_repeats))
    if expected_ids != sorted(set(expected_ids)) or not expected_ids:
        raise ValueError("Invalid expected repetition IDs")
    expected_measured_ids = [value for value in expected_ids if value >= 0]
    expected_warmup_ids = [value for value in expected_ids if value < 0]
    if not expected_measured_ids:
        raise ValueError("A successful schedule must contain a measured fit")
    expected_repeats = len(expected_measured_ids)
    runs = [run for run in raw.get("runs", []) if run["variant"] == method]
    if any(run.get("reused_from") for run in runs):
        raise ValueError("The final seed-101 benchmark cannot reuse older runs")
    if any(run["repeat"] not in expected_ids or bool(run["warmup"]) != (run["repeat"] < 0) for run in runs):
        raise ValueError(f"Run falls outside the approved schedule: {case['case_id']} / {method}")
    measured = [run for run in runs if run["status"] == "ok" and not run["warmup"]]
    warmups = [run for run in runs if run["status"] == "ok" and run["warmup"]]
    failures = [run for run in runs if run["status"] != "ok"]
    repeats = sorted(run["repeat"] for run in measured)
    full_repeats = repeats == expected_measured_ids and sorted(run["repeat"] for run in warmups) == expected_warmup_ids
    status = failures[-1]["status"] if failures else (
        "ok" if full_repeats else "incomplete_repeats" if raw.get("completed") else "pending"
    )
    eligible = status == "ok" and full_repeats
    warnings = [line for run in runs for line in run.get("logs", []) if WARNING.search(line)]
    record = {
        "case_id": case["case_id"], "study": case["study"], "x": case["x"],
        "method": method, "status": status, "eligible_for_curves": eligible,
        "expected_repeats": expected_repeats, "measured_repeats": len(measured),
        "scheduled_repeat_ids": expected_ids, "expected_measured_repeat_ids": expected_measured_ids,
        "expected_warmup_repeat_ids": expected_warmup_ids,
        "measured_repeat_ids": repeats, "expected_discarded_warmups": len(expected_warmup_ids),
        "discarded_successful_warmups": len(warmups),
        "timing_statistic": None, "runtime_variability_estimated": False,
        "warning_runs": sum(any(WARNING.search(line) for line in run.get("logs", [])) for run in runs),
        "warning_messages": sorted(set(warnings)),
        "observed_measured_seconds": [], "observed_ari": [],
        "median_seconds": None, "min_seconds": None, "max_seconds": None,
        "median_ari": None, "ari_values": [], "cluster_counts": [], "noise_counts": [],
        "aggregation_groups": [], "adjacency_connection_buffer_bytes": [],
        "minpts_marker_counts": [], "minpts_marker_object_estimates_bytes": [],
        "median_os_peak_private_mib": None, "median_os_peak_working_set_mib": None,
        "median_prefit_private_mib": None, "median_extra_private_mib": None,
        "private_peak_verified_in_fit_window": None,
        "working_set_peak_verified_in_fit_window": None,
        "partition_stable_across_successful_repeats": None,
        "terminal_evidence": [],
    }
    partitions, logical_inputs = set(), set()
    for run in measured:
        timing = run["fit_done"]["clustering_seconds"]
        quality = run["validated"]["ari"]
        if not math.isfinite(timing) or timing <= 0 or not math.isfinite(quality):
            raise ValueError(f"Invalid successful result for {case['case_id']} / {method}")
        if run["validated"].get("input_unchanged") is False:
            raise ValueError("Successful run changed its input")
        record["observed_measured_seconds"].append(timing)
        record["observed_ari"].append(quality)
        partitions.add(run["validated"]["canonical_partition_sha256"])
        logical_inputs.add(run["validated"]["logical_input_sha256"])
    if len(partitions) > 1 or len(logical_inputs) > 1:
        raise ValueError(f"Repeated fits have changed input/partition: {case['case_id']} / {method}")
    if measured:
        record["partition_stable_across_successful_repeats"] = True if len(measured) > 1 else None
        record["logical_input_sha256"] = next(iter(logical_inputs))
    for run in failures:
        evidence = {key: run[key] for key in (
            "repeat", "warmup", "status", "exit_code", "timeout_phase", "elapsed_in_phase_seconds",
            "limits", "fit_error", "validation_error", "sampled_memory"
        ) if key in run}
        record["terminal_evidence"].append(evidence)
    if not eligible:
        return record
    times = record["observed_measured_seconds"]
    record.update(median_seconds=statistics.median(times), min_seconds=min(times), max_seconds=max(times),
                  median_ari=statistics.median(record["observed_ari"]), ari_values=sorted(set(record["observed_ari"])),
                  timing_statistic="single fit" if len(times) == 1 else "median of measured fits",
                  runtime_variability_estimated=len(times) > 1)
    for destination, source in (
        ("cluster_counts", "clusters"), ("noise_counts", "noise_points"),
        ("aggregation_groups", "aggregation_groups"), ("adjacency_connection_buffer_bytes", "adjacency_bytes"),
        ("minpts_marker_counts", "minpts_marker_count"),
        ("minpts_marker_object_estimates_bytes", "minpts_marker_storage_bytes"),
    ):
        record[destination] = sorted({run["validated"][source] for run in measured
                                      if run["validated"].get(source) is not None})
    for metric, name in (("private_commit", "private"), ("working_set", "working_set")):
        key = "lifetime_peak_" + metric + "_bytes"
        peaks = [run["fit_done"]["after_fit"][key] for run in measured]
        record[f"median_os_peak_{name}_mib"] = statistics.median(peaks) / MIB
        record[f"{name}_peak_verified_in_fit_window"] = all(
            run["fit_done"]["after_fit"][key] > run["ready"]["before_fit"][key] for run in measured
        )
    baseline = [run["ready"]["before_fit"]["private_commit_bytes"] for run in measured]
    record["median_prefit_private_mib"] = statistics.median(baseline) / MIB
    if record["private_peak_verified_in_fit_window"]:
        record["median_extra_private_mib"] = statistics.median(
            run["fit_done"]["after_fit"]["lifetime_peak_private_commit_bytes"]
            - run["ready"]["before_fit"]["private_commit_bytes"] for run in measured
        ) / MIB
    return record


def measurement_note_lines(summary, study, metric):
    """State the actual repetition policy without implying single-run variation."""
    prefix = "PARTIAL RESULTS; " if not summary["complete_case_results"] else ""
    if summary.get("repetition_policy"):
        groups = {}
        for method in METHODS:
            counts = tuple(sorted({row["expected_repeats"] for row in summary["records"]
                                   if row["study"] == study and row["method"] == method}))
            groups.setdefault(counts, []).append(SHORT_NAMES[method])
        lines = []
        for counts, names in groups.items():
            statistic = " / ".join("1 fit" if count == 1 else f"median of {count} fits" for count in counts)
            lines.append("/".join(names) + ": " + statistic + ".")
        lines[0] = prefix + "Seed 101; " + lines[0]
    else:
        counts = sorted({protocol.get("measured_repeats", summary["plan"]["measured_repeats"])
                         for protocol in summary["recorded_protocols"]})
        repetitions = "/".join(str(value) for value in counts) or str(summary["plan"]["measured_repeats"])
        lines = [prefix + f"Seed 101; {repetitions} measured repetitions; medians."]
    if study == "radius":
        cases = [case for case in summary["plan"]["cases"] if case["study"] == "radius"]
        sizes = sorted({case["rows"] for case in cases})
        dimensions = sorted({case["features"] for case in cases})
        if len(sizes) != 1 or len(dimensions) != 1:
            raise ValueError("Radius figure requires one fixed dataset size and dimension")
        lines.append(f"Radius study: {sizes[0]:,} points; {dimensions[0]:,} dimensions; same dataset at every radius.")
    if study == "sample_count" and "sample_count_100000" in summary.get("excluded_case_ids", []):
        lines.append("The cancelled 100,000-point setting is excluded from this study.")
    if metric == "memory":
        lines.append("Process peaks include input and imports.")
    return lines


def figures(summary, destination):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.transforms import blended_transform_factory
    import numpy as np

    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 12,
        "axes.labelsize": 10, "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.spines.top": True, "axes.spines.right": True,
        "axes.grid": False, "figure.facecolor": "white", "axes.facecolor": "white",
        "svg.fonttype": "none", "pdf.fonttype": 42,
    })
    destination.mkdir(parents=True, exist_ok=True)
    lookup = {(row["case_id"], row["method"]): row for row in summary["records"]}
    artifacts = []
    for study, title, xlabel, time_scale in STUDIES:
        cases = sorted((case for case in summary["plan"]["cases"] if case["study"] == study), key=lambda case: case["x"])
        xs = np.array([case["x"] for case in cases], dtype=float)
        rows_by_method = {method: [lookup[(case["case_id"], method)] for case in cases] for method in METHODS}
        unavailable_rows = [(method, rows) for method, rows in rows_by_method.items()
                            if any(not row["eligible_for_curves"] for row in rows)]
        used_statuses = sorted({status_mark(row["status"]) for _, rows in unavailable_rows
                               for row in rows if not row["eligible_for_curves"]})
        key_lines = textwrap.wrap("Status: " + "; ".join(f"{mark} = {label}" for mark, label in used_statuses),
                                  width=104) if used_statuses else []
        span = float(xs[-1] - xs[0])
        padding = .05 * span if span else max(1, abs(float(xs[0])) * .05)
        for metric, key, ylabel in (
            ("runtime", "median_seconds", "Time taken (in seconds)"),
            ("ari", "median_ari", "Adjusted Rand index"),
            ("memory", "median_os_peak_private_mib", "Peak process private memory (MiB)"),
        ):
            clean_sample_runtime = study == "sample_count" and metric == "runtime"
            clean_popcount_runtime = study == "popcount_separation" and metric == "runtime"
            clean_dimension_runtime = study == "dimension" and metric == "runtime"
            clean_runtime = clean_sample_runtime or clean_popcount_runtime or clean_dimension_runtime
            note_lines = measurement_note_lines(summary, study, metric)
            note_top = .018 + .023 * (len(note_lines) - 1)
            key_top = note_top + .030 + .023 * max(0, len(key_lines) - 1)
            status_bottom = key_top + .034
            main_bottom = (.18 + .028 * len(unavailable_rows) + .023 * max(0, len(key_lines) - 1)
                           + .023 * (len(note_lines) - 1))
            fig, ax = plt.subplots(figsize=(6.4, 4.0) if clean_runtime else (6.4, 4.8))
            fig.subplots_adjust(left=.14, right=.97, top=.9, bottom=.15 if clean_runtime else main_bottom)
            handles, finite_values = [], []
            for method, (label, color, marker, linestyle) in METHODS.items():
                if clean_runtime:
                    label = label.replace(" (packed)", "").replace(" (reference)", "")
                if method == "dbscan" and study == "popcount_separation":
                    linestyle = "-." if clean_popcount_runtime else "--"
                rows = rows_by_method[method]
                ys = np.array([row[key] if row["eligible_for_curves"] else np.nan for row in rows], dtype=float)
                finite_values.extend(ys[np.isfinite(ys)].tolist())
                ax.plot(xs, ys, color=color, linestyle=linestyle, marker=marker, linewidth=1.5, markersize=6)
                handles.append(Line2D([0], [0], label=label, color=color, linestyle=linestyle, marker=marker,
                                      linewidth=1.5, markersize=6))
            ax.set_title(title)
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            # Keep failed endpoints visible; numerical curves cannot set limits.
            plot_padding = 4000 if clean_sample_runtime else (20 if clean_popcount_runtime else padding)
            ax.set_xlim(float(xs[0]) - plot_padding, float(xs[-1]) + plot_padding)
            ax.set_xticks(xs)
            ax.tick_params(axis="x", labelsize=10 if clean_popcount_runtime else (8 if study == "sample_count" else 9))
            if metric == "runtime":
                ax.set_yscale(time_scale)
                if not finite_values:
                    ax.set_ylim((.01, 10) if time_scale == "log" else (0, 1))
                elif time_scale == "linear":
                    ax.set_ylim(bottom=0)
            elif metric == "ari":
                ax.set_ylim(min(-.05, min(finite_values, default=0) - .05), 1.055)
            else:
                ax.set_ylim(bottom=0)
            location = "upper right" if study == "popcount_separation" else "upper left"
            legend_location = "upper left" if clean_runtime and not clean_sample_runtime else ("best" if finite_values else location)
            ax.legend(handles=handles, loc=legend_location, fontsize=8, frameon=True)
            if clean_popcount_runtime and finite_values:
                # Reserve enough log-axis headroom for the fixed upper-left legend.
                fig.canvas.draw()
                lower = ax.get_ylim()[0]
                box = ax.get_legend().get_window_extent(fig.canvas.get_renderer())
                fraction = (box.y0 - 6 * fig.dpi / 100 - ax.bbox.y0) / ax.bbox.height
                needed = lower * (max(finite_values) / lower) ** (1 / fraction)
                unit = 10 ** math.floor(math.log10(needed))
                ax.set_ylim(top=math.ceil(needed / unit) * unit)
            if not finite_values:
                ax.text(.5, .43, "No completed measurements", transform=ax.transAxes,
                        ha="center", va="center", fontsize=10, color="#777777")
                if metric == "memory":
                    ax.set_ylim(0, 1)
            if summary["is_pilot"]:
                ax.text(.5, .5, f"PILOT: {summary['pilot_rows']} rows per setting\nFull-data x labels show intended settings",
                        transform=ax.transAxes, ha="center", va="center", fontsize=11, color="#777777", alpha=.8)
            if not clean_runtime:
                for index, line in enumerate(note_lines):
                    fig.text(.5, note_top - .023 * index, line, ha="center", fontsize=7)
                if unavailable_rows:
                    # These are figure text rows, not a second plot or y observations.
                    # x follows the main data axis, while y is in figure coordinates.
                    status_transform = blended_transform_factory(ax.transData, fig.transFigure)
                    for index, (method, rows) in enumerate(unavailable_rows):
                        y = status_bottom + .028 * (len(unavailable_rows) - 1 - index)
                        color = METHODS[method][1]
                        fig.text(.133, y, SHORT_NAMES[method], ha="right", va="center", fontsize=7, color=color)
                        for x, row in zip(xs, rows):
                            if not row["eligible_for_curves"]:
                                mark, _ = status_mark(row["status"])
                                fig.text(x, y, mark, transform=status_transform, ha="center", va="center",
                                         fontsize=7.5, color=color, fontweight="bold")
                    for index, line in enumerate(key_lines):
                        fig.text(.5, key_top - .023 * index, line, ha="center", fontsize=7)
                else:
                    fig.text(.5, note_top + .037, "Fixed nominal settings; different methods use different clustering criteria.", ha="center", fontsize=7)
            paths = []
            for extension in ("png", "pdf", "svg"):
                path = destination / f"{study}_{metric}.{extension}"
                fig.savefig(path, dpi=300, facecolor="white")
                paths.append(str(path.resolve()))
            plt.close(fig)
            artifacts.append({"study": study, "metric": metric, "paths": paths,
                              "yscale": time_scale if metric == "runtime" else "linear",
                              "measurement_notes": note_lines,
                              "x_limits": [float(xs[0]) - plot_padding, float(xs[-1]) + plot_padding],
                              "status_rows": [{"method": method, "points": [
                                  {"x": row["x"], "status": row["status"], "mark": status_mark(row["status"])[0]}
                                  for row in rows if not row["eligible_for_curves"]]}
                                  for method, rows in unavailable_rows]})
    return artifacts


def collect_fresh(root, run_directory):
    plan = json.loads((root/"plan.json").read_text())
    policy = json.loads((root/"design/repetition_policy.json").read_text())
    records, missing, protocols = [], [], []
    pilot_values = set()
    raws = {}
    for case in plan["cases"]:
        path = run_directory/"results"/(case["case_id"]+".json")
        raw = json.loads(path.read_text()) if path.is_file() else {}
        raws[case["case_id"]] = raw
        if raw:
            assert raw["case"] == case
            pilot_values.add(raw.get("pilot_rows"))
            protocols.append(raw["protocol"])
        if not raw.get("completed"):
            missing.append(case["case_id"])
    assert len(pilot_values) <= 1
    pilot_rows = next(iter(pilot_values)) if pilot_values else None
    for case in plan["cases"]:
        raw = raws[case["case_id"]]
        for method in METHODS:
            ids = [0] if pilot_rows else policy["schedules"][case["case_id"]][method]
            records.append(method_record(case, raw, method, len([i for i in ids if i >= 0]), ids))
    return {"plan": plan, "records": records, "complete_case_results": not missing,
            "missing_or_unfinished_cases": missing, "repetition_policy": policy,
            "excluded_case_ids": ["sample_count_100000"], "recorded_protocols": protocols,
            "is_pilot": bool(pilot_rows), "pilot_rows": pilot_rows,
            "data_origin": "Fresh generation and measurements from this notebook; no historical result values loaded."}


def verify_fresh(root, run_directory):
    import numpy as np
    from sklearn.metrics import adjusted_rand_score
    import hashlib
    plan = json.loads((root/"plan.json").read_text())
    policy = json.loads((root/"design/repetition_policy.json").read_text())
    from reproduce import method_finished
    successful, failures, missing = [], [], []
    for case in plan["cases"]:
        path = run_directory/"results"/(case["case_id"]+".json")
        if not path.is_file():
            missing.append(case["case_id"])
            continue
        raw = json.loads(path.read_text())
        assert raw["case"] == case
        manifest = json.loads((run_directory/"data"/case["data_id"]/"manifest.json").read_text())
        truth = np.load(run_directory/"data"/case["data_id"]/manifest["files"]["truth"]["file"])
        for method in METHODS:
            runs = [r for r in raw["runs"] if r["variant"] == method]
            ids = [0] if raw.get("pilot_rows") else policy["schedules"][case["case_id"]][method]
            if not method_finished(runs, ids):
                missing.append(case["case_id"]+"/"+method)
            for run in runs:
                assert run["protocol"] == raw["protocol"]
                if run["status"] != "ok":
                    failures.append({"case_id": case["case_id"], "method": method, "status": run["status"]})
                    continue
                checked = run["validated"]
                relative = checked["output_relative_assignments_file"]
                label_path = (run_directory/relative).resolve()
                assert label_path.is_relative_to(run_directory.resolve())
                labels = np.load(label_path, allow_pickle=False)
                expected_n = raw.get("pilot_rows") or case["rows"]
                assert labels.shape == (expected_n,)
                score = adjusted_rand_score(truth[:len(labels)], labels)
                assert math.isclose(score, checked["ari"], rel_tol=1e-12, abs_tol=1e-12)
                _, counts = np.unique(labels[labels >= 0], return_counts=True)
                assert len(counts) == checked["clusters"]
                assert int(np.count_nonzero(labels < 0)) == checked["noise_points"]
                mapping = {}
                canonical = np.asarray([mapping.setdefault(int(v), len(mapping)) for v in labels], dtype=np.int64)
                assert hashlib.sha256(memoryview(canonical).cast("B")).hexdigest() == checked["canonical_partition_sha256"]
                if not raw.get("pilot_rows"):
                    assert checked["input_unchanged"] is True
                    assert checked["logical_input_sha256"] == manifest["files"]["int32"]["sha256"]
                successful.append({"case_id": case["case_id"], "method": method, "repeat": run["repeat"], "ari": score})
    return {"successful_assignments_checked": len(successful), "successful": successful,
            "terminal_failures": failures, "missing_or_incomplete": missing,
            "ari_reference": "sklearn.metrics.adjusted_rand_score; freshly generated truth"}
'''


In [ ]:
UPSTREAM_SOURCE_SHA256 = json.loads(r'''{
  "code/generate_data_snapshot.py": "519fd5bad59501c493e8189d3dc13a6f62c5abe917a9cc90e9cc82d85cd14d0d",
  "code/worker.py": "5b29c32d03708880e3c29532545c7f9d860af0be16a90e16e3105029be45ec0c",
  "code/quality.py": "411c4127263a6bb7b1b5abb369dad0f031dfc069498cc5e28c18d4a9f33b2a9f",
  "code/state_checks.py": "0fd7929b44d87b2ac9fd44d2fad248d8569f2c87e229aeca4ffbfd28093ff61c",
  "code/memory_win.py": "4df105252a835e9abf9a858956d377bcb88742c3d4945520d5b85d072b675c99",
  "code/job_limit.py": "f31f41203dcea0f607ccc90fed2e70f9b3a37f5695289f871147dd2d28ca58d3",
  "code/classix/classix_t_packed_adjbits.py": "09173bad948bc4826bda083814bdff7883eeb97bccc29273a8d9dbb19dec8ff7",
  "code/classix/classix_t_reference_adjbits.py": "e7c6972a5985677ad0ce2483a12844300e5652dfcc59a52b5512afc1238ae710",
  "code/classix/packed_adjacency.py": "b85b5425dac6ec625f8a5ade9d4de0a26cf649c50a4f54de389322ec0d103146",
  "code/classix/packed_native.py": "fb0815c9e57299994af4b62bab6052cb4906add82a971985b139d700575e586c",
  "code/classix/packed_kernel.c": "65bfeb98b9d18026fb9e6d2ebc3be5db094adf6ab99e6012ab8017b565c6048e",
  "code/classix/spsubmatxvec.c": "10c53fe53a9f1ef92d3495574715190cd1c65395eff50abc238b53cac7bcc2a0",
  "code/classix/LICENSE": "9d739b5199e77aef4757893620fe6a0c0c5549ffddb13f9f98100fb003c93ddb",
  "vendor/bitbirch/bitbirch/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
  "vendor/bitbirch/bitbirch/bitbirch.py": "ff88e988306854ff5a45edcd9e656a6ae79f398dd61a4b60ade89f61d6d33b22",
  "vendor/bitbirch/bitbirch/pruning.py": "9eb65eb920112a10767d516abc5ca6064eee578b5969ff1489370ef1719ce0f5",
  "vendor/bitbirch/LICENSE": "6557adc4bc1d59232e27567a185dd70cf2ecca3154018b0be61ea29cf9e814e2",
  "code/reproduce.py": "c8aebeb59db486395bafdc8f345254f98794344c0428a8965ccf2e2a9197feb2",
  "code/render_results.py": "dd22ea2e4bf23dbf4db3f76601737b4f22ac27a5cc10e03651b54b0f24223098"
}''')
# These identify source code, not saved results.
SOURCE_HASHES = {name: hashlib.sha256(text.encode("utf-8")).hexdigest() for name, text in SOURCES.items()}

## Materialize sources, install dependencies and compile native kernels

Prerequisites are Windows x64, Python 3.12, and Visual Studio Build Tools with the x64 C++ toolset and Windows SDK. Package installation requires internet access. Original BitBIRCH’s pinned core is included as readable source with its license; its upstream package metadata requests NumPy <2, while this benchmark deliberately uses the inspected core under NumPy 2.5.3. The other methods use the pinned package versions below. BBLean’s native Windows wheel is installed from PyPI, not embedded.

Sources and design files are hashed before execution. Re-execution reuses identical sources and compatible checkpoints; a conflicting file is rejected instead of silently changing a running protocol. Delete neither checkpoints nor control files automatically: remove `PAUSE` or `STOP_CURRENT` yourself when ready to resume. Both are placed inside the printed run directory.

In [ ]:
PINNED_REQUIREMENTS = r'''cloudpickle==3.1.2
colorama==0.4.6
contourpy==1.4.0
cycler==0.12.1
fonttools==4.65.0
joblib==1.6.0
kiwisolver==1.5.1
markdown-it-py==4.2.0
matplotlib==3.11.2
mdurl==0.1.2
narwhals==2.26.0
numpy==2.5.3
packaging==26.3
pillow==12.3.0
psutil==7.2.2
Pygments==2.21.0
pyparsing==3.3.3
python-dateutil==2.9.0.post0
rdkit==2026.3.6
rich==15.0.0
scikit-learn==1.9.1
scipy==1.18.1
six==1.17.0
threadpoolctl==3.7.0
tqdm==4.70.1
typing_extensions==4.16.0
'''

BBLEAN_WHEEL = json.loads(r'''{
  "filename": "bblean-0.10.3-cp312-cp312-win_amd64.whl",
  "url": "https://files.pythonhosted.org/packages/b7/2b/f7f088fe70597d72f3705ee050e71110ca3a56e8bb5c1f954025c6f9fac9/bblean-0.10.3-cp312-cp312-win_amd64.whl",
  "sha256": "36d10341de071024a0585020d5907d12950c93e251c03f74e92a14eee6a78fd9"
}''')

## Post-fit AMI and silhouette implementation

The following readable sources evaluate only freshly saved assignments. They are materialized with the other sources; metric scoring runs after timed fitting and is excluded from clustering runtimes.


**`code/saved_scores.py`**


In [ ]:
SOURCES['code/saved_scores.py'] = '''"""AMI and sampled-query/full-reference Tanimoto silhouettes of SAVED fits.

No estimator is imported or refitted. All writes stay in --output-dir.
Repeated partitions must agree; differing repetitions require an explicit analysis.
"""
from __future__ import annotations
import os
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMBA_NUM_THREADS'):
    os.environ[key] = '1'
import argparse
from collections import Counter, defaultdict
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import sys
import subprocess
import time
import numpy as np
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score

CODE_ROOT = Path(__file__).resolve().parents[1]
ROOT = CODE_ROOT / 'derived/scores'
RESULTS = ROOT / 'results'
ARGS = None
METHODS = ['classix_packed','dbscan','bitbirch','bblean','butina']
SEED, SAMPLE_SIZE, BLOCK_SIZE = 101, 2000, 64


def read(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024**2),b''): h.update(chunk)
    return h.hexdigest()


def array_sha(array):
    return hashlib.sha256(memoryview(np.ascontiguousarray(array)).cast('B')).hexdigest()


def save(path, value):
    path = Path(path)
    assert path.resolve().is_relative_to(ROOT.resolve()), path
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value, indent=2, allow_nan=False)+'\\n',encoding='utf-8')
    for attempt in range(60):
        try:
            temp.replace(path)
            break
        except PermissionError:
            if attempt == 59: raise
            time.sleep(.05)


def utc():
    return datetime.now(timezone.utc).isoformat()


def progress(stage, **fields):
    value = dict(stage=stage,pid=os.getpid(),updated_utc=utc(),**fields)
    save(ROOT/'progress.json', value)
    print(json.dumps(value), flush=True)
    if (ROOT/'STOP').exists(): raise SystemExit('Paused at a safe checkpoint; remove STOP explicitly to resume.')


def canonical(labels):
    _,first,inverse = np.unique(labels,return_index=True,return_inverse=True)
    order = np.argsort(first)
    remap = np.empty(len(order), dtype=np.int64)
    remap[order] = np.arange(len(order), dtype=np.int64)
    return remap[inverse]


def policy(scope, method):
    return 'nonnoise_cluster_size_strictly_greater_than_10' if scope=='landrum' and method in ('bitbirch','bblean') else 'all_nonnoise_clusters'


def retained(labels, rule):
    keep = labels >= 0
    names, counts = np.unique(labels[keep],return_counts=True)
    if rule=='nonnoise_cluster_size_strictly_greater_than_10':
        keep &= np.isin(labels,names[counts>10])
    names,inverse,counts = np.unique(labels[keep], return_inverse=True,return_counts=True)
    compact = np.full(len(labels),-1,dtype=np.int32)
    compact[keep] = inverse.astype(np.int32)
    return keep,compact,counts.astype(np.int64)


def prepare_inventory():
    """Check raw evidence and aligned arrays; derive exact AMI without clustering."""
    RESULTS.mkdir(parents=True,exist_ok=True)
    sources, datasets, records, states = {}, {}, [], {}
    ami_cache = {}
    def source(path):
        relative = str(Path(path).resolve())
        if relative not in sources: sources[relative] = sha(path)
        return relative
    source(Path(__file__))
    protected = {}
    def dataset(scope,case):
        key = scope+'_'+case['data_id']
        if key in datasets: return datasets[key]
        directory = ARGS.data_dir/case['data_id']
        manifest_path = directory/'manifest.json'; manifest=read(manifest_path)
        assert manifest['shape']==[case['rows'],case['features']]
        if scope=='synthetic': assert manifest['parameters']==case['generator']
        arrays = {}
        for kind in ('packed_little','truth'):
            path = directory/manifest['files'][kind]['file']
            value = np.load(path,allow_pickle=False,mmap_mode='r')
            assert array_sha(value)==manifest['files'][kind]['sha256'], (key,kind)
            arrays[kind]=value; source(path)
        packed,truth = arrays['packed_little'],arrays['truth']
        assert packed.dtype==np.uint8 and packed.shape==(case['rows'],(case['features']+7)//8)
        assert truth.shape==(case['rows'],) and truth.dtype==np.int32
        if case['features']%8:
            assert not np.any(packed[:,-1] & np.uint8(255 ^ ((1<<(case['features']%8))-1)))
        rows = min(ARGS.pilot_rows, case['rows']) if ARGS.pilot_rows else case['rows']
        packed, truth = packed[:rows], truth[:rows]
        logical_hash = manifest['files']['int32']['sha256']
        if rows != manifest['shape'][0]:
            dense = np.unpackbits(packed, axis=1, bitorder='little', count=case['features']).astype(np.int32)
            logical_hash = array_sha(dense)
        item = dict(key=key,scope=scope,data_id=case['data_id'],rows=rows,features=case['features'],
                    directory=str(directory.resolve()),
                    packed_file=source(directory/'packed_little.npy'), truth_file=source(directory/'truth.npy'),
                    manifest_file=source(manifest_path),manifest_sha256=sha(manifest_path),
                    packed_array_sha256=array_sha(packed),truth_array_sha256=array_sha(truth),
                    logical_int32_array_sha256=logical_hash,
                    reference_classes=int(np.unique(truth).size))
        datasets[key]=item
        return item
    plan = read(ARGS.plan); source(ARGS.plan)
    cases = plan['cases']
    if ARGS.cases:
        assert ARGS.pilot_rows, 'A reduced case grid must be explicitly marked as pilot.'
        requested = set(ARGS.cases)
        assert len(requested) == len(ARGS.cases)
        assert requested.issubset({c['case_id'] for c in cases})
        cases = [c for c in cases if c['case_id'] in requested]
    assert bool(plan.get('diagnostic_pilot', False)) == bool(ARGS.pilot_rows) or ARGS.scope == 'synthetic'
    methods = METHODS if ARGS.scope == 'synthetic' else ['classix_packed','bitbirch','bblean']
    contract = dict(scope=ARGS.scope, is_pilot=bool(ARGS.pilot_rows), pilot_rows=ARGS.pilot_rows,
                    expected_cases=[c['case_id'] for c in cases], expected_methods=methods,
                    complete_case_results=True)
    from render_scores import summary_contract
    summary_contract(contract)
    policy_candidates = [ARGS.plan.parent/'design/repetition_policy.json',
                         ARGS.plan.parent/'repetition_amendment.json',
                         ARGS.plan.parent/'records/repetition_amendment.json']
    amendment_file = next((p for p in policy_candidates if p.exists()), policy_candidates[0])
    amendment = read(amendment_file) if amendment_file.exists() else {}
    if amendment: source(amendment_file)
    for scope in [ARGS.scope]:
        for i,case in enumerate(cases):
            progress('inventory_and_exact_ami',scope=scope,case=i+1,total_cases=len(cases),case_id=case['case_id'])
            input_info = dataset(scope,case)
            truth=np.load(input_info['truth_file'],allow_pickle=False)[:input_info['rows']]
            raw_path = ARGS.results_dir/(case['case_id']+'.json')
            raw = read(raw_path); source(raw_path)
            assert raw['case']==case and raw.get('completed') is True
            if raw.get('data_manifest_sha256'):
                assert raw['data_manifest_sha256']==input_info['manifest_sha256']
            if raw.get('reproduction'):
                assert all(run.get('protocol')==raw['protocol'] for run in raw['runs'])
            if scope == 'synthetic': assert raw.get('pilot_rows') == ARGS.pilot_rows
            elif not ARGS.pilot_rows: assert raw.get('pilot_rows') is None

            for method in METHODS if scope=='synthetic' else ['classix_packed','bitbirch','bblean']:
                runs = [r for r in raw['runs'] if r['variant']==method]
                schedules = ({m:[0] for m in methods} if ARGS.pilot_rows else
                             case.get('method_repeat_ids') if scope=='landrum' else
                             amendment.get('schedules', {}).get(case['case_id']))
                assert schedules is not None, 'Missing authoritative repeat schedule.'
                if raw.get('method_repeat_ids') is not None:
                    assert raw['method_repeat_ids']==schedules, 'Saved schedule differs from the intended schedule.'
                expected = schedules[method]
                assert [r['repeat'] for r in runs]==expected[:len(runs)]
                failed = [r for r in runs if r['status']!='ok']
                base=dict(scope=scope,case_id=case['case_id'],data_key=input_info['key'],method=method,
                          study=case['study'],x=case['x'] if scope=='synthetic' else case['grid']['similarity'],
                          n_total=input_info['rows'],n_features=case['features'],source_record=source(raw_path),
                          rule=policy(scope,method),case_parameters=case)
                if failed:
                    assert len(failed)==1 and failed[0] is runs[-1]
                    assert failed[0]['status'] not in ('pending','running','ok'), 'Fit is not terminal.'
                    records.append(dict(base,status=failed[0]['status'],ami_all=None,ami_retained=None,
                                        silhouette_mean=None,silhouette_se=None,silhouette_status='not_available',
                                        silhouette_reason='No complete measured repetition schedule: '+str(failed[0]['status'])+'.',
                                        n_retained=None,coverage=None,n_clusters=None,n_queries=0,exact_silhouette=False))
                    continue
                assert [r['repeat'] for r in runs]==expected
                measured=[r for r in runs if not r['warmup']]
                assert measured and all(r['repeat']>=0 for r in measured)
                reference=None; evidence=[]; partitions=set()
                for run in measured:
                    checked=run['validated']
                    relative = checked.get('output_relative_assignments_file')
                    labels_path = ARGS.results_dir.parent/relative if relative else Path(checked['assignments_file'])
                    assert labels_path.is_file() and labels_path.resolve().is_relative_to(raw_path.parent.resolve())
                    labels=np.load(labels_path,allow_pickle=False)
                    assert labels.dtype==np.int64 and labels.shape==truth.shape and np.all(labels>=-1)
                    canonical_hash=array_sha(canonical(labels))
                    assert canonical_hash==checked['canonical_partition_sha256']
                    assert checked['logical_input_sha256']==input_info['logical_int32_array_sha256'] and (checked['input_unchanged'] is True or (ARGS.pilot_rows and checked['input_unchanged'] is None))
                    assert int((labels<0).sum())==checked['noise_points']
                    assert int(np.unique(labels[labels>=0]).size)==checked['clusters']
                    ari=float(adjusted_rand_score(truth,labels))
                    assert abs(ari-checked['ari'])<1e-12
                    noise_hash=array_sha(labels<0)
                    partitions.add((canonical_hash,noise_hash))
                    evidence.append(dict(repeat=run['repeat'],assignment_file=source(labels_path),
                                         assignment_file_sha256=sha(labels_path),assignment_array_sha256=array_sha(labels),
                                         canonical_partition_sha256=canonical_hash,noise_mask_sha256=noise_hash,ari=ari))
                    if reference is None: reference=labels
                assert len(partitions)==1, 'Repeated partitions differ; score every partition rather than choosing one.'
                canon,noise=next(iter(partitions)); rule=base['rule']
                state_key=hashlib.sha256((input_info['key']+'|'+input_info['packed_array_sha256']+'|'+input_info['truth_array_sha256']+'|'+canon+'|'+noise+'|'+rule).encode()).hexdigest()[:24]
                if state_key not in states:
                    keep,compact,counts=retained(reference,rule)
                    retained_n=int(keep.sum()); cluster_n=len(counts)
                    key=(input_info['key'],canon,noise)
                    if key not in ami_cache:
                        ami_cache[key]=float(adjusted_mutual_info_score(truth,reference,average_method='arithmetic'))
                    valid=2<=cluster_n<=retained_n-1
                    reason=None if valid else ('Fewer than two retained clusters.' if cluster_n<2 else 'Every retained point is a singleton cluster.')
                    order=np.random.default_rng(SEED).permutation(len(reference))
                    query=order[keep[order]][:min(SAMPLE_SIZE,retained_n)] if valid else np.empty(0,dtype=np.int64)
                    query=np.sort(query).astype(np.int64)
                    target=RESULTS/'states'/f'{state_key}.npz'
                    target.parent.mkdir(parents=True,exist_ok=True)
                    np.savez_compressed(target,labels=compact,counts=counts,queries=query)
                    states[state_key]=dict(state_key=state_key,data_key=input_info['key'],rule=rule,
                        canonical_partition_sha256=canon,noise_mask_sha256=noise,
                        source_assignment=evidence[0],state_file=target.relative_to(ROOT).as_posix(),
                        compact_array_sha256=array_sha(compact),retained_mask_sha256=array_sha(keep),
                        counts_array_sha256=array_sha(counts),query_array_sha256=array_sha(query),
                        n_total=len(reference),n_retained=retained_n,coverage=retained_n/len(reference),
                        n_clusters=cluster_n,n_queries=len(query),exact_silhouette=len(query)==retained_n,
                        ami_all=ami_cache[key],
                        ami_retained=float(adjusted_mutual_info_score(truth[keep],reference[keep],average_method='arithmetic')) if retained_n else None,
                        silhouette_status='pending' if valid else 'undefined',silhouette_reason=reason,
                        silhouette_mean=None,silhouette_se=None)
                state=states[state_key]
                fields={k:state[k] for k in ('ami_all','ami_retained','n_retained','coverage','n_clusters','n_queries',
                        'exact_silhouette','silhouette_status','silhouette_reason','silhouette_mean','silhouette_se')}
                records.append(dict(base,status='ok',state_key=state_key,measured_repeats=len(measured),
                                    repeated_partitions_identical=True,assignments=evidence,**fields))
    assert len(records)==len(cases)*len(methods)
    protocol=dict(seed=SEED,query_limit=SAMPLE_SIZE,block_size=BLOCK_SIZE,
        ami='sklearn adjusted_mutual_info_score, arithmetic normalizer, full reference labels/assignments including noise as the recorded -1 category; no sample and no tuning.',
        silhouette='Tanimoto/Jaccard distance on binary fingerprints; exact distances to every retained cluster member for each uniformly selected query. Self excluded, singleton silhouette zero, a=b=0 gives zero; valid only for 2 <= K <= N_retained-1.',
        query_sampling='One shared random row priority per dataset from default_rng(101). For each retained population take its first min(2000,N_retained) rows; sort query IDs for computation. Equal populations use identical queries. Every population is a uniform sample without replacement. Query unions allow distance reuse without changing any mean.',
        estimate='Mean of sampled individual silhouettes estimates the full retained-population mean, not a silhouette computed only within a sampled submatrix.',
        uncertainty='Design-based Monte Carlo standard error = sample SD/sqrt(m)*sqrt(1-m/N_retained). Exact when m=N_retained; pointwise approximate normal bands describe query sampling only, not clustering stability or statistical significance.',
        populations={'synthetic':'Exclude recorded noise (-1); keep all nonnoise clusters, including singletons.',
                     'landrum_classix':'All final nonnoise clusters, matching the current minPts=10 study.',
                     'landrum_bitbirch_lean':'Keep final clusters strictly larger than 10 (>10), matching the existing minimum-size quality evaluation. Do not reinterpret as >=10.'},
        aggregation='All measured assignments are checked. Repeated partitions must agree; otherwise scoring stops explicitly for a repetition-aware analysis, rather than choosing a fit. Score each distinct partition/population once. Repeated deterministic fits are not independent observations.',
        scope=f'{ARGS.scope}: {len(cases)} settings, {len(methods)} methods; pilot={bool(ARGS.pilot_rows)}. No estimator fits or tuning.',
        definitions={'silhouette':'https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.silhouette_samples.html',
                     'ami':'https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.adjusted_mutual_info_score.html'})
    inventory=dict(schema=2,**contract,created_utc=utc(),protocol=protocol,datasets=datasets,states=states,records=records,
                   input_file_sha256=sources,protected_file_sha256=protected,
                   environment=dict(python=sys.version,platform=platform.platform(),
                                    packages={p:importlib.metadata.version(p) for p in ('numpy','scipy','scikit-learn','matplotlib','threadpoolctl')}))
    save(ROOT/'protocol.json',protocol)
    save(RESULTS/'inventory.json',inventory)
    refresh_summary(inventory)
    progress('exact_ami_complete',records=len(records),complete_settings=sum(r['status']=='ok' for r in records),failed_settings=sum(r['status']!='ok' for r in records),distinct_states=len(states))
    return inventory


def scoring_identity(inventory):
    identity={name:sha(CODE_ROOT/'code'/name) for name in ('saved_scores.py','native_silhouette.py','silhouette_kernel.c')}
    identity['environment']=inventory['environment']
    identity['silhouette_kernel.dll']=sha(CODE_ROOT/'native/silhouette_kernel.dll')
    return identity


def refresh_summary(inventory):
    records=[]
    producer=scoring_identity(inventory)
    for record in inventory['records']:
        record=dict(record)
        if record['status']=='ok':
            score_path=RESULTS/'scores'/(record['state_key']+'.json')
            if score_path.exists():
                score=read(score_path)
                state=inventory['states'][record['state_key']]
                valid=(score.get('scoring_source')==producer and all(score.get(k)==state[k] for k in
                       ('query_array_sha256','compact_array_sha256','counts_array_sha256')))
                if valid:
                    for key in ('silhouette_mean','silhouette_se','silhouette_status','silhouette_reason'):
                        record[key]=score[key]
        records.append(record)
    summary=dict(schema=2,**{k:inventory[k] for k in ('scope','is_pilot','pilot_rows','expected_cases','expected_methods','complete_case_results')},updated_utc=utc(),protocol=inventory['protocol'],records=records,
                 counts={'settings':len(records),'successful_fits':sum(r['status']=='ok' for r in records),
                         'silhouette_status':dict(Counter(r['silhouette_status'] for r in records))},
                 limitations=['AMI uses synthetic truth or Landrum target categories; Landrum target categories need not uniquely define molecular clusters. The same SMILES may occur under multiple targets.',
                              'Silhouette uses method-specific retained populations; read coverage alongside scores. All-point AMI has the same full population for every method in each setting.',
                              'Sample-query silhouette is explicitly an estimate unless all retained rows were evaluated. No clustering reruns or timing claims are made.'])
    save(RESULTS/'summary.json',summary)


def evaluate(inventory, only_datasets=None):
    from native_silhouette import distance_block, prepare_packed, prepare_labels, reduce_block
    producer={name:sha(CODE_ROOT/'code'/name) for name in ('saved_scores.py','native_silhouette.py','silhouette_kernel.c')}
    producer = scoring_identity(inventory)
    save(RESULTS/'native_build.json', {p.name:sha(p) for p in (CODE_ROOT/'native').glob('*') if p.is_file()})
    save(RESULTS/'scoring_source.json',producer)
    grouping=defaultdict(list)
    for key,state in inventory['states'].items(): grouping[state['data_key']].append(state)
    for dataset_index,(data_key,states) in enumerate(grouping.items()):
        if only_datasets and data_key not in only_datasets: continue
        info=inventory['datasets'][data_key]
        packed=np.load(info['packed_file'],allow_pickle=False)[:info['rows']]
        assert array_sha(packed)==info['packed_array_sha256']
        reference=prepare_packed(packed,n_features=info['features'])
        active=[]
        for state in states:
            target=RESULTS/'scores'/(state['state_key']+'.json')
            if target.exists():
                old=read(target)
                assert old['scoring_source']==producer and all(old[k]==state[k] for k in ('query_array_sha256','compact_array_sha256','counts_array_sha256')), 'Stale score cache; use a fresh output directory.'
                continue
            if state['silhouette_status']=='undefined':
                save(target,dict(state,scoring_source=producer,computed_utc=utc()))
                continue
            with np.load(ROOT/state['state_file'],allow_pickle=False) as values:
                labels,counts,queries=values['labels'],values['counts'],values['queries']
            assert array_sha(labels)==state['compact_array_sha256'] and array_sha(counts)==state['counts_array_sha256'] and array_sha(queries)==state['query_array_sha256']
            active.append(dict(state=state,labels=labels,counts=counts,native_labels=prepare_labels(labels,counts),queries=queries,
                               silhouette=np.full(len(queries),np.nan),a=np.full(len(queries),np.nan),b=np.full(len(queries),np.nan)))
        if not active:
            refresh_summary(inventory)
            continue
        union=np.unique(np.concatenate([item['queries'] for item in active]))
        started=time.perf_counter()
        progress('silhouette_dataset',dataset=data_key,dataset_index=dataset_index+1,datasets=len(grouping),
                 rows=info['rows'],features=info['features'],partitions=len(active),union_queries=len(union))
        for start in range(0,len(union),BLOCK_SIZE):
            query=union[start:start+BLOCK_SIZE]
            distances=distance_block(reference,query)
            for item in active:
                where=np.searchsorted(item['queries'],query)
                good=where<len(item['queries'])
                good[good] &= item['queries'][where[good]]==query[good]
                if not good.any(): continue
                positions=where[good]
                block=distances if good.all() else np.ascontiguousarray(distances[good])
                q=query if good.all() else query[good]
                values,a,b=reduce_block(block,item['native_labels'],q,check_distances=False)
                item['silhouette'][positions]=values; item['a'][positions]=a; item['b'][positions]=b
            if start==0 or (start//BLOCK_SIZE+1)%8==0:
                progress('silhouette_block',dataset=data_key,queries_done=min(start+BLOCK_SIZE,len(union)),
                         total_queries=len(union),partitions=len(active),elapsed_seconds=time.perf_counter()-started)
        for item in active:
            state=item['state']; values=item['silhouette']; n=state['n_retained']; m=len(values)
            assert np.isfinite(values).all() and np.all((values>=-1-1e-12)&(values<=1+1e-12))
            mean=float(values.mean())
            se=0.0 if m==n else float(values.std(ddof=1)/np.sqrt(m)*np.sqrt(1-m/n))
            arrays_path=RESULTS/'samples'/(state['state_key']+'.npz')
            arrays_path.parent.mkdir(parents=True,exist_ok=True)
            np.savez_compressed(arrays_path,queries=item['queries'],silhouette=values,a=item['a'],b=item['b'])
            save(RESULTS/'scores'/(state['state_key']+'.json'),dict(state,silhouette_status='ok',silhouette_reason=None,
                 silhouette_mean=mean,silhouette_se=se,scoring_source=producer,computed_utc=utc(),
                 samples_file=arrays_path.relative_to(ROOT).as_posix(),samples_file_sha256=sha(arrays_path),
                 sample_silhouette_array_sha256=array_sha(values),dataset_elapsed_seconds=time.perf_counter()-started))
        refresh_summary(inventory)
        progress('silhouette_dataset_complete',dataset=data_key,elapsed_seconds=time.perf_counter()-started)
    refresh_summary(inventory)
    remaining=sum(row['silhouette_status']=='pending' for row in read(RESULTS/'summary.json')['records'])
    progress('complete' if remaining==0 else 'partial',remaining_silhouette_settings=remaining)


def main():
    global ARGS, ROOT, RESULTS
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--scope', choices=['synthetic','landrum'], required=True)
    for name in ('plan','results-dir','data-dir','output-dir'):
        parser.add_argument('--'+name, type=Path, required=True)
    parser.add_argument('--pilot-rows',type=int)
    parser.add_argument('--cases',nargs='+')
    ARGS=parser.parse_args()
    if ARGS.pilot_rows is not None: assert ARGS.pilot_rows>=2
    for name in ('plan','results_dir','data_dir','output_dir'):
        setattr(ARGS,name,getattr(ARGS,name).resolve())
    ROOT=ARGS.output_dir; RESULTS=ROOT/'results'
    assert ROOT != CODE_ROOT
    assert not ROOT.is_relative_to(ARGS.data_dir) and not ROOT.is_relative_to(ARGS.results_dir)
    assert not list(ARGS.results_dir.parent.rglob('run.lock')), 'Timed fits must finish before scoring.'
    inventory=prepare_inventory()  # Recheck input and assignment integrity on every invocation.
    evaluate(inventory)
    for path, expected in inventory['input_file_sha256'].items():
        assert sha(path)==expected, 'Input changed during scoring: '+path
    summary=read(RESULTS/'summary.json')
    assert not any(r['silhouette_status']=='pending' for r in summary['records'])
    subprocess.run([sys.executable,'-B',str(CODE_ROOT/'code/render_scores.py'),'--root',str(ROOT)],check=True)
    save(ROOT/'verification.json',dict(status='passed',checked_inputs=len(inventory['input_file_sha256']),
         settings=len(summary['records']),summary_sha256=sha(RESULTS/'summary.json'),
         definitions='Arithmetic AMI and sampled-query/full-reference Tanimoto silhouette.',
         timed_fits_rerun=False,checked_utc=utc()))


if __name__=='__main__': main()
'''


**`code/native_silhouette.py`**


In [ ]:
SOURCES['code/native_silhouette.py'] = '''"""Serial native packed Tanimoto/Jaccard distances and silhouette reduction.

Use prepare_packed once per dataset, prepare_labels once per partition, then
distance_block and reduce_block on at most 128 query indices per block. Every
retained reference row participates. Do not mutate prepared arrays afterwards.
No query sampling or clustering occurs in this module.
"""
from __future__ import annotations

import ctypes as ct
from dataclasses import dataclass
import hashlib
from pathlib import Path
import platform

import numpy as np

ROOT = Path(__file__).resolve().parents[1]
DLL_PATH = ROOT / "native" / ("silhouette_kernel.dll" if platform.system() == "Windows" else "libsilhouette_kernel.so")
MAX_QUERIES = 128
_library = None


def _native():
    global _library
    if _library is None:
        library = ct.CDLL(str(DLL_PATH))
        u8 = ct.POINTER(ct.c_uint8)
        i32 = ct.POINTER(ct.c_int32)
        i64 = ct.POINTER(ct.c_int64)
        f64 = ct.POINTER(ct.c_double)
        library.packed_row_popcounts.argtypes = [u8, ct.c_int64, ct.c_int64, i64]
        library.packed_row_popcounts.restype = ct.c_int
        library.tanimoto_distance_block.argtypes = [u8, i64, ct.c_int64, ct.c_int64, i64, ct.c_int64, f64]
        library.tanimoto_distance_block.restype = ct.c_int
        library.reduce_silhouette_block.argtypes = [f64, ct.c_int64, ct.c_int64, i32, i64, ct.c_int64, i64, f64, f64, f64]
        library.reduce_silhouette_block.restype = ct.c_int
        library.silhouette_has_hardware_popcount.argtypes = []
        library.silhouette_has_hardware_popcount.restype = ct.c_int
        _library = library
    return _library


def _pointer(array, ctype):
    return array.ctypes.data_as(ct.POINTER(ctype))


def _check(code, operation):
    if code:
        raise RuntimeError(f"Native {operation} failed with status {code}")


def _indices(indices, n):
    value = np.asarray(indices)
    if value.ndim != 1 or value.dtype.kind not in "iu":
        raise TypeError("Query indices must be a one-dimensional integer array")
    if len(value) > MAX_QUERIES:
        raise ValueError(f"At most {MAX_QUERIES} query rows are allowed per distance/reduction block")
    if len(value) and (np.any(value < 0) or np.any(value >= n)):
        raise IndexError("Query index outside the reference rows")
    return np.ascontiguousarray(value, dtype=np.int64)


def validate_packed(packed, n_features=None):
    """Validate packing once; optional true feature count checks zero tail bits."""
    value = np.asarray(packed)
    if value.dtype != np.uint8 or value.ndim != 2 or not value.flags.c_contiguous:
        raise TypeError("packed must be a C-contiguous two-dimensional uint8 array")
    if not all(value.shape):
        raise ValueError("packed must contain at least one row and one byte per row")
    if n_features is not None:
        if not isinstance(n_features, (int, np.integer)) or n_features < 1:
            raise ValueError("n_features must be a positive integer")
        if value.shape[1] != (int(n_features) + 7) // 8:
            raise ValueError("Packed byte width does not match n_features")
        used = int(n_features) % 8
        if used and np.any(value[:, -1] & np.uint8(255 ^ ((1 << used) - 1))):
            raise ValueError("Nonzero tail padding: expected little-endian packbits and zero unused bits")
    return value


@dataclass(frozen=True)
class PackedReference:
    packed: np.ndarray
    popcounts: np.ndarray
    n_features: int | None

    def distance_block(self, query_indices):
        return distance_block(self, query_indices)


@dataclass(frozen=True)
class LabelState:
    labels: np.ndarray
    counts: np.ndarray


def prepare_packed(packed, n_features=None):
    """Validate once and cache exact row popcounts; no full dataset copy."""
    value = validate_packed(packed, n_features)
    counts = np.empty(value.shape[0], dtype=np.int64)
    _check(_native().packed_row_popcounts(_pointer(value, ct.c_uint8), *value.shape,
                                         _pointer(counts, ct.c_int64)), "row popcounts")
    view = value.view()
    view.flags.writeable = False
    counts.flags.writeable = False
    return PackedReference(view, counts, int(n_features) if n_features is not None else None)


def prepare_labels(labels, counts=None):
    """Check compact 0..K-1 cluster IDs and -1 noise once per partition."""
    value = np.asarray(labels)
    if value.ndim != 1 or value.dtype.kind not in "iu" or not len(value):
        raise TypeError("labels must be a nonempty one-dimensional integer array")
    if np.any(value < -1) or np.any(value > np.iinfo(np.int32).max):
        raise ValueError("Labels must be -1 or nonnegative int32 cluster IDs")
    value = np.ascontiguousarray(value, dtype=np.int32)
    retained = value[value >= 0]
    if not len(retained):
        raise ValueError("No retained cluster members")
    unique, actual = np.unique(retained, return_counts=True)
    if not np.array_equal(unique, np.arange(len(unique), dtype=np.int32)):
        raise ValueError("Non-noise labels must be compact 0..K-1")
    actual = np.ascontiguousarray(actual, dtype=np.int64)
    if len(actual) < 2:
        raise ValueError("Silhouette requires at least two retained clusters")
    if counts is not None:
        supplied = np.asarray(counts)
        if supplied.ndim != 1 or supplied.dtype.kind not in "iu" or not np.array_equal(supplied, actual):
            raise ValueError("Cluster counts do not match retained reference labels")
    view = value.view()
    view.flags.writeable = False
    actual.flags.writeable = False
    return LabelState(view, actual)


def distance_block(packed, query_indices, *, n_features=None):
    """Return exact float64 Jaccard distances M×N, including zero/zero = 0.

    Pass PackedReference to avoid revalidating packing and recomputing popcounts.
    Query indices may be out of order or repeated and refer to reference rows.
    """
    reference = packed if isinstance(packed, PackedReference) else prepare_packed(packed, n_features)
    if n_features is not None and reference.n_features != int(n_features):
        raise ValueError("Prepared feature count differs; prepare with the correct n_features first")
    n, width = reference.packed.shape
    queries = _indices(query_indices, n)
    output = np.empty((len(queries), n), dtype=np.float64)
    if len(queries):
        _check(_native().tanimoto_distance_block(_pointer(reference.packed, ct.c_uint8),
            _pointer(reference.popcounts, ct.c_int64), n, width, _pointer(queries, ct.c_int64),
            len(queries), _pointer(output, ct.c_double)), "distance block")
    return output


def reduce_block(distances, state, query_indices, *, check_distances=True):
    """Return (silhouette, a, b) against all retained reference rows.

    state must come from prepare_labels. check_distances=False skips a full D
    scan and is intended only for untouched distance_block output (or row subsets
    thereof) whose validity is already established. Shape/dtype/query/noise checks
    always run. Input arrays must remain unchanged after their preparation.
    """
    if not isinstance(state, LabelState):
        raise TypeError("state must be produced by prepare_labels")
    value = np.asarray(distances)
    queries = _indices(query_indices, len(state.labels))
    if value.dtype != np.float64 or value.shape != (len(queries), len(state.labels)) or not value.flags.c_contiguous:
        raise TypeError("distances must be C-contiguous float64 with shape (len(queries), len(labels))")
    if np.any(state.labels[queries] < 0):
        raise ValueError("Noise query rows must be removed before silhouette reduction")
    if check_distances and (not np.isfinite(value).all() or np.any(value < 0) or np.any(value > 1)):
        raise ValueError("Expected finite Tanimoto distances in [0,1]")
    silhouette, within, nearest = [np.empty(len(queries), dtype=np.float64) for _ in range(3)]
    if len(queries):
        _check(_native().reduce_silhouette_block(_pointer(value, ct.c_double), len(queries), len(state.labels),
            _pointer(state.labels, ct.c_int32), _pointer(state.counts, ct.c_int64), len(state.counts),
            _pointer(queries, ct.c_int64), _pointer(silhouette, ct.c_double), _pointer(within, ct.c_double),
            _pointer(nearest, ct.c_double)), "silhouette reduction")
    return silhouette, within, nearest


def silhouette_reduce(distances, labels, counts, query_indices, *, check_distances=True):
    """Convenience checked API; prepare_labels/reduce_block amortize validation."""
    state = labels if isinstance(labels, LabelState) else prepare_labels(labels, counts)
    if isinstance(labels, LabelState) and counts is not None and not np.array_equal(counts, labels.counts):
        raise ValueError("Supplied counts differ from prepared counts")
    return reduce_block(distances, state, query_indices, check_distances=check_distances)


def native_metadata():
    source = Path(__file__).with_name("silhouette_kernel.c")
    sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
    return {"dll": str(DLL_PATH), "dll_sha256": sha(DLL_PATH), "c_source_sha256": sha(source),
            "wrapper_sha256": sha(Path(__file__)), "hardware_popcount": bool(_native().silhouette_has_hardware_popcount()),
            "threading": "serial C loops; no OpenMP or BLAS", "max_queries_per_block": MAX_QUERIES,
            "zero_union_distance": 0.0, "singleton_silhouette": 0.0,
            "reference_policy": "all non-noise reference members; self excluded by source row index"}
'''


**`code/silhouette_kernel.c`**


In [ ]:
SOURCES['code/silhouette_kernel.c'] = '''/* Serial packed-binary Jaccard distances and full-reference silhouette reduction.
   Build with MSVC x64. No OpenMP, numerical libraries or clustering code. */
#include <stdint.h>
#include <stddef.h>
#include <stdlib.h>
#include <string.h>
#include <float.h>
#if defined(_MSC_VER)
#include <intrin.h>
#define API __declspec(dllexport)
#define INLINE __forceinline
#else
#define API __attribute__((visibility("default")))
#define INLINE inline
#endif

static INLINE uint64_t software_popcount(uint64_t x) {
    x -= (x >> 1) & UINT64_C(0x5555555555555555);
    x = (x & UINT64_C(0x3333333333333333)) + ((x >> 2) & UINT64_C(0x3333333333333333));
    x = (x + (x >> 4)) & UINT64_C(0x0f0f0f0f0f0f0f0f);
    return (x * UINT64_C(0x0101010101010101)) >> 56;
}

API int silhouette_has_hardware_popcount(void) {
#if defined(_MSC_VER) && defined(_M_X64)
    int info[4];
    __cpuid(info, 1);
    return (info[2] & (1 << 23)) != 0;
#elif defined(__GNUC__) && (defined(__x86_64__) || defined(__i386__))
    return __builtin_cpu_supports("popcnt") != 0;
#else
    return 0;
#endif
}

static INLINE uint64_t popcount(uint64_t x, int hardware) {
#if defined(_MSC_VER) && defined(_M_X64)
    if (hardware) return (uint64_t)__popcnt64(x);
#else
    (void)hardware;
#endif
    return software_popcount(x);
}

static INLINE uint64_t load_word(const uint8_t *p) {
    uint64_t result;
    memcpy(&result, p, sizeof(result));
    return result;
}

API int packed_row_popcounts(const uint8_t *packed, int64_t n, int64_t bytes,
                             int64_t *counts) {
    int64_t row, k;
    int hardware = silhouette_has_hardware_popcount();
    if (!packed || !counts || n < 1 || bytes < 1) return 1;
    if ((uint64_t)n > SIZE_MAX / (uint64_t)bytes) return 2;
    for (row = 0; row < n; ++row) {
        const uint8_t *p = packed + (size_t)row * (size_t)bytes;
        uint64_t total = 0;
        for (k = 0; k + 8 <= bytes; k += 8) total += popcount(load_word(p + k), hardware);
        for (; k < bytes; ++k) total += software_popcount(p[k]);
        counts[row] = (int64_t)total;
    }
    return 0;
}

API int tanimoto_distance_block(const uint8_t *packed, const int64_t *popcounts,
                                int64_t n, int64_t bytes, const int64_t *queries,
                                int64_t m, double *output) {
    int64_t qi, row, k;
    int hardware = silhouette_has_hardware_popcount();
    if (!packed || !popcounts || !queries || !output || n < 1 || bytes < 1 || m < 0 || m > 128) return 1;
    if ((uint64_t)n > SIZE_MAX / (uint64_t)bytes || (uint64_t)n > SIZE_MAX / sizeof(double) / (uint64_t)(m ? m : 1)) return 2;
    for (qi = 0; qi < m; ++qi) {
        int64_t q = queries[qi];
        const uint8_t *a;
        double *out = output + (size_t)qi * (size_t)n;
        if (q < 0 || q >= n) return 3;
        a = packed + (size_t)q * (size_t)bytes;
        for (row = 0; row < n; ++row) {
            const uint8_t *b = packed + (size_t)row * (size_t)bytes;
            uint64_t intersection = 0;
            int64_t union_count;
            for (k = 0; k + 32 <= bytes; k += 32) {
                intersection += popcount(load_word(a+k) & load_word(b+k), hardware);
                intersection += popcount(load_word(a+k+8) & load_word(b+k+8), hardware);
                intersection += popcount(load_word(a+k+16) & load_word(b+k+16), hardware);
                intersection += popcount(load_word(a+k+24) & load_word(b+k+24), hardware);
            }
            for (; k + 8 <= bytes; k += 8) intersection += popcount(load_word(a+k) & load_word(b+k), hardware);
            for (; k < bytes; ++k) intersection += software_popcount((uint64_t)(a[k] & b[k]));
            union_count = popcounts[q] + popcounts[row] - (int64_t)intersection;
            out[row] = union_count ? (double)(union_count - (int64_t)intersection) / (double)union_count : 0.0;
        }
    }
    return 0;
}

API int reduce_silhouette_block(const double *distances, int64_t m, int64_t n,
                                const int32_t *labels, const int64_t *counts,
                                int64_t clusters, const int64_t *queries,
                                double *silhouettes, double *within, double *nearest) {
    int64_t qi, row, cluster;
    double *sums;
    if (!distances || !labels || !counts || !queries || !silhouettes || !within || !nearest ||
        m < 0 || m > 128 || n < 1 || clusters < 2) return 1;
    if ((uint64_t)clusters > SIZE_MAX / sizeof(double)) return 2;
    for (cluster = 0; cluster < clusters; ++cluster) if (counts[cluster] < 1) return 3;
    sums = (double *)malloc((size_t)clusters * sizeof(double));
    if (!sums) return 4;
    for (qi = 0; qi < m; ++qi) {
        int64_t q = queries[qi];
        int32_t own;
        const double *distance = distances + (size_t)qi * (size_t)n;
        double a, b = DBL_MAX;
        if (q < 0 || q >= n) { free(sums); return 5; }
        own = labels[q];
        if (own < 0 || own >= clusters) { free(sums); return 6; }
        memset(sums, 0, (size_t)clusters * sizeof(double));
        for (row = 0; row < n; ++row) {
            int32_t label = labels[row];
            if (label < -1 || label >= clusters) { free(sums); return 7; }
            if (label >= 0 && row != q) sums[label] += distance[row];
        }
        a = counts[own] > 1 ? sums[own] / (double)(counts[own] - 1) : 0.0;
        for (cluster = 0; cluster < clusters; ++cluster) {
            double average;
            if (cluster == own) continue;
            average = sums[cluster] / (double)counts[cluster];
            if (average < b) b = average;
        }
        within[qi] = a;
        nearest[qi] = b;
        silhouettes[qi] = counts[own] == 1 || (a == 0.0 && b == 0.0) ? 0.0 : (b - a) / (a > b ? a : b);
    }
    free(sums);
    return 0;
}
'''


**`code/build_silhouette.ps1`**


In [ ]:
SOURCES['code/build_silhouette.ps1'] = '''$ErrorActionPreference = 'Stop'
$studyDir = (Resolve-Path -LiteralPath (Join-Path $PSScriptRoot '..')).Path
$nativeDir = Join-Path $studyDir 'native'
$source = Join-Path $PSScriptRoot 'silhouette_kernel.c'
$vswhere = 'C:\\Program Files (x86)\\Microsoft Visual Studio\\Installer\\vswhere.exe'
$vsPath = (& $vswhere -latest -products '*' -requires Microsoft.VisualStudio.Component.VC.Tools.x86.x64 -property installationPath)
if (-not $vsPath) { throw 'A local MSVC x64 toolchain was not found.' }
$toolset = Get-ChildItem -LiteralPath (Join-Path $vsPath 'VC\\Tools\\MSVC') -Directory | Sort-Object Name -Descending | Select-Object -First 1
$compilerDir = Join-Path $toolset.FullName 'bin\\Hostx64\\x64'
$compiler = Join-Path $compilerDir 'cl.exe'
$kitRoot = 'C:\\Program Files (x86)\\Windows Kits\\10'
$kitVersion = Get-ChildItem -LiteralPath (Join-Path $kitRoot 'Include') -Directory | Sort-Object Name -Descending | Select-Object -First 1
$includes = @((Join-Path $toolset.FullName 'include'), (Join-Path $kitVersion.FullName 'ucrt'), (Join-Path $kitVersion.FullName 'shared'), (Join-Path $kitVersion.FullName 'um'))
$libs = @((Join-Path $toolset.FullName 'lib\\x64'), (Join-Path $kitRoot ('Lib\\' + $kitVersion.Name + '\\ucrt\\x64')), (Join-Path $kitRoot ('Lib\\' + $kitVersion.Name + '\\um\\x64')))
New-Item -ItemType Directory -Path $nativeDir -Force | Out-Null
$savedPath = $env:PATH
$savedInclude = $env:INCLUDE
$savedLib = $env:LIB
try {
    $env:PATH = $compilerDir + ';' + $savedPath
    $env:INCLUDE = $includes -join ';'
    $env:LIB = $libs -join ';'
    Push-Location -LiteralPath $nativeDir
    try {
        $flags = @('/nologo', '/O2', '/Oi', '/MT', '/LD', '/W4', '/TC', $source, '/Fosilhouette_kernel.obj', '/Fesilhouette_kernel.dll', '/link', '/INCREMENTAL:NO')
        & $compiler @flags 2>&1 | Tee-Object -FilePath 'build.log'
        if ($LASTEXITCODE -ne 0) { throw 'Silhouette kernel compilation failed.' }
        [ordered]@{
            built_at_utc = [DateTime]::UtcNow.ToString('o')
            compiler = $compiler
            compiler_file_version = (Get-Item -LiteralPath $compiler).VersionInfo.FileVersion
            toolset = $toolset.Name
            windows_sdk = $kitVersion.Name
            architecture = 'x64'
            flags = $flags
            source_sha256 = (Get-FileHash -LiteralPath $source -Algorithm SHA256).Hash.ToLowerInvariant()
            dll_sha256 = (Get-FileHash -LiteralPath 'silhouette_kernel.dll' -Algorithm SHA256).Hash.ToLowerInvariant()
            cpu_policy = 'Runtime POPCNT detection; portable software fallback'
            threading = 'serial; no OpenMP or BLAS'
        } | ConvertTo-Json -Depth 5 | Set-Content -LiteralPath 'build_metadata.json' -Encoding UTF8
    } finally { Pop-Location }
} finally {
    $env:PATH = $savedPath
    $env:INCLUDE = $savedInclude
    $env:LIB = $savedLib
}
'''


**`code/render_scores.py`**


In [ ]:
SOURCES['code/render_scores.py'] = '''"""Render saved AMI/silhouette scores only; never fit, rescore or load datasets.

python code/render_scores.py --check
python code/render_scores.py

Reads results/summary.json. Writes plots/{synthetic,landrum},
results/SCORES.md and results/rendering_provenance.json under --root.
Scientific imports are deferred until rendering is explicitly requested.
"""
from __future__ import annotations

import argparse
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import math
import os
from pathlib import Path
import textwrap

ROOT = Path(__file__).resolve().parents[1]
METHODS = {
    "classix_packed": ("CLASSIX_T (packed)", "red", "x", "--"),
    "dbscan": ("DBSCAN", "blue", "o", "-."),
    "bitbirch": ("BitBIRCH", "green", "v", ":"),
    "bblean": ("BitBIRCH-Lean (reference)", "#9467bd", "s", "--"),
    "butina": ("Taylor-Butina", "#e68613", "D", "-"),
}
SHORT_NAMES = {"classix_packed": "CLASSIX_T", "dbscan": "DBSCAN", "bitbirch": "BitBIRCH",
               "bblean": "Lean", "butina": "Butina"}
SYNTHETIC_STUDIES = (
    ("sample_count", "Scaling with number of data points", "Number of points"),
    ("popcount_separation", "Scaling with separability of data", "Max pop of seed vector"),
    ("cluster_count", "Scaling with number of clusters", "Number of clusters"),
    ("dimension", "Scaling with dimensionality", "Number of dimensions"),
    ("radius", "Sensitivity to Tanimoto radius", "Tanimoto radius r"),
)
LANDRUM_METHODS = ("classix_packed", "bitbirch", "bblean")
METRICS = (
    ("ami", "ami_all", "Adjusted mutual information (all rows)"),
    ("silhouette", "silhouette_mean", "Mean Tanimoto silhouette"),
    ("coverage", "coverage", "Retained points / all input points"),
)
STATUS_NAMES = {"M": "memory-limited fit", "U": "undefined score", "N": "score unavailable",
                "P": "pending", "E": "failed fit"}
FULL_SYNTHETIC_CASES = frozenset(
    [f"sample_count_{n}" for n in range(10000, 90001, 10000)]
    + [f"popcount_separation_{p}" for p in range(50, 501, 50)]
    + [f"cluster_count_{k}" for k in (10, 20, 25, 40, 50, 80, 100)]
    + [f"dimension_{d}" for d in range(500, 5001, 500)]
    + [f"radius_0p{r}" for r in range(2, 8)])
FULL_LANDRUM_CASES = frozenset(f"morgan_min10_landrum_s{i:03d}" for i in range(100))


def summary_contract(summary):
    """Validate the intended experiment, independently of which records exist."""
    scope = summary.get("scope")
    if scope is None:
        # Compatibility for the original combined scratch score report only.
        if summary.get("is_pilot") or "expected_cases" in summary or "expected_methods" in summary:
            raise ValueError("An explicit experiment contract requires scope")
        return {"scope": "both", "is_pilot": False,
                "cases": {"synthetic": FULL_SYNTHETIC_CASES, "landrum": FULL_LANDRUM_CASES},
                "methods": {"synthetic": set(METHODS), "landrum": set(LANDRUM_METHODS)}}
    if scope not in ("synthetic", "landrum") or not isinstance(summary.get("is_pilot"), bool):
        raise ValueError("Scoped summary requires synthetic/landrum scope and boolean is_pilot")
    cases, methods = summary.get("expected_cases"), summary.get("expected_methods")
    for name, values in (("expected_cases", cases), ("expected_methods", methods)):
        if (not isinstance(values, list) or not values or
                any(not isinstance(item, str) or not item for item in values) or len(values) != len(set(values))):
            raise ValueError(name + " must be a nonempty list of unique identifiers")
    full_cases = FULL_SYNTHETIC_CASES if scope == "synthetic" else FULL_LANDRUM_CASES
    full_methods = set(METHODS) if scope == "synthetic" else set(LANDRUM_METHODS)
    if set(methods) != full_methods:
        raise ValueError("Expected methods do not match the complete method roster")
    if summary["is_pilot"]:
        if not set(cases).issubset(full_cases):
            raise ValueError("Pilot cases are not part of the intended full experiment")
        pilot_rows = summary.get("pilot_rows")
        if pilot_rows is not None and (not isinstance(pilot_rows, int) or isinstance(pilot_rows, bool) or pilot_rows < 2):
            raise ValueError("Invalid pilot_rows")
    elif set(cases) != full_cases:
        raise ValueError("A full report must declare every original case; reduced grids require is_pilot=true")
    return {"scope": scope, "is_pilot": summary["is_pilot"],
            "cases": {scope: frozenset(cases)}, "methods": {scope: set(methods)}}


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def finite(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def save(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n", encoding="utf-8")
    temporary.replace(path)


def value_and_status(row, metric):
    """A missing or undefined score never becomes a numeric zero."""
    if row is None:
        return None, "P", "missing setting/method record"
    status = row.get("status")
    if status != "ok":
        mark = "M" if status == "memory_limit" else "P" if status in ("pending", "running") else "E"
        return None, mark, str(status)
    key = dict((name, key) for name, key, _ in METRICS)[metric]
    value = row.get(key)
    if metric == "silhouette":
        quality_status = row.get("silhouette_status")
        if quality_status != "ok":
            mark = "U" if quality_status == "undefined" else "P" if quality_status in ("pending", "running") else "N"
            return None, mark, row.get("silhouette_reason") or str(quality_status)
    if value is None:
        return None, "N", row.get(metric + "_reason") or (key + " unavailable")
    if not finite(value):
        raise ValueError(f"Nonfinite {key}: {row.get('case_id')} / {row.get('method')}")
    return float(value), None, None


def validate_records(summary, allow_partial=False):
    contract = summary_contract(summary)
    records = summary.get("records")
    if not isinstance(records, list) or not records:
        raise ValueError("summary.json must contain a nonempty records list")
    seen, case_methods, case_x = set(), {}, {}
    for row in records:
        scope, method = row.get("scope"), row.get("method")
        if scope not in ("synthetic", "landrum") or method not in METHODS:
            raise ValueError("Unknown scope or method")
        if (scope not in contract["cases"] or row.get("case_id") not in contract["cases"][scope]
                or method not in contract["methods"][scope]):
            raise ValueError("Record lies outside the declared experiment contract")
        if scope == "landrum" and method not in LANDRUM_METHODS:
            raise ValueError("Unexpected Landrum method")
        if scope == "synthetic" and row.get("study") not in {item[0] for item in SYNTHETIC_STUDIES}:
            raise ValueError("Unknown synthetic study")
        if not finite(row.get("x")) or not row.get("case_id"):
            raise ValueError("Missing finite x or case ID")
        identity = (scope, row["case_id"], method)
        if identity in seen:
            raise ValueError("Duplicate setting/method record: " + str(identity))
        seen.add(identity)
        case_key = (scope, row["case_id"])
        definition = (row.get("study"), row["x"])
        if case_key in case_x and case_x[case_key] != definition:
            raise ValueError("Inconsistent study/x for one setting")
        case_x[case_key] = definition
        case_methods.setdefault(case_key, set()).add(method)
        for metric, _, _ in METRICS:
            value, _, _ = value_and_status(row, metric)
            if metric == "coverage" and value is not None and not 0 <= value <= 1:
                raise ValueError("Coverage is outside [0,1]")
            if metric == "silhouette" and value is not None and not -1.0000001 <= value <= 1.0000001:
                raise ValueError("Silhouette is outside [-1,1]")
        if row.get("ami_retained") is not None and not finite(row["ami_retained"]):
            raise ValueError("Nonfinite retained-population AMI")
        if row.get("silhouette_status") == "ok" and row.get("status") == "ok":
            if not isinstance(row.get("exact_silhouette"), bool):
                raise ValueError("Successful silhouette must state whether it is exact")
            if not row["exact_silhouette"] and (not finite(row.get("silhouette_se")) or row["silhouette_se"] < 0):
                raise ValueError("Sampled silhouette lacks a valid standard error")
            if not isinstance(row.get("n_queries"), int) or row["n_queries"] <= 0:
                raise ValueError("Successful silhouette lacks query count")
    expected = {(scope, case, method) for scope, cases in contract["cases"].items()
                for case in cases for method in contract["methods"][scope]}
    complete = seen == expected and summary.get("complete_case_results", True) is True
    if not complete and not allow_partial:
        raise ValueError("Missing expected case/method records or unfinished case results; use --allow-partial only for labelled previews")
    return records, complete


def groups(records):
    for study, title, xlabel in SYNTHETIC_STUDIES:
        rows = [row for row in records if row["scope"] == "synthetic" and row["study"] == study]
        if rows:
            yield "synthetic", study, title, xlabel, tuple(METHODS), rows
    rows = [row for row in records if row["scope"] == "landrum"]
    if rows:
        yield "landrum", "similarity", "Landrum / Morgan 1024 / min10 study", "Tanimoto similarity threshold s", LANDRUM_METHODS, rows


def measurement_notes(scope, study, metric, rows, complete):
    notes = [] if complete else ["PARTIAL RESULTS - only saved available records are shown."]
    if metric == "ami":
        notes += ["AMI uses all input rows and saved assignments; retained-population AMI is separate."]
        if scope == "landrum":
            notes += ["Reference: 35 assay-target categories; repeated structures can have different targets."]
    elif scope == "landrum":
        notes += ["CLASSIX: all nonnoise final clusters after minPts=10; BitBIRCH/Lean: size >10."]
    else:
        notes += ["Quality population excludes algorithm noise; coverage reports the retained fraction."]
    if metric == "silhouette":
        valid = [row for row in rows if value_and_status(row, metric)[0] is not None]
        sampled = [row for row in valid if not row["exact_silhouette"]]
        exact = [row for row in valid if row["exact_silhouette"]]
        if sampled:
            q = sorted({row["n_queries"] for row in sampled})
            qtext = str(q[0]) if len(q) == 1 else f"{q[0]}-{q[-1]}"
            notes += [f"Estimated from {qtext} query rows; shading is pointwise +/-1.96 Monte Carlo SE.",
                      "Sampling uncertainty only; no clustering-repeat uncertainty is estimated."]
        if exact:
            notes += ["Some points use all retained rows (exact); no sampling band is drawn for those points."] if sampled else ["All available silhouettes use every retained row (exact; no sampling band)."]
        notes += ["Distances are evaluated against retained reference rows; singleton silhouettes are zero."]
    if scope == "synthetic":
        notes += ["Seed 101; saved partitions checked across repeats; radius study uses 10,000 points."] if study == "radius" else ["Seed 101; one score per setting after checking saved partitions across repeats."]
    else:
        notes += ["CLASSIX radius=1-s, mergeScale=1.35; BitBIRCH/Lean diameter threshold=s."]
    return [line for note in notes for line in textwrap.wrap(note, width=106)]


def finite_marker_indices(values):
    available = [i for i, value in enumerate(values) if math.isfinite(value)]
    if len(values) <= 20:
        return available
    step = max(1, len(values) // 12)
    candidates = set(available[::step])
    for i in available:
        if i == 0 or i == len(values) - 1 or not math.isfinite(values[i-1]) or not math.isfinite(values[i+1]):
            candidates.add(i)
    return sorted(candidates)


def coincident_bb_reference(evaluated):
    """Only annotate equality when both curves have the same defined values."""
    first, second = evaluated.get("bitbirch"), evaluated.get("bblean")
    if not first or not second or len(first) != len(second):
        return False
    defined = 0
    for left, right in zip(first, second):
        if (left[0] is None) != (right[0] is None):
            return False
        if left[0] is not None:
            defined += 1
            if left[0] != right[0]:
                return False
    return defined > 0


def render_figures(summary, root, records, complete):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.transforms import blended_transform_factory
    import numpy as np

    contract = summary_contract(summary)
    is_pilot = contract["is_pilot"]
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 12,
        "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 10,
        "axes.spines.top": True, "axes.spines.right": True, "axes.grid": False,
        "figure.facecolor": "white", "axes.facecolor": "white", "svg.fonttype": "none", "pdf.fonttype": 42})
    artifacts = []
    for scope, study, title, xlabel, methods, group_rows in groups(records):
        cases = sorted({(row["x"], row["case_id"]) for row in group_rows})
        xs = np.asarray([x for x, _ in cases], dtype=float)
        if len(set(xs)) != len(xs):
            raise ValueError("A plotted study has multiple settings at one x")
        lookup = {(row["case_id"], row["method"]): row for row in group_rows}
        method_rows = {method: [lookup.get((case, method)) for _, case in cases] for method in methods}
        span = float(xs[-1] - xs[0])
        pad = 0.035 * span if span else max(0.02, abs(float(xs[0])) * .04)
        destination = root / "plots" / scope
        destination.mkdir(parents=True, exist_ok=True)
        for metric, _, ylabel in METRICS:
            evaluated = {method: [value_and_status(row, metric) for row in method_rows[method]] for method in methods}
            missing = [(method, values) for method, values in evaluated.items() if any(mark for _, mark, _ in values)]
            used_marks = sorted({mark for _, values in missing for _, mark, _ in values if mark})
            notes = measurement_notes(scope, study, metric, group_rows, complete)
            if is_pilot:
                actual_rows = sorted({row["n_total"] for row in group_rows if isinstance(row.get("n_total"), int)})
                actual = str(actual_rows[0]) if len(actual_rows) == 1 else "/".join(map(str, actual_rows))
                notes = ["PILOT CHECK ONLY - not full experiment results.",
                         f"X positions identify intended full settings; actual pilot input rows: {actual or 'see records'}."] + notes
                notes = [note.replace("radius study uses 10,000 points.", "radius study is a reduced-row pilot.") for note in notes]
                notes = [note.replace("saved partitions checked across repeats", "one pilot fit per setting")
                         .replace("one score per setting after checking saved partitions across repeats", "one pilot fit per setting") for note in notes]
            coincident = scope == "landrum" and coincident_bb_reference(evaluated)
            if coincident:
                notes.append("BitBIRCH and BitBIRCH-Lean values coincide; both curves are drawn without shifting values.")
            note_height = .024 * len(notes)
            key_lines = textwrap.wrap("Status strips: " + "; ".join(f"{mark}={STATUS_NAMES[mark]}" for mark in used_marks), 105) if used_marks else []
            note_top = .020 + .024 * (len(notes) - 1)
            key_top = note_top + .032 + .023 * max(0, len(key_lines) - 1)
            status_bottom = key_top + .032
            bottom = max(.20, .13 + note_height + .027 * len(missing) + .023 * len(key_lines))
            fig, ax = plt.subplots(figsize=(7.4, 5.4))
            fig.subplots_adjust(left=.14, right=.975, top=.90, bottom=bottom)
            if is_pilot:
                ax.text(.5, .55, "PILOT", transform=ax.transAxes, ha="center", va="center",
                        rotation=25, fontsize=54, color="#999999", alpha=.18, zorder=0)
            handles, all_values = {}, []
            bands = []
            # Draw the open reference markers first, retaining the requested legend order.
            draw_methods = sorted(methods, key=lambda method: method != "bblean")
            for method in draw_methods:
                label, color, marker, linestyle = METHODS[method]
                if method == "dbscan" and study == "popcount_separation":
                    linestyle = "--"
                values = evaluated[method]
                ys = np.asarray([value if value is not None else np.nan for value, _, _ in values])
                all_values.extend(ys[np.isfinite(ys)].tolist())
                line_options = {"markerfacecolor": "white", "alpha": .80} if method == "bblean" else {}
                marker_indices = finite_marker_indices(ys)
                if method == "bblean" and len(xs) > 20:
                    # Offset only marker positions, never the plotted coordinates or scores.
                    marker_indices = [i for i in range(4, len(xs), 10) if np.isfinite(ys[i])]
                ax.plot(xs, ys, label=label, color=color, marker=marker, linestyle=linestyle,
                        linewidth=1.3 if method == "bblean" else 1.5, markersize=4.5 if len(xs) > 20 else 5.5,
                        markevery=marker_indices, **line_options)
                handles[method] = Line2D([0], [0], label=label, color=color, marker=marker, linestyle=linestyle,
                                        linewidth=1.5, markersize=5, **line_options)
                if metric == "silhouette":
                    halfwidth = np.asarray([1.96 * row["silhouette_se"] if row is not None and value_and_status(row, metric)[0] is not None
                                            and not row["exact_silhouette"] else np.nan for row in method_rows[method]])
                    sampled = np.isfinite(halfwidth) & np.isfinite(ys)
                    if sampled.any():
                        low, high = ys - halfwidth, ys + halfwidth
                        # Pointwise descriptive Monte Carlo bounds, not a simultaneous interval.
                        ax.fill_between(xs, low, high, where=sampled, color=color, alpha=.12, linewidth=0)
                        # Isolated sampled points still show their SE when there is no neighbouring band segment.
                        if np.count_nonzero(sampled) == 1:
                            ax.errorbar(xs[sampled], ys[sampled], yerr=halfwidth[sampled], fmt="none", color=color, alpha=.5, capsize=2)
                        bands.append({"method": method, "sampled_points": int(sampled.sum())})
                        all_values.extend(low[sampled].tolist() + high[sampled].tolist())
            ax.set_title(("PILOT - " if is_pilot else "") + ("PARTIAL - " if not complete else "") + title)
            ax.set_xlabel(xlabel + (" (intended setting)" if is_pilot else ""))
            ax.set_ylabel(ylabel)
            ax.set_xlim(float(xs[0]) - pad, float(xs[-1]) + pad)
            if len(xs) <= 15:
                ax.set_xticks(xs)
            if study == "sample_count":
                ax.tick_params(axis="x", labelsize=8)
            if metric == "coverage":
                ax.set_ylim(-.025, 1.055)
            elif metric == "ami":
                ax.set_ylim(min(-.05, min(all_values, default=0) - .04), 1.055)
            else:
                ax.set_ylim(min(-.05, min(all_values, default=0) - .05), max(1.05, max(all_values, default=1) + .04))
            ax.legend(handles=[handles[method] for method in methods], loc="best", fontsize=8, frameon=True)
            if not all_values:
                ax.text(.5, .48, "No defined scores", transform=ax.transAxes, ha="center", color="#777777")
            for index, line in enumerate(notes):
                fig.text(.5, note_top - .024 * index, line, ha="center", fontsize=7)
            transform = blended_transform_factory(ax.transData, fig.transFigure)
            status_counts = {}
            for index, (method, values) in enumerate(missing):
                y = status_bottom + .027 * (len(missing) - index - 1)
                color = METHODS[method][1]
                counts = Counter(mark for _, mark, _ in values if mark)
                status_counts[method] = dict(counts)
                label = SHORT_NAMES[method] + " " + "/".join(f"{mark}:{count}" for mark, count in sorted(counts.items()))
                fig.text(.132, y, label, ha="right", va="center", fontsize=6.4, color=color)
                for x, (_, mark, _) in zip(xs, values):
                    if not mark:
                        continue
                    # Dense 100-threshold strips use a narrow tick; the row label identifies U/M/N counts.
                    symbol = mark if len(xs) <= 25 else {"U": "|", "M": "x", "N": ".", "P": ".", "E": "+"}[mark]
                    fig.text(x, y, symbol, transform=transform, ha="center", va="center", fontsize=7, color=color, fontweight="bold")
            for index, line in enumerate(key_lines):
                if len(xs) > 25:
                    line += " (dense strips: U=|, M=x, N/P=.)" if index == len(key_lines) - 1 else ""
                fig.text(.5, key_top - .023 * index, line, ha="center", fontsize=7)
            files = []
            for extension in ("png", "pdf", "svg"):
                path = destination / f"{study}_{metric}.{extension}"
                fig.savefig(path, dpi=300, facecolor="white")
                files.append({"file": path.relative_to(root).as_posix(), "sha256": sha(path)})
            plt.close(fig)
            artifacts.append({"scope": scope, "study": study, "metric": metric, "files": files,
                              "yscale": "linear", "x_limits": [float(xs[0]) - pad, float(xs[-1]) + pad],
                              "status_counts": status_counts, "notes": notes, "sampling_bands": bands,
                              "bitbirch_reference_coincident": coincident, "is_pilot": is_pilot})
    return artifacts


def availability_table(records):
    lines = ["| Scope / study | Method | Records | Fit failures | AMI all available | Silhouette available (exact / sampled) | Undefined / unavailable | Coverage available |",
             "|---|---|---:|---:|---:|---:|---:|---:|"]
    reasons = []
    for scope, study, _, _, methods, rows in groups(records):
        for method in methods:
            subset = [row for row in rows if row["method"] == method]
            ami = sum(value_and_status(row, "ami")[0] is not None for row in subset)
            coverage = sum(value_and_status(row, "coverage")[0] is not None for row in subset)
            valid = [row for row in subset if value_and_status(row, "silhouette")[0] is not None]
            exact = sum(row["exact_silhouette"] for row in valid)
            missing = Counter(value_and_status(row, "silhouette")[1] for row in subset)
            failed = sum(row["status"] != "ok" for row in subset)
            lines.append(f"| {scope} / {study} | {METHODS[method][0]} | {len(subset)} | {failed} | {ami} | {len(valid)} ({exact} / {len(valid)-exact}) | {missing['U']} / {missing['N']} | {coverage} |")
            counts = Counter(reason for row in subset for _, mark, reason in [value_and_status(row, "silhouette")] if mark)
            if counts:
                reasons.append((scope, study, method, counts))
    lines += ["", "Unavailable silhouette reasons:", ""]
    if not reasons:
        lines.append("None.")
    for scope, study, method, counts in reasons:
        lines.append(f"- {scope}/{study}, {METHODS[method][0]}: " + "; ".join(f"{reason}: {count}" for reason, count in sorted(counts.items())) + ".")
    return lines


def write_report(summary, root, summary_path, records, complete, artifacts):
    counts = Counter((row["scope"], row["status"]) for row in records)
    contract = summary_contract(summary)
    scopes = set(contract["cases"])
    lines = ["# Saved-partition AMI and Tanimoto silhouette", "",
        ("Complete setting coverage." if complete else "**PARTIAL RESULTS: this report does not cover every planned setting.**"), "",
        "This report reads saved score records. It performs no clustering or metric evaluations. Numerical values and scoring provenance remain in [summary.json](summary.json). The main AMI figures use `ami_all`; `ami_retained` remains an explicitly different population in the numerical records.", "",
        "One score is shown per method/setting after the scoring pipeline checks repeated saved partitions. Clustering repeats are not treated as independent silhouette estimates. For sampled silhouettes, shading denotes pointwise mean +/- 1.96 reported Monte Carlo standard errors, not uncertainty across fits or a simultaneous confidence band. Exact full-population silhouettes have no sampling band. The scoring protocol uses a shared seed-101 random priority per dataset and the first min(2000, retained population) eligible query rows. Equal populations share queries; differently filtered populations may use different queries. Each query is compared with every retained reference row. Reported standard errors include the finite-population correction sqrt(1-m/N).", "",
        "Silhouette uses Tanimoto distances in the retained population. Singleton silhouettes use zero; fewer than two retained clusters or a cluster for every retained point is undefined. Undefined scores and failed fits remain disconnected gaps. M denotes a memory-limited fit; U an undefined score; N an unavailable score. Their status strips do not represent numerical observations.", "",
        "AMI compares every saved row with its reference category, including any -1 noise assignments as a predicted category. For Landrum, silhouette and coverage follow the earlier quality populations: all nonnoise final CLASSIX clusters after minPts=10, and only BitBIRCH/Lean clusters of size strictly greater than 10. These populations need not match; compare coverage when interpreting silhouette differences. CLASSIX minPts performs reassignment during fitting and does not guarantee every final cluster reaches 10 points. Synthetic coverage likewise records removal of any algorithm noise; it does not invent assignments for failed fits. Where the saved BitBIRCH and Lean values coincide exactly, the Landrum figure states this explicitly and uses open reference markers at different x positions without shifting any values.", "",
        "The Landrum input has 90,498 pKi>4 records, 35 target categories and 55,513 distinct literal SMILES. It preserves 34,985 repeated-SMILES rows beyond the first occurrence. Of the distinct SMILES, 14,533 appear against multiple targets, accounting for 42,162 rows. Thus these reference categories are not a unique structure-based ground truth. Duplicates are retained in both external-label and internal-distance scores.", "",
        "## Availability", "",
        "; ".join(f"{scope}: {status}={count}" for (scope, status), count in sorted(counts.items())) + ".", ""]
    if "landrum" not in scopes:
        lines = [line for line in lines if not line.startswith("The Landrum input has")]
        lines = [line.replace("For Landrum, silhouette and coverage follow the earlier quality populations: all nonnoise final CLASSIX clusters after minPts=10, and only BitBIRCH/Lean clusters of size strictly greater than 10. These populations need not match; compare coverage when interpreting silhouette differences. CLASSIX minPts performs reassignment during fitting and does not guarantee every final cluster reaches 10 points. ", "").replace(" Where the saved BitBIRCH and Lean values coincide exactly, the Landrum figure states this explicitly and uses open reference markers at different x positions without shifting any values.", "") for line in lines]
    if contract["is_pilot"]:
        lines[0] = "# PILOT ONLY: saved-partition AMI and Tanimoto silhouette"
        lines[2] = "**PILOT CHECK ONLY. These reduced-row results are not the full experiment or publication figures.**"
        lines.insert(4, "Axes identify the selected intended full experiment settings. Actual input rows are recorded per setting; pilot estimates and populations can differ substantially from the full experiment.\\n")
        lines = [line.replace("The Landrum input has 90,498", "The full Landrum source (not this reduced-row pilot) has 90,498") for line in lines]
    lines += availability_table(records)
    lines += ["", "## Figures", "", "Every linked PNG also has PDF and SVG versions in the same directory.", ""]
    for scope, study, title, _, _, _ in groups(records):
        links = []
        for metric, _, _ in METRICS:
            links.append(f"[{metric}](../plots/{scope}/{study}_{metric}.png)")
        lines.append(f"- {scope}: {title} - " + ", ".join(links) + ".")
    lines += ["", "## Provenance", "",
        f"- Score summary SHA256: `{sha(summary_path)}`.",
        f"- Renderer SHA256: `{sha(__file__)}`.",
        "- Figure hashes, availability markers and measurement notes: [rendering_provenance.json](rendering_provenance.json).",
        "- Method colours and symbols follow the earlier saved-run renderer; all quality axes are linear. BitBIRCH-Lean remains a separately labelled optimized reference.", ""]
    for key in ("protocol", "scoring_protocol", "sampling", "sources", "provenance"):
        if key in summary:
            lines += [f"Recorded `{key}`:", "", "```json", json.dumps(summary[key], indent=2, allow_nan=False), "```", ""]
    output = root / "results" / "SCORES.md"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text("\\n".join(lines), encoding="utf-8")
    return output


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--root", type=Path, default=ROOT)
    parser.add_argument("--summary", type=Path)
    parser.add_argument("--check", action="store_true", help="Validate record structure without importing plotting libraries or writing output")
    parser.add_argument("--allow-partial", action="store_true", help="Create explicitly labelled previews from incomplete setting coverage")
    args = parser.parse_args()
    root = args.root.resolve()
    summary_path = args.summary.resolve() if args.summary else root / "results" / "summary.json"
    initial_hash = sha(summary_path)
    summary = read(summary_path)
    records, complete = validate_records(summary, args.allow_partial)
    contract = summary_contract(summary)
    pending = sum(row.get("status") in ("pending", "running") or (
        row.get("status") == "ok" and row.get("silhouette_status") in ("pending", "running")) for row in records)
    if args.check:
        print(json.dumps({"records": len(records), "complete_setting_coverage": complete,
                          "scope": contract["scope"], "is_pilot": contract["is_pilot"],
                          "pending_scores": pending, "status": "ok", "writes": 0}))
        return 0
    if pending and not args.allow_partial:
        raise ValueError("Scoring is still pending; wait for completion or explicitly use --allow-partial")
    complete = complete and pending == 0
    for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[key] = "1"
    artifacts = render_figures(summary, root, records, complete)
    expected_figures = 3 * sum(1 for _ in groups(records))
    if complete and len(artifacts) != expected_figures:
        raise ValueError("A complete report requires three figures per present study")
    if complete and not contract["is_pilot"] and len(artifacts) != {"synthetic": 15, "landrum": 3, "both": 18}[contract["scope"]]:
        raise ValueError("The full report does not cover every study")
    if sha(summary_path) != initial_hash:
        raise RuntimeError("Score summary changed during rendering")
    report = write_report(summary, root, summary_path, records, complete, artifacts)
    save({"status": "complete" if complete else "partial", "operation": "saved-score rendering only",
          "generated_utc": datetime.now(timezone.utc).isoformat(), "records": len(records),
          "scope": contract["scope"], "is_pilot": contract["is_pilot"],
          "summary_sha256": initial_hash, "renderer_sha256": sha(__file__),
          "fits": 0, "metric_evaluations": 0, "report": report.relative_to(root).as_posix(),
          "figures": artifacts}, root / "results" / "rendering_provenance.json")
    print(json.dumps({"status": "complete" if complete else "partial", "figures": len(artifacts),
                      "files": sum(len(item["files"]) for item in artifacts), "report": str(report)}))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''


In [ ]:
assert os.name == "nt" and platform.machine().lower() in {"amd64", "x86_64"},     "Fresh runs use the original Windows x64 native memory protocol."
WORK_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_ROOT = WORK_DIR / "runtime"
RUNTIME_ROOT.mkdir(exist_ok=True)

def write_if_identical(path, text):
    payload = text.encode("utf-8")
    if path.exists():
        assert path.read_bytes() == payload, f"Source/design changed; choose a new WORK_DIR: {path}"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(payload)

for relative, text in SOURCES.items():
    write_if_identical(RUNTIME_ROOT / relative, text)
write_if_identical(RUNTIME_ROOT / "plan.json", json.dumps(PLAN, indent=2) + "\n")
write_if_identical(RUNTIME_ROOT / "design/repetition_policy.json", json.dumps(REPETITION_POLICY, indent=2) + "\n")
source_manifest = {name: hashlib.sha256(text.encode("utf-8")).hexdigest() for name, text in SOURCES.items()}
write_if_identical(RUNTIME_ROOT / "source_manifest.json", json.dumps(source_manifest, indent=2) + "\n")

if PYTHON_FOR_RUNS is None:
    venv_dir = WORK_DIR / ".venv"
    python = venv_dir / "Scripts/python.exe"
    if not python.exists():
        base = [sys.executable] if sys.version_info[:2] == (3, 12) else ["py", "-3.12"]
        subprocess.run(base + ["-m", "venv", str(venv_dir)], check=True, creationflags=subprocess.CREATE_NO_WINDOW)
else:
    python = Path(PYTHON_FOR_RUNS).expanduser().resolve()
subprocess.run([str(python), "-c", "import sys; assert sys.version_info[:2] == (3,12) and sys.maxsize > 2**32"],
               check=True, creationflags=subprocess.CREATE_NO_WINDOW)

write_if_identical(RUNTIME_ROOT/"requirements.txt", PINNED_REQUIREMENTS)
if INSTALL_DEPENDENCIES:
    import urllib.request
    subprocess.run([str(python), "-m", "pip", "install", "-r", str(RUNTIME_ROOT/"requirements.txt")],
                   check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    download_dir = WORK_DIR / "downloads"
    download_dir.mkdir(exist_ok=True)
    wheel = download_dir / BBLEAN_WHEEL["filename"]
    if not wheel.is_file():
        urllib.request.urlretrieve(BBLEAN_WHEEL["url"], wheel)
    assert hashlib.sha256(wheel.read_bytes()).hexdigest() == BBLEAN_WHEEL["sha256"], "BBLean wheel hash mismatch"
    subprocess.run([str(python), "-m", "pip", "install", "--no-deps", str(wheel)],
                   check=True, creationflags=subprocess.CREATE_NO_WINDOW)
print("Runtime sources:", RUNTIME_ROOT)
print("Python:", python)

In [ ]:
import shutil
native_dir = RUNTIME_ROOT / "code/classix"
build_dir = WORK_DIR / "native_build"
build_dir.mkdir(exist_ok=True)
kernel_sources = [native_dir / "packed_kernel.c", native_dir / "spsubmatxvec.c"]
kernel_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in kernel_sources}
build_record = build_dir / "build.json"
if build_record.exists():
    build_info = json.loads(build_record.read_text())
    assert build_info["source_sha256"] == kernel_hashes
    for name, expected in build_info["dll_sha256"].items():
        assert hashlib.sha256((native_dir/name).read_bytes()).hexdigest() == expected
else:
    vswhere = Path(os.environ.get("ProgramFiles(x86)", "C:/Program Files (x86)")) / "Microsoft Visual Studio/Installer/vswhere.exe"
    assert vswhere.is_file(), "Install Visual Studio C++ Build Tools and Windows SDK first."
    install = subprocess.check_output([str(vswhere), "-latest", "-products", "*", "-requires",
        "Microsoft.VisualStudio.Component.VC.Tools.x86.x64", "-property", "installationPath"], text=True).strip()
    assert install, "The x64 C++ toolset was not found."
    vcvars = Path(install) / "VC/Auxiliary/Build/vcvars64.bat"
    assert vcvars.is_file()
    # A fixed local build script avoids carrying machine-specific binaries.
    commands = ["@echo off", f'call "{vcvars}"', "if errorlevel 1 exit /b 1",
        f'cl /nologo /O2 /Oi /MT /LD /W4 /TC "{kernel_sources[0]}" /Fopacked_kernel.obj /Fepacked_kernel.dll /link /INCREMENTAL:NO',
        "if errorlevel 1 exit /b 1",
        f'cl /nologo /O2 /Oi /MT /LD /W4 /TC "{kernel_sources[1]}" /Fooriginal_spmv.obj /Feoriginal_spmv.dll /link /EXPORT:multiply /INCREMENTAL:NO',
        "if errorlevel 1 exit /b 1"]
    script = build_dir / "build.cmd"
    script.write_text("\n".join(commands) + "\n", encoding="utf-8")
    with (build_dir/"build.log").open("w", encoding="utf-8") as log:
        subprocess.run(["cmd.exe", "/d", "/c", str(script)], cwd=build_dir, stdout=log,
                       stderr=subprocess.STDOUT, check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    for name in ("packed_kernel.dll", "original_spmv.dll"):
        shutil.copy2(build_dir/name, native_dir/name)
    build_info = {"source_sha256": kernel_hashes,
        "dll_sha256": {name: hashlib.sha256((native_dir/name).read_bytes()).hexdigest()
                       for name in ("packed_kernel.dll", "original_spmv.dll")},
        "visual_studio_installation": install, "flags": "/O2 /Oi /MT /LD /W4 /TC",
        "compiler_output": "build.log"}
    build_record.write_text(json.dumps(build_info, indent=2) + "\n")
print("Compiled native kernels:", build_info["dll_sha256"])

## Run or resume

`full` uses all 42 settings and their explicit schedules. `pilot` uses the generated 10k dataset's first 256 shuffled rows, all five methods, and one fit each. Full input arrays are loaded before slicing, so pilot memory is diagnostic only. `check` checks imports and the design without generating data or fitting.

The controller prevents idle sleep while running. Create `PAUSE` in the run directory to pause after the current fit, or `STOP_CURRENT` to interrupt the current attempt and preserve its evidence. Remove the control file and rerun this cell to resume. Successful checkpoints are reused only under the identical generated source/plan/package protocol. New measurements and pilot results remain separate.

For a **Figure 5-only rerun**, finish the setup/native-build cells above, then execute this example in place of the full-run cell below. It selects only the ten new 10,000-row popcount cases. Skip the later full-suite report and AMI/silhouette cells for this targeted run; its saved measurements stay in the separate `figure5` work directory.

```python
figure5_cases = [c["case_id"] for c in PLAN["cases"] if c["study"] == "popcount_separation"]
figure5_run = RUNTIME_ROOT / "runs" / "figure5"
subprocess.run([str(python), "-B", str(RUNTIME_ROOT / "code/reproduce.py"),
                "--run", "--output-dir", str(figure5_run), "--cases", *figure5_cases],
               cwd=RUNTIME_ROOT, check=True, creationflags=subprocess.CREATE_NO_WINDOW)
```

New case and dataset IDs, source/protocol checks, and exact case checks prevent reuse of the former 25,000-row results.


In [ ]:
RUN_DIR = RUNTIME_ROOT / "runs" / ("pilot" if RUN_MODE == "pilot" else "reproduction")
command = [str(python), "-B", str(RUNTIME_ROOT/"code/reproduce.py"),
           "--check" if RUN_MODE == "check" else "--run", "--output-dir", str(RUN_DIR)]
if RUN_MODE == "pilot":
    command += ["--cases", PILOT_CASE, "--pilot-rows", str(PILOT_ROWS)]
print("Run directory:", RUN_DIR)
print("Command:", subprocess.list2cmdline(command))
subprocess.run(command, cwd=RUNTIME_ROOT, check=True, creationflags=subprocess.CREATE_NO_WINDOW)

## Verify new assignments and produce figures

ARI is independently recomputed using scikit-learn against freshly generated truth, with label counts and partition hashes checked. Status rows preserve failures and missing settings. Partial and pilot figures are explicitly marked. Plots use the validated method colours, markers, axes, fonts, repetition footnotes and resource-failure annotations.

No saved measurements are needed to create these figures. New runtime/memory values need not equal the supplied historical figures. A complete new run writes all 45 figure files to `PLOT_DIR`; partial/pilot figures go under the work directory so they cannot replace the historical full-sweep figures.

In [ ]:
if RUN_MODE == "check":
    print("Import/design check completed; no figures generated.")
else:
    report_script = RUNTIME_ROOT / "code/finish_report.py"
    # Run postprocessing in the isolated environment after all fit workers exit.
    report_source = r'''import json
from pathlib import Path
import sys
from notebook_report import collect_fresh, verify_fresh, figures
root, run_dir, requested = map(Path, sys.argv[1:4])
assert not list((root/"runs").rglob("run.lock")), "Wait for fit controllers to exit."
verification = verify_fresh(root, run_dir)
summary = collect_fresh(root, run_dir)
report = run_dir/"report"
report.mkdir(exist_ok=True)
(report/"verification.json").write_text(json.dumps(verification, indent=2, allow_nan=False)+"\n")
destination = requested if summary["complete_case_results"] and not summary["is_pilot"] else report/"plots"
artifacts = figures(summary, destination)
summary["figures"] = artifacts
(report/"summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False)+"\n")
print(json.dumps({"successful_assignments_checked": verification["successful_assignments_checked"],
                  "terminal_failures": len(verification["terminal_failures"]),
                  "figures": len(artifacts), "plots": str(destination), "report": str(report)}, indent=2))
'''
    write_if_identical(report_script, report_source)
    subprocess.run([str(python), "-B", str(report_script), str(RUNTIME_ROOT), str(RUN_DIR), str(PLOT_DIR.resolve())],
                   check=True, cwd=RUNTIME_ROOT, creationflags=subprocess.CREATE_NO_WINDOW)

## AMI and Tanimoto silhouette from the new assignments

This step starts only after every timed fit worker has exited. It does not fit or tune any estimator. **AMI** uses all generated reference labels and all saved assignments, including recorded noise (`-1`) as its assigned category; it is computed on the full dataset with the arithmetic normalizer.

**Silhouette** uses binary Tanimoto/Jaccard distance. Recorded noise is excluded; every nonnoise cluster, including singletons, is retained. Each sampled query is compared with **all retained reference points**, not just other sampled queries. The same `default_rng(101)` row priority is used across methods on a dataset; the first up to 2,000 retained rows are sampled, with source-row self exclusion and singleton silhouette zero. Thus the plotted mean is an estimate of the full retained-population silhouette unless every retained row is evaluated. Error bands show finite-population Monte Carlo sampling error, not fit variability or statistical significance.

Coverage accompanies the scores because retained populations can differ. Mathematically undefined silhouettes (fewer than two retained clusters or every retained point a singleton) and unavailable fits remain missing values; curves do not bridge these gaps. Every measured assignment is checked. Repeated identical partitions are scored once and are not counted as independent observations. If a new run produces different partitions across measured repetitions, scoring stops explicitly rather than selecting one repetition; evaluating that variability requires a separately specified aggregation policy.

The additional AMI, silhouette and retained-coverage figures use the existing method colours and study axes. Completed full runs publish 15 additional figures (PNG/PDF/SVG) to `PLOT_DIR`; pilot or incomplete-run outputs stay under `WORK_DIR`. All scores, labels, manifests, compiler products and caches are generated outside this notebook folder. No historical score arrays are embedded.


In [ ]:
if RUN_MODE == "check":
    print("Import/design check completed; AMI and silhouette evaluation skipped.")
else:
    assert not list((RUNTIME_ROOT / "runs").rglob("run.lock")), "Wait for all timed fit controllers to exit."
    metrics_output = RUNTIME_ROOT / "derived" / ("ami_silhouette_pilot" if RUN_MODE == "pilot" else "ami_silhouette_full")
    silhouette_build = RUNTIME_ROOT / "native/build_metadata.json"
    silhouette_dll = RUNTIME_ROOT / "native/silhouette_kernel.dll"
    if silhouette_build.exists():
        metadata = json.loads(silhouette_build.read_text(encoding="utf-8-sig"))
        assert metadata["source_sha256"] == hashlib.sha256((RUNTIME_ROOT / "code/silhouette_kernel.c").read_bytes()).hexdigest()
        assert metadata["dll_sha256"] == hashlib.sha256(silhouette_dll.read_bytes()).hexdigest()
    else:
        assert not silhouette_dll.exists(), "Native DLL has no build provenance; use a fresh WORK_DIR."
        subprocess.run(["powershell", "-NoProfile", "-ExecutionPolicy", "Bypass", "-File",
                        str(RUNTIME_ROOT / "code/build_silhouette.ps1")],
                       cwd=RUNTIME_ROOT, check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    metrics_command = [str(python), "-B", str(RUNTIME_ROOT / "code/saved_scores.py"),
                       "--scope", "synthetic", "--plan", str(RUNTIME_ROOT / "plan.json"),
                       "--results-dir", str(RUN_DIR / "results"), "--data-dir", str(RUN_DIR / "data"),
                       "--output-dir", str(metrics_output)]
    if RUN_MODE == "pilot":
        metrics_command += ["--cases", PILOT_CASE, "--pilot-rows", str(PILOT_ROWS)]
    print("Scoring saved assignments:", subprocess.list2cmdline(metrics_command))
    subprocess.run(metrics_command, cwd=RUNTIME_ROOT, check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    original_report = json.loads((RUN_DIR / "report/summary.json").read_text(encoding="utf-8"))
    metric_summary_path = metrics_output / "results/summary.json"
    metric_summary = json.loads(metric_summary_path.read_text(encoding="utf-8"))
    metric_rendering = json.loads((metrics_output / "results/rendering_provenance.json").read_text(encoding="utf-8"))
    metric_verification = json.loads((metrics_output / "verification.json").read_text(encoding="utf-8"))
    metric_summary_hash = hashlib.sha256(metric_summary_path.read_bytes()).hexdigest()
    assert metric_summary["scope"] == metric_rendering["scope"] == "synthetic"
    assert metric_summary["is_pilot"] == metric_rendering["is_pilot"] == (RUN_MODE == "pilot")
    assert metric_rendering["status"] == "complete" and metric_verification["status"] == "passed"
    assert metric_rendering["summary_sha256"] == metric_verification["summary_sha256"] == metric_summary_hash
    assert metric_rendering["renderer_sha256"] == hashlib.sha256((RUNTIME_ROOT / "code/render_scores.py").read_bytes()).hexdigest()
    assert metric_summary["complete_case_results"] and not any(row["silhouette_status"] == "pending" for row in metric_summary["records"])
    expected_metric_cases = [PILOT_CASE] if RUN_MODE == "pilot" else [case["case_id"] for case in PLAN["cases"]]
    assert set(metric_summary["expected_cases"]) == set(expected_metric_cases)
    assert set(metric_summary["expected_methods"]) == set(PLAN["methods"])
    metric_files = [item for figure in metric_rendering["figures"] for item in figure["files"]]
    assert len({item["file"] for item in metric_files}) == len(metric_files)
    for item in metric_files:
        figure = metrics_output / item["file"]
        assert figure.resolve().is_relative_to((metrics_output / "plots/synthetic").resolve())
        assert hashlib.sha256(figure.read_bytes()).hexdigest() == item["sha256"]
    metric_figures = sorted(p for p in (metrics_output / "plots/synthetic").iterdir()
                            if p.is_file() and p.suffix.lower() in {".png", ".pdf", ".svg"})
    assert {str(p.relative_to(metrics_output).as_posix()) for p in metric_figures} == {item["file"] for item in metric_files}
    assert metric_figures, "No AMI/silhouette/coverage figures were generated."
    metric_destination = metrics_output / "plots/synthetic"
    if RUN_MODE == "full" and original_report["complete_case_results"] and not original_report["is_pilot"]:
        assert len(metric_figures) == 45, "A complete synthetic sweep must produce 15 figures in three formats."
        metric_destination = PLOT_DIR.resolve()
        metric_destination.mkdir(parents=True, exist_ok=True)
        for source in metric_figures:
            target = metric_destination / source.name
            shutil.copyfile(source, target)
            assert hashlib.sha256(target.read_bytes()).digest() == hashlib.sha256(source.read_bytes()).digest()
    print("AMI/silhouette report:", metrics_output)
    print("AMI/silhouette/coverage plots:", metric_destination)


## Source and reproducibility notes

The readable source cells preserve the original synthetic generator, fresh-process timed worker, packed CLASSIX implementation, adjacency storage, native kernels and BitBIRCH core. Changes concern generation of fresh manifests, notebook setup, source materialization, and collection/verification of new results. The original aggregation and measurement procedures are preserved. The sample-count, popcount and dimension runtime plots use the final manuscript presentation: plain method names and no footer annotations; the sample-count legend uses automatic placement. Source hashes identify the implementation origin; newly materialized source hashes, package versions, platform and run protocols are generated in the work directory.

CLASSIX source carries its MIT license. Included original BitBIRCH source is from commit `311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f`, retaining its license and attribution. BBLean 0.10.3 and other dependencies are installed as their own distributions with their accompanying licenses. No upstream core code is modified to improve the comparison.